# Notebook 12 — Course Location and Timezone Mapping

## Purpose

Build and validate the reusable physical racecourse-location reference required by Notebook 11.

This notebook will:

- inspect the existing 394-row course-location scaffold and validator;
- design a cached, rate-limited geocoding workflow;
- retain every query and raw provider response for audit;
- validate candidate venues against course identity and jurisdiction;
- derive historical IANA timezones from accepted coordinates;
- classify every course as validated, manually reviewed, or explicitly unresolved;
- save the completed reference to `data/reference/course_locations.csv`.

No source racing data will be modified, and no bulk API requests will be made until the environment, provider requirements, cache design, and validation rules have been inspected.

> **Archived completed research notebook**
>
> This notebook preserves the original course-timezone resolution process and
> its executed outputs. It is not intended to be rerun against the completed
> permanent reference, because the underlying reference state changed when the
> final timezone assignments were persisted.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
repo_root = Path.cwd().resolve().parent

paths_to_check = [
    repo_root / "data/reference/course_locations.csv",
    repo_root / "src/inside_rails/course_locations.py",
    repo_root / "src/inside_rails/course_jurisdiction.py",
    repo_root / "data/reference",
    repo_root / "data/cache",
]

print("Repository root:", repo_root)

for path in paths_to_check:
    status = "exists" if path.exists() else "missing"
    kind = "directory" if path.is_dir() else "file"
    print(f"{status:7} | {kind:9} | {path.relative_to(repo_root)}")

Repository root: /home/rob/Documents/inside-rails-horse-racing
exists  | file      | data/reference/course_locations.csv
exists  | file      | src/inside_rails/course_locations.py
exists  | file      | src/inside_rails/course_jurisdiction.py
exists  | directory | data/reference
exists  | directory | data/cache


In [2]:
import pandas as pd

course_locations_path = repo_root / "data/reference/course_locations.csv"
course_locations = pd.read_csv(course_locations_path)

print("Rows:", len(course_locations))
print("Columns:", len(course_locations.columns))
print("\nColumn names:")
print(course_locations.columns.tolist())

display(course_locations.head(3))

Rows: 394
Columns: 22

Column names:
['candidate_course_label', 'candidate_jurisdiction', 'physical_venue_name', 'locality', 'region', 'country', 'latitude', 'longitude', 'iana_timezone', 'location_evidence', 'location_validation_status', 'raw_course_labels', 'provisional_races', 'meeting_dates', 'earliest_date', 'latest_date', 'has_reusable_location', 'address_line', 'suburb', 'postcode', 'provider_place_id', 'provider_display_name']


,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_evidence,...,provisional_races,meeting_dates,earliest_date,latest_date,has_reusable_location,address_line,suburb,postcode,provider_place_id,provider_display_name
0,La Plata,Argentina,Hipódromo de La Plata,La Plata,Buenos Aires,Argentina,-34.901267,-57.943804,America/Argentina/Buenos_Aires,Nominatim manual selection from query 'Hipódro...,...,37,28,2015-01-18,2025-09-21,True,NaN,NaN,NaN,NaN,NaN
1,Palermo,Argentina,Hipódromo Argentino de Palermo,Buenos Aires,Autonomous City of Buenos Aires,Argentina,-34.566398,-58.425727,America/Argentina/Buenos_Aires,Nominatim manual selection from query 'Palermo...,...,200,76,2015-01-03,2025-10-11,True,4101 Avenida Del Libertador,Palermo,C1426BSD,way:18772836,"Hipódromo Argentino de Palermo, 4101, Avenida ..."
2,San Isidro,Argentina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,179,80,2015-02-07,2025-10-04,False,NaN,NaN,NaN,NaN,NaN


In [3]:
course_locations_module_path = (
    repo_root / "src/inside_rails/course_locations.py"
)

print(course_locations_module_path.read_text())


"""Load and validate the curated course-location reference."""

from __future__ import annotations

from pathlib import Path
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import pandas as pd


IDENTITY_COLUMNS = [
    "candidate_course_label",
    "candidate_jurisdiction",
]

REQUIRED_COLUMNS = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "physical_venue_name",
    "locality",
    "region",
    "country",
    "latitude",
    "longitude",
    "iana_timezone",
    "location_evidence",
    "location_validation_status",
]


def load_course_locations(path: str | Path) -> pd.DataFrame:
    """Load and validate the curated course-location reference."""

    reference_path = Path(path)
    frame = pd.read_csv(reference_path)

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            "Course-location reference is missing required columns: "

In [4]:
from importlib.util import find_spec

packages = [
    "geopy",
    "timezonefinder",
    "requests",
    "requests_cache",
]

print("Package availability:")
for package in packages:
    status = "installed" if find_spec(package) else "missing"
    print(f"{status:9} | {package}")

print("\nExisting cache-like paths:")
cache_like_paths = sorted(
    path.relative_to(repo_root)
    for path in repo_root.rglob("*")
    if "cache" in path.name.lower()
    and ".git" not in path.parts
)

if cache_like_paths:
    for path in cache_like_paths[:50]:
        print(path)
else:
    print("None found")

Package availability:
installed | geopy
installed | timezonefinder
installed | requests
missing   | requests_cache

Existing cache-like paths:
.venv/lib/python3.12/site-packages/IPython/__pycache__
.venv/lib/python3.12/site-packages/IPython/core/__pycache__
.venv/lib/python3.12/site-packages/IPython/core/magics/__pycache__
.venv/lib/python3.12/site-packages/IPython/extensions/__pycache__
.venv/lib/python3.12/site-packages/IPython/extensions/deduperreload/__pycache__
.venv/lib/python3.12/site-packages/IPython/external/__pycache__
.venv/lib/python3.12/site-packages/IPython/lib/__pycache__
.venv/lib/python3.12/site-packages/IPython/sphinxext/__pycache__
.venv/lib/python3.12/site-packages/IPython/sphinxext/tests/__pycache__
.venv/lib/python3.12/site-packages/IPython/terminal/__pycache__
.venv/lib/python3.12/site-packages/IPython/terminal/pt_inputhooks/__pycache__
.venv/lib/python3.12/site-packages/IPython/terminal/shortcuts/__pycache__
.venv/lib/python3.12/site-packages/IPython/testing/__p

In [5]:
dependency_files = [
    repo_root / "pyproject.toml",
    repo_root / "requirements.txt",
    repo_root / "requirements-dev.txt",
    repo_root / "environment.yml",
]

for path in dependency_files:
    if path.exists():
        print(f"\n--- {path.relative_to(repo_root)} ---")
        print(path.read_text())


--- pyproject.toml ---
[project]
name = "inside-rails"
version = "0.1.0"
description = "Notebook-led UK and Ireland horse-racing database project"
requires-python = ">=3.11"

[tool.pytest.ini_options]
pythonpath = ["src"]
testpaths = ["tests"]

[tool.ruff]
line-length = 100
target-version = "py311"


--- requirements.txt ---
jupyterlab
ipykernel
pandas
numpy
matplotlib
sqlalchemy
pyarrow
pytest
ruff



In [6]:
%pip install geopy timezonefinder

Note: you may need to restart the kernel to use updated packages.


In [7]:
import geopy
import timezonefinder
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

print("geopy version:", geopy.__version__)
print(
    "timezonefinder version:",
    getattr(timezonefinder, "__version__", "not exposed"),
)

timezone_finder = TimezoneFinder()

test_timezone = timezone_finder.timezone_at(
    lat=51.5074,
    lng=-0.1278,
)

print("London coordinate result:", test_timezone)
print("zoneinfo validation:", ZoneInfo(test_timezone))

geopy version: 2.5.0
timezonefinder version: not exposed
London coordinate result: Europe/London
zoneinfo validation: Europe/London


In [8]:
# Define simple file-based caches for the geocoding workflow.
#
# A CSV manifest will record one row per attempted query so that the notebook
# can detect previous successful, empty, ambiguous, rejected, or failed searches.
#
# A JSONL file will preserve each complete raw provider response without
# flattening or discarding nested fields.
#
# The final accepted course-location dimension remains:
# data/reference/course_locations.csv

cache_dir = repo_root / "data/cache"
cache_dir.mkdir(parents=True, exist_ok=True)

geocoding_cache_path = cache_dir / "course_geocoding_cache.csv"
geocoding_raw_responses_path = (
    cache_dir / "course_geocoding_responses.jsonl"
)

print("Cache directory:", cache_dir)
print("Query cache:", geocoding_cache_path)
print("Raw-response cache:", geocoding_raw_responses_path)
print("Query cache exists:", geocoding_cache_path.exists())
print(
    "Raw-response cache exists:",
    geocoding_raw_responses_path.exists(),
)

Cache directory: /home/rob/Documents/inside-rails-horse-racing/data/cache
Query cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/course_geocoding_cache.csv
Raw-response cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/course_geocoding_responses.jsonl
Query cache exists: True
Raw-response cache exists: True


In [9]:
# Inspect the candidate jurisdictions before designing geocoding queries.
#
# This cell does not modify any files or send any external requests.
#
# It shows:
# - how many candidate course identities belong to each jurisdiction;
# - representative course labels from each jurisdiction;
# - whether jurisdiction values are already consistent enough to map to
#   geocoder country names and country codes.

jurisdiction_profile = (
    course_locations
    .groupby("candidate_jurisdiction", dropna=False)
    .agg(
        candidate_courses=("candidate_course_label", "size"),
        sample_courses=(
            "candidate_course_label",
            lambda values: ", ".join(sorted(values.astype(str))[:5]),
        ),
    )
    .reset_index()
    .sort_values(
        ["candidate_courses", "candidate_jurisdiction"],
        ascending=[False, True],
    )
)

print("Distinct jurisdictions:", len(jurisdiction_profile))
display(jurisdiction_profile)

Distinct jurisdictions: 36


,candidate_jurisdiction,candidate_courses,sample_courses
10,France,73,"Amiens, Angers, Angouleme, Argentan, Auch"
12,Great Britain,65,"Aintree, Ascot, Ayr, Bangor-on-Dee, Bath"
34,United States,56,"Aqueduct, Arlington Park, Belmont Park, Belter..."
1,Australia,51,"Albury, Alice springs, Armidale, Ascot, Balaklava"
16,Ireland,27,"Ballinrobe, Bellewstown, Clonmel, Cork, Curragh"
18,Japan,21,"Chukyo, Fukushima, Funabashi, Hakodate, Hanshin"
11,Germany,17,"Bad Doberan, Baden-Baden, Bremen, Cologne, Dor..."
20,New Zealand,14,"Ashburton, Awapuni, Ellerslie, Hastings, Matamata"
27,South Africa,9,"Durbanville, Flamingo Park, Greyville, Kenilwo..."
17,Italy,7,"Capannelle, Merano, Naples, Pisa, San Siro"


In [10]:
# Identify course labels that occur in more than one jurisdiction.
#
# These collisions are especially important for geocoding because a query
# based only on the course name could return a geographically valid but
# completely wrong racing venue.
#
# This cell is read-only and sends no external requests.

duplicate_label_mask = course_locations.duplicated(
    subset=["candidate_course_label"],
    keep=False,
)

cross_jurisdiction_labels = (
    course_locations.loc[
        duplicate_label_mask,
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "raw_course_labels",
            "provisional_races",
        ],
    ]
    .sort_values(
        ["candidate_course_label", "candidate_jurisdiction"]
    )
    .reset_index(drop=True)
)

print(
    "Course labels appearing in multiple jurisdictions:",
    cross_jurisdiction_labels["candidate_course_label"].nunique(),
)

display(cross_jurisdiction_labels)

Course labels appearing in multiple jurisdictions: 3


,candidate_course_label,candidate_jurisdiction,raw_course_labels,provisional_races
0,Ascot,Australia,Ascot (AUS),204
1,Ascot,Great Britain,Ascot,1734
2,Newcastle,Australia,Newcastle (AUS),77
3,Newcastle,Great Britain,Newcastle,930
4,Sandown,Australia,Sandown (AUS),65
5,Sandown,Great Britain,Sandown,1631


## Jurisdiction and country-code controls

Geocoding will use an explicit jurisdiction mapping rather than passing the
source jurisdiction text directly to the provider.

For each candidate jurisdiction, the mapping will define:

- the country or territory name used in the search query;
- the ISO 3166-1 alpha-2 code used to restrict or validate results;
- acceptable country codes returned by the provider;
- any special territorial treatment.

The full candidate identity will always be preserved:

`candidate_course_label + candidate_jurisdiction`

This is essential for the cross-jurisdiction collisions at Ascot, Newcastle
and Sandown. Neither API caching nor result selection may use the course label
alone.

In [11]:
# Define the country and territory controls used by the geocoding workflow.
#
# Each candidate jurisdiction maps to:
# - query_country: text included in the search query;
# - country_codes: ISO alpha-2 codes acceptable in returned results.
#
# The mapping is deliberately explicit so provider naming differences do not
# silently weaken country validation. This cell sends no external requests.

JURISDICTION_GEOCODING_RULES = {
    "Argentina": {
        "query_country": "Argentina",
        "country_codes": {"ar"},
    },
    "Australia": {
        "query_country": "Australia",
        "country_codes": {"au"},
    },
    "Bahrain": {
        "query_country": "Bahrain",
        "country_codes": {"bh"},
    },
    "Belgium": {
        "query_country": "Belgium",
        "country_codes": {"be"},
    },
    "Brazil": {
        "query_country": "Brazil",
        "country_codes": {"br"},
    },
    "Canada": {
        "query_country": "Canada",
        "country_codes": {"ca"},
    },
    "Chile": {
        "query_country": "Chile",
        "country_codes": {"cl"},
    },
    "China": {
        "query_country": "China",
        "country_codes": {"cn"},
    },
    "Czech Republic": {
        "query_country": "Czechia",
        "country_codes": {"cz"},
    },
    "Denmark": {
        "query_country": "Denmark",
        "country_codes": {"dk"},
    },
    "France": {
        "query_country": "France",
        "country_codes": {"fr"},
    },
    "Germany": {
        "query_country": "Germany",
        "country_codes": {"de"},
    },
    "Great Britain": {
        "query_country": "United Kingdom",
        "country_codes": {"gb"},
    },
    "Guernsey": {
        "query_country": "Guernsey",
        "country_codes": {"gg"},
    },
    "Hong Kong": {
        "query_country": "Hong Kong",
        "country_codes": {"hk"},
    },
    "Hungary": {
        "query_country": "Hungary",
        "country_codes": {"hu"},
    },
    "Ireland": {
        "query_country": "Ireland",
        "country_codes": {"ie"},
    },
    "Italy": {
        "query_country": "Italy",
        "country_codes": {"it"},
    },
    "Japan": {
        "query_country": "Japan",
        "country_codes": {"jp"},
    },
    "Jersey": {
        "query_country": "Jersey",
        "country_codes": {"je"},
    },
    "New Zealand": {
        "query_country": "New Zealand",
        "country_codes": {"nz"},
    },
    "Norway": {
        "query_country": "Norway",
        "country_codes": {"no"},
    },
    "Peru": {
        "query_country": "Peru",
        "country_codes": {"pe"},
    },
    "Poland": {
        "query_country": "Poland",
        "country_codes": {"pl"},
    },
    "Qatar": {
        "query_country": "Qatar",
        "country_codes": {"qa"},
    },
    "Saudi Arabia": {
        "query_country": "Saudi Arabia",
        "country_codes": {"sa"},
    },
    "Singapore": {
        "query_country": "Singapore",
        "country_codes": {"sg"},
    },
    "South Africa": {
        "query_country": "South Africa",
        "country_codes": {"za"},
    },
    "South Korea": {
        "query_country": "South Korea",
        "country_codes": {"kr"},
    },
    "Spain": {
        "query_country": "Spain",
        "country_codes": {"es"},
    },
    "Sweden": {
        "query_country": "Sweden",
        "country_codes": {"se"},
    },
    "Switzerland": {
        "query_country": "Switzerland",
        "country_codes": {"ch"},
    },
    "Turkey": {
        "query_country": "Turkey",
        "country_codes": {"tr"},
    },
    "United Arab Emirates": {
        "query_country": "United Arab Emirates",
        "country_codes": {"ae"},
    },
    "United States": {
        "query_country": "United States",
        "country_codes": {"us"},
    },
    "Uruguay": {
        "query_country": "Uruguay",
        "country_codes": {"uy"},
    },
}

observed_jurisdictions = set(
    course_locations["candidate_jurisdiction"].dropna().unique()
)
mapped_jurisdictions = set(JURISDICTION_GEOCODING_RULES)

print("Observed jurisdictions:", len(observed_jurisdictions))
print("Mapped jurisdictions:", len(mapped_jurisdictions))
print("Missing mappings:", sorted(observed_jurisdictions - mapped_jurisdictions))
print("Unused mappings:", sorted(mapped_jurisdictions - observed_jurisdictions))

Observed jurisdictions: 36
Mapped jurisdictions: 36
Missing mappings: []
Unused mappings: []


## Geocoding query strategy

The first geocoding pass will use a conservative venue-oriented query:

`<candidate course label> racecourse, <mapped country or territory>`

The exact query text will be preserved in the cache.

The provider response will not be accepted automatically. Results will be
reviewed against:

- returned country code;
- venue name;
- address;
- place category and type;
- coordinate plausibility;
- alternative returned matches.

A small test batch will be used before any wider geocoding run.

In [12]:
# Construct the exact first-pass geocoding query for every course identity.
#
# This cell does not send any API requests or create any cache files.
# It only prepares the query text that would later be sent and confirms that:
# - every course has a query;
# - queries are unique by full candidate identity;
# - cross-jurisdiction labels produce different country-qualified queries.

geocoding_candidates = course_locations[
    [
        "candidate_course_label",
        "candidate_jurisdiction",
        "raw_course_labels",
        "provisional_races",
    ]
].copy()

geocoding_candidates["query_country"] = (
    geocoding_candidates["candidate_jurisdiction"]
    .map(
        lambda jurisdiction: (
            JURISDICTION_GEOCODING_RULES[jurisdiction]["query_country"]
        )
    )
)

geocoding_candidates["geocoding_query"] = (
    geocoding_candidates["candidate_course_label"]
    + " racecourse, "
    + geocoding_candidates["query_country"]
)

print("Candidate identities:", len(geocoding_candidates))
print(
    "Blank queries:",
    geocoding_candidates["geocoding_query"].isna().sum(),
)
print(
    "Distinct queries:",
    geocoding_candidates["geocoding_query"].nunique(),
)

sample_labels = [
    "Ascot",
    "Newcastle",
    "Sandown",
    "Happy Valley",
    "San Isidro",
]

display(
    geocoding_candidates.loc[
        geocoding_candidates["candidate_course_label"].isin(sample_labels),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "geocoding_query",
        ],
    ].sort_values(
        ["candidate_course_label", "candidate_jurisdiction"]
    )
)

Candidate identities: 394
Blank queries: 0
Distinct queries: 394


,candidate_course_label,candidate_jurisdiction,geocoding_query
6,Ascot,Australia,"Ascot racecourse, Australia"
164,Ascot,Great Britain,"Ascot racecourse, United Kingdom"
229,Happy Valley,Hong Kong,"Happy Valley racecourse, Hong Kong"
36,Newcastle,Australia,"Newcastle racecourse, Australia"
199,Newcastle,Great Britain,"Newcastle racecourse, United Kingdom"
2,San Isidro,Argentina,"San Isidro racecourse, Argentina"
43,Sandown,Australia,"Sandown racecourse, Australia"
211,Sandown,Great Britain,"Sandown racecourse, United Kingdom"


## Geocoding cache structure

The cache will use two files with different purposes.

### Query manifest

`data/cache/course_geocoding_cache.csv`

This will contain one row per exact provider request, including unsuccessful
and ambiguous searches. Its key will be:

`provider + candidate_course_label + candidate_jurisdiction + exact_query`

The manifest will record:

- candidate identity;
- exact query;
- provider and request parameters;
- request timestamp;
- request outcome;
- number of returned results;
- selected provider result identifier, where applicable;
- review status and notes;
- a reference to the corresponding raw-response record.

### Raw response archive

`data/cache/course_geocoding_responses.jsonl`

This will preserve the complete provider response for each request as one JSON
object per line.

The CSV will support simple inspection and duplicate prevention. The JSONL
archive will preserve nested provider data without flattening or loss.

In [13]:
# Define the columns required for the geocoding query manifest.
#
# This cell does not create or modify any files and sends no API requests.
# It establishes the audit fields that every attempted lookup must preserve.
#
# The manifest separates:
# - the candidate course identity;
# - the exact provider request;
# - the request outcome;
# - any selected result;
# - later validation and review decisions.

GEOCODING_CACHE_COLUMNS = [
    "cache_record_id",
    "provider",
    "candidate_course_label",
    "candidate_jurisdiction",
    "exact_query",
    "country_code_filter",
    "result_limit",
    "requested_at_utc",
    "request_status",
    "http_status",
    "result_count",
    "selected_result_index",
    "selected_provider_place_id",
    "selected_display_name",
    "selected_latitude",
    "selected_longitude",
    "country_code_match",
    "venue_type_match",
    "name_match_status",
    "review_status",
    "review_notes",
    "raw_response_record_id",
]

print("Geocoding cache columns:", len(GEOCODING_CACHE_COLUMNS))

for position, column in enumerate(GEOCODING_CACHE_COLUMNS, start=1):
    print(f"{position:2}. {column}")

    

Geocoding cache columns: 22
 1. cache_record_id
 2. provider
 3. candidate_course_label
 4. candidate_jurisdiction
 5. exact_query
 6. country_code_filter
 7. result_limit
 8. requested_at_utc
 9. request_status
10. http_status
11. result_count
12. selected_result_index
13. selected_provider_place_id
14. selected_display_name
15. selected_latitude
16. selected_longitude
17. country_code_match
18. venue_type_match
19. name_match_status
20. review_status
21. review_notes
22. raw_response_record_id


In [14]:
# Create deterministic identifiers for geocoding requests and raw responses.
#
# A SHA-256 digest is derived from the provider, full candidate identity,
# exact query, country filter and result limit.
#
# Using request content rather than row position means:
# - restarting the notebook produces the same identifier;
# - the same request can be detected before contacting the provider;
# - course-label collisions remain separate;
# - changing the query or request parameters creates a new cache record.
#
# This cell does not create files or send external requests.

import hashlib
import json


def build_geocoding_cache_record_id(
    *,
    provider,
    candidate_course_label,
    candidate_jurisdiction,
    exact_query,
    country_code_filter,
    result_limit,
):
    """Return a stable identifier for one exact geocoding request."""

    request_identity = {
        "provider": provider,
        "candidate_course_label": candidate_course_label,
        "candidate_jurisdiction": candidate_jurisdiction,
        "exact_query": exact_query,
        "country_code_filter": country_code_filter,
        "result_limit": result_limit,
    }

    canonical_json = json.dumps(
        request_identity,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


test_record_ids = []

for _, row in geocoding_candidates.head(3).iterrows():
    country_code = next(
        iter(
            JURISDICTION_GEOCODING_RULES[
                row["candidate_jurisdiction"]
            ]["country_codes"]
        )
    )

    test_record_ids.append(
        build_geocoding_cache_record_id(
            provider="nominatim",
            candidate_course_label=row["candidate_course_label"],
            candidate_jurisdiction=row["candidate_jurisdiction"],
            exact_query=row["geocoding_query"],
            country_code_filter=country_code,
            result_limit=5,
        )
    )

print("Test identifiers:", len(test_record_ids))
print("Distinct identifiers:", len(set(test_record_ids)))

for record_id in test_record_ids:
    print(record_id)

Test identifiers: 3
Distinct identifiers: 3
220c07f19395afd0935729df72037fc812d49e8a8db26bb701e6e9589d23be66
954f6813392a7e79ba128eab014b498af8acb4ac0cf45b497fd9bc326f783f2b
3973806340927b704c2b88fdb68bd794386962bfcce3b2c9adcaf224d0c9241e


## Reuse of accepted course locations

Racecourses are treated as stable physical venues.

Before preparing or sending any geocoding request, the workflow must check the
existing course-location reference.

A candidate course identity requires no further geocoding when it already has:

- an accepted validation status;
- a physical venue name;
- valid latitude and longitude;
- a valid IANA timezone.

The accepted reference takes precedence over the request cache. This prevents
repeat lookups even when the original provider query or response cache is not
available.

A new external request is permitted only when the course is:

- unassigned;
- unresolved;
- ambiguous;
- previously rejected without an accepted alternative;
- explicitly marked for refresh because the stored evidence is defective.

The cache prevents duplicate requests. The completed course-location reference
prevents unnecessary requests permanently.

In [15]:
# Classify whether each course identity already has a reusable accepted location.
#
# A course is considered complete only when it has:
# - an accepted validation status;
# - a nonblank physical venue name;
# - valid numeric coordinates;
# - a nonblank IANA timezone.
#
# This cell is read-only. It does not create cache files, modify the reference,
# or send any external requests.

ACCEPTED_LOCATION_STATUSES = {
    "automatically_validated",
    "manually_validated",
    "validated",
}


def nonblank_text(series):
    """Return True where values contain non-whitespace text."""

    return (
        series.notna()
        & series.astype(str).str.strip().ne("")
    )


latitude_values = pd.to_numeric(
    course_locations["latitude"],
    errors="coerce",
)
longitude_values = pd.to_numeric(
    course_locations["longitude"],
    errors="coerce",
)

course_locations["has_reusable_location"] = (
    course_locations["location_validation_status"].isin(
        ACCEPTED_LOCATION_STATUSES
    )
    & nonblank_text(course_locations["physical_venue_name"])
    & latitude_values.between(-90, 90)
    & longitude_values.between(-180, 180)
    & nonblank_text(course_locations["iana_timezone"])
)

print(
    "Reusable accepted locations:",
    int(course_locations["has_reusable_location"].sum()),
)
print(
    "Course identities requiring processing:",
    int((~course_locations["has_reusable_location"]).sum()),
)

display(
    course_locations[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "location_validation_status",
            "has_reusable_location",
        ]
    ]
    .groupby(
        ["location_validation_status", "has_reusable_location"],
        dropna=False,
    )
    .size()
    .rename("candidate_courses")
    .reset_index()
)

Reusable accepted locations: 36
Course identities requiring processing: 358


,location_validation_status,has_reusable_location,candidate_courses
0,automatically_validated,True,34
1,manually_validated,True,2
2,unassigned,False,355
3,NaN,False,3


In [16]:
# Define the Nominatim provider configuration.
#
# This cell sends no external requests.
#
# The public Nominatim service requires:
# - an application-specific User-Agent;
# - a maximum rate of one request per second;
# - local caching of all responses;
# - single-threaded use for a small one-off batch.
#
# The repository URL identifies the application without exposing personal data.
# Country codes will be supplied as hard result filters, while multiple results
# are retained so that the first match is never accepted automatically.

NOMINATIM_PROVIDER = "nominatim"
NOMINATIM_DOMAIN = "nominatim.openstreetmap.org"
NOMINATIM_USER_AGENT = (
    "inside-rails-course-location-mapping/"
    "0.1 "
    "(https://github.com/rjmac22/inside-rails-horse-racing)"
)

NOMINATIM_RESULT_LIMIT = 5
NOMINATIM_MINIMUM_DELAY_SECONDS = 1.1
NOMINATIM_LANGUAGE = "en"

NOMINATIM_REQUEST_OPTIONS = {
    "addressdetails": True,
    "extratags": True,
    "namedetails": True,
    "exactly_one": False,
    "limit": NOMINATIM_RESULT_LIMIT,
    "language": NOMINATIM_LANGUAGE,
}

print("Provider:", NOMINATIM_PROVIDER)
print("Domain:", NOMINATIM_DOMAIN)
print("User-Agent:", NOMINATIM_USER_AGENT)
print("Result limit:", NOMINATIM_RESULT_LIMIT)
print("Minimum delay:", NOMINATIM_MINIMUM_DELAY_SECONDS)
print("Request options:", NOMINATIM_REQUEST_OPTIONS)

Provider: nominatim
Domain: nominatim.openstreetmap.org
User-Agent: inside-rails-course-location-mapping/0.1 (https://github.com/rjmac22/inside-rails-horse-racing)
Result limit: 5
Minimum delay: 1.1
Request options: {'addressdetails': True, 'extratags': True, 'namedetails': True, 'exactly_one': False, 'limit': 5, 'language': 'en'}


## Physical venue identity and historical validity

A racecourse label is not assumed to identify one permanent physical site for
all historical periods.

The model distinguishes:

- the source-facing course label;
- the jurisdiction;
- the physical venue;
- the period during which that venue identity is valid.

A course that relocates is treated as a different physical venue record even
when the public course name remains unchanged.

For the current 2015-present source population, one physical venue is expected
to cover most course identities. However, accepted locations must remain
reviewable against `earliest_date` and `latest_date` so that later historical
expansion does not incorrectly apply modern coordinates to an earlier site.

The long-term reusable key may therefore require:

`candidate_course_label + candidate_jurisdiction + valid_from + valid_to`

rather than assuming that a course label has one eternal location.

## Physical venues and course configurations

The model distinguishes between a physical venue and the racing configuration
used at that venue.

A relocation creates a new physical venue record because the coordinates,
local environment and potentially the timezone have changed.

A material layout or surface change creates a new course-configuration record
even when the physical venue remains unchanged.

Potential configuration boundaries include:

- a change between turf, dirt, synthetic or another surface;
- replacement with a materially different synthetic surface;
- major track rebuilding or rerouting;
- a change in racing direction;
- material changes to circumference, bends, straight length, starts or finish;
- distinct inner, outer, hurdles or chase layouts where these affect comparison.

Minor maintenance, temporary rail movements, weather, going and meeting-level
conditions do not by themselves create a new permanent configuration. They
remain race or meeting attributes.

The long-term structure should therefore separate:

`course label -> physical venue -> dated course configuration`

Later course-comparison work can add dated configuration identities without
repeating the location lookup unless the change also affects the physical
venue, coordinates, timezone, or validity of the existing location evidence.

In [17]:
# Load existing geocoding cache files when they are present.
#
# This cell is read-only:
# - it does not create either cache file;
# - it does not modify the course-location reference;
# - it does not send any external requests.
#
# An absent manifest is represented by an empty DataFrame with the agreed
# schema. This lets later cells use the same logic on both first and resumed
# notebook runs.

if geocoding_cache_path.exists():
    geocoding_cache = pd.read_csv(geocoding_cache_path)
else:
    geocoding_cache = pd.DataFrame(
        columns=GEOCODING_CACHE_COLUMNS
    )

if geocoding_raw_responses_path.exists():
    raw_response_line_count = sum(
        1
        for line in geocoding_raw_responses_path.open(
            "r",
            encoding="utf-8",
        )
        if line.strip()
    )
else:
    raw_response_line_count = 0

missing_cache_columns = [
    column
    for column in GEOCODING_CACHE_COLUMNS
    if column not in geocoding_cache.columns
]

print("Manifest rows:", len(geocoding_cache))
print("Raw-response records:", raw_response_line_count)
print("Missing manifest columns:", missing_cache_columns)

Manifest rows: 76
Raw-response records: 76
Missing manifest columns: []


In [18]:
# Create the geocoding client and enforce the provider delay centrally.
#
# This cell does not contact Nominatim.
#
# All later lookups must use `rate_limited_geocode` rather than calling the
# geocoder directly. This ensures that every request observes the configured
# minimum delay and remains single-threaded.
#
# Raw result dictionaries are requested so that provider names, addresses,
# place identifiers, classifications and coordinates can be preserved in the
# audit cache.

from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim


nominatim_geocoder = Nominatim(
    user_agent=NOMINATIM_USER_AGENT,
    domain=NOMINATIM_DOMAIN,
    timeout=20,
)

rate_limited_geocode = RateLimiter(
    nominatim_geocoder.geocode,
    min_delay_seconds=NOMINATIM_MINIMUM_DELAY_SECONDS,
    max_retries=1,
    error_wait_seconds=5,
    swallow_exceptions=False,
)

print("Geocoder class:", type(nominatim_geocoder).__name__)
print("Provider domain:", nominatim_geocoder.domain)
print("Minimum delay:", NOMINATIM_MINIMUM_DELAY_SECONDS)
print("Client created without sending a request.")

Geocoder class: Nominatim
Provider domain: nominatim.openstreetmap.org
Minimum delay: 1.1
Client created without sending a request.


In [19]:
# Prepare one controlled geocoding test without sending the request.
#
# La Plata is used because it is the first scaffold identity and should have a
# clearly identifiable racing venue. The cell:
# - retrieves the full candidate identity;
# - applies the jurisdiction country-code rule;
# - generates the deterministic cache identifier;
# - checks the completed reference and request cache;
# - prints the exact request that the next cell would send.
#
# No external request is made and no file is created or modified.

test_candidate = geocoding_candidates.iloc[0]

test_jurisdiction = test_candidate["candidate_jurisdiction"]
test_country_code = sorted(
    JURISDICTION_GEOCODING_RULES[test_jurisdiction]["country_codes"]
)[0]

test_cache_record_id = build_geocoding_cache_record_id(
    provider=NOMINATIM_PROVIDER,
    candidate_course_label=test_candidate["candidate_course_label"],
    candidate_jurisdiction=test_jurisdiction,
    exact_query=test_candidate["geocoding_query"],
    country_code_filter=test_country_code,
    result_limit=NOMINATIM_RESULT_LIMIT,
)

test_reference_row = course_locations.loc[
    (course_locations["candidate_course_label"]
     == test_candidate["candidate_course_label"])
    & (course_locations["candidate_jurisdiction"]
       == test_jurisdiction)
].iloc[0]

test_is_reusable = bool(
    test_reference_row["has_reusable_location"]
)

test_is_cached = bool(
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(test_cache_record_id)
    .any()
)

print("Candidate:", test_candidate["candidate_course_label"])
print("Jurisdiction:", test_jurisdiction)
print("Exact query:", test_candidate["geocoding_query"])
print("Country-code filter:", test_country_code)
print("Result limit:", NOMINATIM_RESULT_LIMIT)
print("Cache record ID:", test_cache_record_id)
print("Reusable accepted location:", test_is_reusable)
print("Exact request already cached:", test_is_cached)
print("Eligible for test request:", not test_is_reusable and not test_is_cached)

Candidate: La Plata
Jurisdiction: Argentina
Exact query: La Plata racecourse, Argentina
Country-code filter: ar
Result limit: 5
Cache record ID: 220c07f19395afd0935729df72037fc812d49e8a8db26bb701e6e9589d23be66
Reusable accepted location: True
Exact request already cached: True
Eligible for test request: False


## Historical timezone interpretation

A physical venue may remain at the same coordinates while its legal civil-time
rules change.

The reference stores an IANA location-based timezone such as
`Europe/Istanbul` or `Europe/London`, not a fixed UTC offset.

The IANA timezone must be applied to each race date so that historical changes
to standard offsets and daylight-saving rules are reconstructed automatically.

For the current 2015-present population, the timezone derived from the venue
coordinates is expected to provide the required historical rules.

If substantially older records are added later, the timezone assignment must
be reviewed against the relevant historical validity period. Coordinates may
remain reusable while the applicable civil-time region or the reliability of
pre-1970 timezone evidence requires further validation.

In [20]:
# Send one controlled Nominatim test request for La Plata and cache the
# complete outcome, or reuse the same request when it is already cached.
#
# Exact request:
# - La Plata racecourse, Argentina
#
# This is the first, generic English-language query in the controlled
# La Plata test. It tests whether the provider can identify the racecourse
# directly from the source course label, a generic venue term and country.
#
# The request uses:
# - the rate-limited Nominatim client created earlier;
# - the Argentina country-code filter;
# - the fixed result limit defined for this notebook.
#
# Before sending anything, the cell builds the deterministic cache record ID
# from the provider and exact request parameters. This means the same request
# has the same identity on every notebook run.
#
# When the exact request is already present in the manifest:
# - no external request is sent;
# - the saved request status and result count are reused;
# - neither cache file receives a duplicate record.
#
# When the request is not already cached, the cell writes to:
# - data/cache/course_geocoding_cache.csv
# - data/cache/course_geocoding_responses.jsonl
#
# The manifest stores the searchable request summary and later review fields.
# The JSONL file preserves the complete raw provider response, including
# successful zero-result responses and request errors.
#
# This cell does not:
# - select a returned result;
# - infer that the first result is correct;
# - derive a timezone;
# - update data/reference/course_locations.csv.

generic_query = test_candidate["geocoding_query"]

# Build the stable identity for this exact provider request. Any material
# change to the query, country filter or result limit creates a different
# cache record rather than overwriting this request.
generic_cache_record_id = build_geocoding_cache_record_id(
    provider=NOMINATIM_PROVIDER,
    candidate_course_label=test_candidate[
        "candidate_course_label"
    ],
    candidate_jurisdiction=test_jurisdiction,
    exact_query=generic_query,
    country_code_filter=test_country_code,
    result_limit=NOMINATIM_RESULT_LIMIT,
)

generic_cache_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(generic_cache_record_id)
)

generic_is_cached = bool(generic_cache_mask.any())

if generic_is_cached:
    # Reuse the previously recorded outcome. This is the expected path when
    # the notebook is run again after the initial request.
    cached_generic_row = geocoding_cache.loc[
        generic_cache_mask
    ].iloc[0]

    request_status = cached_generic_row[
        "request_status"
    ]

    cached_result_count = pd.to_numeric(
        cached_generic_row["result_count"],
        errors="coerce",
    )

    result_count = (
        0
        if pd.isna(cached_result_count)
        else int(cached_result_count)
    )

    print("Request action: reused cached result")

else:
    # Record the request time before making the external call so the raw
    # response and manifest row share the same audit timestamp.
    requested_at_utc = datetime.now(
        timezone.utc
    ).isoformat()

    raw_response_record_id = (
        generic_cache_record_id
    )

    request_status = "pending"
    result_count = 0
    raw_results = []
    error_type = None
    error_message = None

    try:
        # Send exactly one request through the previously configured
        # RateLimiter. The country filter restricts results to Argentina.
        locations = rate_limited_geocode(
            generic_query,
            country_codes=test_country_code,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        # Preserve the provider's complete raw records rather than reducing
        # them immediately to selected fields.
        raw_results = [
            location.raw
            for location in locations
        ]

        result_count = len(raw_results)

        request_status = (
            "success_with_results"
            if result_count > 0
            else "success_no_results"
        )

    except Exception as exc:
        # Request failures are cached as outcomes as well. This prevents an
        # unnoticed retry loop and preserves the error for later inspection.
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_response_record = {
        "raw_response_record_id": (
            raw_response_record_id
        ),
        "cache_record_id": (
            generic_cache_record_id
        ),
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": (
            test_jurisdiction
        ),
        "exact_query": generic_query,
        "country_code_filter": (
            test_country_code
        ),
        "result_limit": (
            NOMINATIM_RESULT_LIMIT
        ),
        "requested_at_utc": (
            requested_at_utc
        ),
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    # Append the complete outcome as one immutable JSONL audit record.
    with geocoding_raw_responses_path.open(
        "a",
        encoding="utf-8",
    ) as raw_file:
        raw_file.write(
            json.dumps(
                raw_response_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    # Add the corresponding searchable manifest row. Result-selection fields
    # remain blank because this request cell does not perform review.
    manifest_row = {
        "cache_record_id": (
            generic_cache_record_id
        ),
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": (
            test_jurisdiction
        ),
        "exact_query": generic_query,
        "country_code_filter": (
            test_country_code
        ),
        "result_limit": (
            NOMINATIM_RESULT_LIMIT
        ),
        "requested_at_utc": (
            requested_at_utc
        ),
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": result_count,
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": (
            raw_response_record_id
        ),
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

    print("Request action: sent and cached")

print("Exact query:", generic_query)
print("Request status:", request_status)
print("Returned results:", result_count)
print("Manifest rows:", len(geocoding_cache))

Request action: reused cached result
Exact query: La Plata racecourse, Argentina
Request status: success_no_results
Returned results: 0
Manifest rows: 76


## Fallback query strategy

A successful API request with no results does not make the course unresolved.
It means only that the exact query formulation failed.

For each unresolved course, geocoding may proceed through an ordered set of
distinct cached queries:

1. `<course label> racecourse, <country>`
2. `<course label>, <country>`
3. a jurisdiction-appropriate racing-venue term, such as:
   - `Hipódromo` for Spanish- and Portuguese-speaking jurisdictions;
   - `Hippodrome` for relevant French-language searches;
   - `Rennbahn` for German-language searches;
4. a manually supplied known venue name or alias where automated formulations
   remain unsuccessful.

Every formulation is a separate cached request with its own deterministic
identifier and raw response.

An empty result is never overwritten or silently retried. The next fallback
query is attempted only because it is materially different from the cached
failed formulation.

In [21]:
# Define ordered venue-search terms for each jurisdiction.
#
# The first term remains the generic English "racecourse".
# Additional terms reflect common local naming conventions and are used only
# when an earlier exact query has already been cached with no usable result.
#
# This cell does not send external requests or modify any files.

JURISDICTION_VENUE_TERMS = {
    "Argentina": ["racecourse", "hipódromo", "hipodromo"],
    "Australia": ["racecourse"],
    "Bahrain": ["racecourse"],
    "Belgium": ["racecourse", "hippodrome"],
    "Brazil": ["racecourse", "hipódromo", "hipodromo"],
    "Canada": ["racecourse", "racetrack"],
    "Chile": ["racecourse", "hipódromo", "hipodromo"],
    "China": ["racecourse"],
    "Czech Republic": ["racecourse", "závodiště"],
    "Denmark": ["racecourse", "galopbane"],
    "France": ["racecourse", "hippodrome"],
    "Germany": ["racecourse", "rennbahn"],
    "Great Britain": ["racecourse"],
    "Guernsey": ["racecourse"],
    "Hong Kong": ["racecourse"],
    "Hungary": ["racecourse", "versenypálya"],
    "Ireland": ["racecourse"],
    "Italy": ["racecourse", "ippodromo"],
    "Japan": ["racecourse"],
    "Jersey": ["racecourse"],
    "New Zealand": ["racecourse"],
    "Norway": ["racecourse", "galoppbane"],
    "Peru": ["racecourse", "hipódromo", "hipodromo"],
    "Poland": ["racecourse", "tor wyścigów konnych"],
    "Qatar": ["racecourse"],
    "Saudi Arabia": ["racecourse"],
    "Singapore": ["racecourse"],
    "South Africa": ["racecourse"],
    "South Korea": ["racecourse"],
    "Spain": ["racecourse", "hipódromo", "hipodromo"],
    "Sweden": ["racecourse", "galoppbana"],
    "Switzerland": ["racecourse", "hippodrome", "rennbahn"],
    "Turkey": ["racecourse", "hipodromu"],
    "United Arab Emirates": ["racecourse"],
    "United States": ["racecourse", "racetrack"],
    "Uruguay": ["racecourse", "hipódromo", "hipodromo"],
}

missing_term_rules = sorted(
    observed_jurisdictions - set(JURISDICTION_VENUE_TERMS)
)

empty_term_rules = sorted(
    jurisdiction
    for jurisdiction, terms in JURISDICTION_VENUE_TERMS.items()
    if not terms
)

print("Jurisdictions with venue terms:", len(JURISDICTION_VENUE_TERMS))
print("Missing venue-term rules:", missing_term_rules)
print("Empty venue-term rules:", empty_term_rules)
print("Argentina terms:", JURISDICTION_VENUE_TERMS["Argentina"])

Jurisdictions with venue terms: 36
Missing venue-term rules: []
Empty venue-term rules: []
Argentina terms: ['racecourse', 'hipódromo', 'hipodromo']


In [22]:
# Generate the ordered fallback queries for the La Plata test identity.
#
# This cell sends no external requests and modifies no files.
#
# Each materially different query receives its own deterministic cache ID.
# A query is eligible only when:
# - the exact request is not already cached; and
# - the course does not already have a reusable accepted location.
#
# `not test_is_reusable` is calculated once because it is a single Python
# boolean, while `~already_cached` operates element-by-element on a Series.

test_venue_terms = JURISDICTION_VENUE_TERMS[test_jurisdiction]

test_fallback_queries = pd.DataFrame(
    {
        "venue_term": test_venue_terms,
        "exact_query": [
            (
                f"{test_candidate['candidate_course_label']} "
                f"{venue_term}, "
                f"{test_candidate['query_country']}"
            )
            for venue_term in test_venue_terms
        ],
    }
)

test_fallback_queries["cache_record_id"] = (
    test_fallback_queries.apply(
        lambda row: build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=(
                test_candidate["candidate_course_label"]
            ),
            candidate_jurisdiction=test_jurisdiction,
            exact_query=row["exact_query"],
            country_code_filter=test_country_code,
            result_limit=NOMINATIM_RESULT_LIMIT,
        ),
        axis=1,
    )
)

cached_record_ids = set(
    geocoding_cache["cache_record_id"].astype(str)
)

test_fallback_queries["already_cached"] = (
    test_fallback_queries["cache_record_id"].isin(
        cached_record_ids
    )
)

reference_requires_lookup = not test_is_reusable

test_fallback_queries["eligible_for_request"] = (
    ~test_fallback_queries["already_cached"]
    & reference_requires_lookup
)

display(
    test_fallback_queries[
        [
            "venue_term",
            "exact_query",
            "already_cached",
            "eligible_for_request",
        ]
    ]
)

,venue_term,exact_query,already_cached,eligible_for_request
0,racecourse,"La Plata racecourse, Argentina",True,False
1,hipódromo,"La Plata hipódromo, Argentina",True,False
2,hipodromo,"La Plata hipodromo, Argentina",True,False


In [23]:
# Send or reuse the accented local-term request for La Plata.
#
# This cell is safe to rerun:
# - when the exact request exists in the cache, it reuses the saved outcome;
# - otherwise it sends exactly one rate-limited request and caches it.
#
# It does not select or accept a course location.

fallback_request = (
    test_fallback_queries.loc[
        test_fallback_queries["venue_term"].eq("hipódromo")
    ]
    .iloc[0]
)

fallback_query = fallback_request["exact_query"]
fallback_cache_record_id = fallback_request["cache_record_id"]

fallback_cache_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(fallback_cache_record_id)
)

fallback_is_cached = bool(fallback_cache_mask.any())

if fallback_is_cached:
    cached_fallback_row = geocoding_cache.loc[
        fallback_cache_mask
    ].iloc[0]

    request_status = cached_fallback_row["request_status"]

    cached_result_count = pd.to_numeric(
        cached_fallback_row["result_count"],
        errors="coerce",
    )

    result_count = (
        0
        if pd.isna(cached_result_count)
        else int(cached_result_count)
    )

    print("Request action: reused cached result")

else:
    requested_at_utc = datetime.now(timezone.utc).isoformat()
    raw_response_record_id = fallback_cache_record_id

    request_status = "pending"
    result_count = 0
    raw_results = []
    error_type = None
    error_message = None

    try:
        locations = rate_limited_geocode(
            fallback_query,
            country_codes=test_country_code,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        raw_results = [
            location.raw
            for location in locations
        ]

        result_count = len(raw_results)
        request_status = (
            "success_with_results"
            if result_count > 0
            else "success_no_results"
        )

    except Exception as exc:
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_response_record = {
        "raw_response_record_id": raw_response_record_id,
        "cache_record_id": fallback_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": test_jurisdiction,
        "exact_query": fallback_query,
        "country_code_filter": test_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    with geocoding_raw_responses_path.open(
        "a",
        encoding="utf-8",
    ) as raw_file:
        raw_file.write(
            json.dumps(
                raw_response_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    manifest_row = {
        "cache_record_id": fallback_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": test_jurisdiction,
        "exact_query": fallback_query,
        "country_code_filter": test_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": result_count,
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": raw_response_record_id,
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

    print("Request action: sent and cached")

print("Exact query:", fallback_query)
print("Request status:", request_status)
print("Returned results:", result_count)
print("Manifest rows:", len(geocoding_cache))

Request action: reused cached result
Exact query: La Plata hipódromo, Argentina
Request status: success_with_results
Returned results: 3
Manifest rows: 76


In [24]:
# Read the cached raw response for the successful La Plata fallback and present
# the returned alternatives in a compact review table.
#
# This cell is read-only:
# - it sends no external request;
# - it does not modify either cache file;
# - it does not update course_locations.csv.
#
# The table preserves the provider result order and shows the fields needed to
# judge whether a result is genuinely the racecourse rather than merely a place
# elsewhere in La Plata.

fallback_raw_record = None

with geocoding_raw_responses_path.open(
    "r",
    encoding="utf-8",
) as raw_file:
    for line in raw_file:
        if not line.strip():
            continue

        record = json.loads(line)

        if (
            record.get("raw_response_record_id")
            == fallback_cache_record_id
        ):
            fallback_raw_record = record
            break

if fallback_raw_record is None:
    raise RuntimeError(
        "The cached raw response for the fallback request was not found."
    )

review_rows = []

for result_index, result in enumerate(
    fallback_raw_record["results"]
):
    address = result.get("address", {})

    review_rows.append(
        {
            "result_index": result_index,
            "display_name": result.get("display_name"),
            "latitude": result.get("lat"),
            "longitude": result.get("lon"),
            "osm_type": result.get("osm_type"),
            "osm_id": result.get("osm_id"),
            "category": result.get("category"),
            "type": result.get("type"),
            "name": result.get("name"),
            "city_or_locality": (
                address.get("city")
                or address.get("town")
                or address.get("municipality")
                or address.get("village")
            ),
            "state_or_region": (
                address.get("state")
                or address.get("province")
            ),
            "country": address.get("country"),
            "country_code": address.get("country_code"),
        }
    )

fallback_review = pd.DataFrame(review_rows)

display(fallback_review)

,result_index,display_name,latitude,longitude,osm_type,osm_id,category,type,name,city_or_locality,state_or_region,country,country_code
0,0,"Virreinato del Río de la Plata, Hipódromo, Gua...",-33.0202287,-58.5295775,way,126254630,None,residential,Virreinato del Río de la Plata,Gualeguaychu,Entre Ríos Province,Argentina,ar
1,1,"Haras La Madrugada, Hipódromo, Mar del Plata, ...",-37.9591046,-57.6542732,way,127014198,None,residential,Haras La Madrugada,Mar del Plata,Buenos Aires,Argentina,ar
2,2,"Haras La Biznaga, Hipódromo, Mar del Plata, Pa...",-37.9596086,-57.6536090,way,127014197,None,residential,Haras La Biznaga,Mar del Plata,Buenos Aires,Argentina,ar


In [25]:
# Mark the successful fallback request as reviewed with no usable result.
#
# This cell updates only the geocoding manifest:
# - no external request is sent;
# - the raw provider response remains unchanged;
# - no course location is accepted into course_locations.csv.
#
# The request itself succeeded, but every returned candidate is geographically
# or semantically wrong for the La Plata racecourse. The review therefore
# records that a more precise venue name or alias is required.

fallback_manifest_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(fallback_cache_record_id)
)

matching_manifest_rows = int(
    fallback_manifest_mask.sum()
)

if matching_manifest_rows != 1:
    raise RuntimeError(
        "Expected exactly one manifest row for the fallback request, "
        f"found {matching_manifest_rows}."
    )

geocoding_cache.loc[
    fallback_manifest_mask,
    "review_status",
] = "reviewed_no_usable_result"

geocoding_cache.loc[
    fallback_manifest_mask,
    "review_notes",
] = (
    "All three returned results rejected: one residential feature in "
    "Gualeguaychu and two residential features in Mar del Plata. "
    "None is the La Plata racecourse or a racing venue in La Plata."
)

geocoding_cache.to_csv(
    geocoding_cache_path,
    index=False,
)

display(
    geocoding_cache.loc[
        fallback_manifest_mask,
        [
            "exact_query",
            "request_status",
            "result_count",
            "review_status",
            "review_notes",
        ],
    ]
)

,exact_query,request_status,result_count,review_status,review_notes
1,"La Plata hipódromo, Argentina",success_with_results,3,reviewed_no_usable_result,All three returned results rejected: one resid...


In [26]:
# Prepare a precise manual-alias query for the unresolved La Plata course.
#
# The generic and local-term formulations have already failed to identify the
# venue. This query uses the full local venue-name pattern instead:
# "Hipódromo de La Plata, Argentina".
#
# This cell:
# - generates a separate deterministic cache identifier;
# - confirms that the exact query is not already cached;
# - does not contact Nominatim;
# - does not modify either cache file or course_locations.csv.

manual_alias_query = "Hipódromo de La Plata, Argentina"

manual_alias_cache_record_id = build_geocoding_cache_record_id(
    provider=NOMINATIM_PROVIDER,
    candidate_course_label=test_candidate[
        "candidate_course_label"
    ],
    candidate_jurisdiction=test_jurisdiction,
    exact_query=manual_alias_query,
    country_code_filter=test_country_code,
    result_limit=NOMINATIM_RESULT_LIMIT,
)

manual_alias_is_cached = bool(
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(manual_alias_cache_record_id)
    .any()
)

manual_alias_is_eligible = (
    not test_is_reusable
    and not manual_alias_is_cached
)

print("Manual alias query:", manual_alias_query)
print("Cache record ID:", manual_alias_cache_record_id)
print("Exact request already cached:", manual_alias_is_cached)
print("Eligible for request:", manual_alias_is_eligible)

Manual alias query: Hipódromo de La Plata, Argentina
Cache record ID: f29ac5baf793331c1b0ec642ecdc882d744cd7daf87369ecd20a8ddf98ef08c7
Exact request already cached: True
Eligible for request: False


In [27]:
# Send one precise manual-alias Nominatim request for Hipódromo de La Plata,
# or reuse the same request when it is already present in the cache.
#
# Exact request:
# - Hipódromo de La Plata, Argentina
#
# This is the third query in the controlled La Plata test. It uses the known
# full venue name after:
# - the generic English query returned no results;
# - the broader local-term query returned unrelated residential alternatives.
#
# This is a manual alias, not an automatically inferred course name. The
# reason for using it must therefore remain visible in the notebook and in the
# later location-evidence text.
#
# The request uses:
# - the rate-limited Nominatim client created earlier;
# - the Argentina country-code filter;
# - the fixed result limit defined for this notebook.
#
# The deterministic cache record ID was prepared earlier from the exact
# provider request. It prevents duplicate external requests on later runs.
#
# When the exact request is already present in the manifest:
# - no external request is sent;
# - the saved request status and result count are reused;
# - neither cache file receives a duplicate record.
#
# When the request is not already cached, the cell writes to:
# - data/cache/course_geocoding_cache.csv
# - data/cache/course_geocoding_responses.jsonl
#
# The manifest stores the searchable request summary and later review fields.
# The JSONL file preserves the complete raw provider response, including
# successful zero-result responses and request errors.
#
# This cell does not:
# - automatically select result index 0 or any other result;
# - assume that an exact text match is geographically correct;
# - derive a timezone;
# - update data/reference/course_locations.csv.
#
# Candidate comparison and the manual selection of result index 1 occur in
# later review cells.

manual_alias_cache_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(manual_alias_cache_record_id)
)

manual_alias_is_cached = bool(
    manual_alias_cache_mask.any()
)

if manual_alias_is_cached:
    # Reuse the previously recorded provider outcome. This is the expected
    # path when the notebook is run again after the initial alias request.
    cached_manual_alias_row = geocoding_cache.loc[
        manual_alias_cache_mask
    ].iloc[0]

    request_status = cached_manual_alias_row[
        "request_status"
    ]

    cached_result_count = pd.to_numeric(
        cached_manual_alias_row["result_count"],
        errors="coerce",
    )

    result_count = (
        0
        if pd.isna(cached_result_count)
        else int(cached_result_count)
    )

    print("Request action: reused cached result")

else:
    # Record the request time before making the external call so the raw
    # response and manifest row share the same audit timestamp.
    requested_at_utc = datetime.now(
        timezone.utc
    ).isoformat()

    raw_response_record_id = (
        manual_alias_cache_record_id
    )

    request_status = "pending"
    result_count = 0
    raw_results = []
    error_type = None
    error_message = None

    try:
        # Send exactly one request through the configured RateLimiter.
        # The country filter restricts provider results to Argentina.
        locations = rate_limited_geocode(
            manual_alias_query,
            country_codes=test_country_code,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        # Preserve the complete raw provider records. Review and candidate
        # selection happen later rather than inside the request cell.
        raw_results = [
            location.raw
            for location in locations
        ]

        result_count = len(raw_results)

        request_status = (
            "success_with_results"
            if result_count > 0
            else "success_no_results"
        )

    except Exception as exc:
        # Provider and network failures are cached as explicit outcomes.
        # This preserves the failure and prevents accidental repeated calls.
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_response_record = {
        "raw_response_record_id": (
            raw_response_record_id
        ),
        "cache_record_id": (
            manual_alias_cache_record_id
        ),
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": (
            test_jurisdiction
        ),
        "exact_query": manual_alias_query,
        "country_code_filter": (
            test_country_code
        ),
        "result_limit": (
            NOMINATIM_RESULT_LIMIT
        ),
        "requested_at_utc": (
            requested_at_utc
        ),
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    # Append the complete provider outcome as one immutable JSONL audit
    # record. Existing records are never replaced by this cell.
    with geocoding_raw_responses_path.open(
        "a",
        encoding="utf-8",
    ) as raw_file:
        raw_file.write(
            json.dumps(
                raw_response_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    # Add the searchable manifest row. All selection and validation fields
    # remain blank because this request has not yet been reviewed.
    manifest_row = {
        "cache_record_id": (
            manual_alias_cache_record_id
        ),
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": test_candidate[
            "candidate_course_label"
        ],
        "candidate_jurisdiction": (
            test_jurisdiction
        ),
        "exact_query": manual_alias_query,
        "country_code_filter": (
            test_country_code
        ),
        "result_limit": (
            NOMINATIM_RESULT_LIMIT
        ),
        "requested_at_utc": (
            requested_at_utc
        ),
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": result_count,
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": (
            raw_response_record_id
        ),
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

    print("Request action: sent and cached")

print("Exact query:", manual_alias_query)
print("Request status:", request_status)
print("Returned results:", result_count)
print("Manifest rows:", len(geocoding_cache))

Request action: reused cached result
Exact query: Hipódromo de La Plata, Argentina
Request status: success_with_results
Returned results: 2
Manifest rows: 76


In [28]:
# Read and display the cached alternatives returned for the precise manual alias.
#
# This cell is read-only:
# - it sends no external request;
# - it does not modify either cache file;
# - it does not update course_locations.csv.
#
# The provider order is retained so we can review the exact venue identity,
# coordinates, classification, locality and country before accepting anything.

manual_alias_raw_record = None

with geocoding_raw_responses_path.open(
    "r",
    encoding="utf-8",
) as raw_file:
    for line in raw_file:
        if not line.strip():
            continue

        record = json.loads(line)

        if (
            record.get("raw_response_record_id")
            == manual_alias_cache_record_id
        ):
            manual_alias_raw_record = record
            break

if manual_alias_raw_record is None:
    raise RuntimeError(
        "The cached raw response for the manual-alias request was not found."
    )

manual_alias_review_rows = []

for result_index, result in enumerate(
    manual_alias_raw_record["results"]
):
    address = result.get("address", {})

    manual_alias_review_rows.append(
        {
            "result_index": result_index,
            "display_name": result.get("display_name"),
            "latitude": result.get("lat"),
            "longitude": result.get("lon"),
            "osm_type": result.get("osm_type"),
            "osm_id": result.get("osm_id"),
            "category": result.get("category"),
            "type": result.get("type"),
            "name": result.get("name"),
            "city_or_locality": (
                address.get("city")
                or address.get("town")
                or address.get("municipality")
                or address.get("village")
            ),
            "state_or_region": (
                address.get("state")
                or address.get("province")
            ),
            "country": address.get("country"),
            "country_code": address.get("country_code"),
        }
    )

manual_alias_review = pd.DataFrame(
    manual_alias_review_rows
)

display(manual_alias_review)

,result_index,display_name,latitude,longitude,osm_type,osm_id,category,type,name,city_or_locality,state_or_region,country,country_code
0,0,"Hipódromo de La Plata, B° Palermo I, Salta, Ca...",-24.7904827,-65.4601147,way,171467555,None,tertiary,Hipódromo de La Plata,Salta,Salta,Argentina,ar
1,1,"Hipódromo de La Plata, Calle 121, La Plata, Pa...",-34.9012673,-57.9438043,way,287312802,None,sports_centre,Hipódromo de La Plata,La Plata,Buenos Aires,Argentina,ar


In [29]:
# Derive and validate the IANA timezone for the genuine La Plata result.
#
# Result index 1 has been identified as the plausible course because:
# - its name is Hipódromo de La Plata;
# - its locality is La Plata;
# - its region is Buenos Aires;
# - its provider type is sports_centre.
#
# This cell is read-only:
# - it sends no external request;
# - it does not modify the cache;
# - it does not update course_locations.csv.
#
# The timezone is derived from the selected coordinates and then validated
# through Python's ZoneInfo implementation.

from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo


selected_result_index = 1
selected_result = manual_alias_raw_record["results"][
    selected_result_index
]

selected_latitude = float(selected_result["lat"])
selected_longitude = float(selected_result["lon"])

timezone_finder = TimezoneFinder()

selected_iana_timezone = timezone_finder.timezone_at(
    lat=selected_latitude,
    lng=selected_longitude,
)

timezone_validation_error = None

try:
    ZoneInfo(selected_iana_timezone)
    timezone_is_valid = True
except Exception as exc:
    timezone_is_valid = False
    timezone_validation_error = str(exc)

selected_address = selected_result.get("address", {})

print("Selected result index:", selected_result_index)
print("Selected name:", selected_result.get("name"))
print("Locality:", selected_address.get("city"))
print("Region:", selected_address.get("state"))
print("Latitude:", selected_latitude)
print("Longitude:", selected_longitude)
print("Derived IANA timezone:", selected_iana_timezone)
print("Timezone validates:", timezone_is_valid)

if timezone_validation_error:
    print("Validation error:", timezone_validation_error)

Selected result index: 1
Selected name: Hipódromo de La Plata
Locality: La Plata
Region: Buenos Aires
Latitude: -34.9012673
Longitude: -57.9438043
Derived IANA timezone: America/Argentina/Buenos_Aires
Timezone validates: True


In [30]:
# Record the reviewed La Plata selection in the geocoding manifest.
#
# This cell updates only course_geocoding_cache.csv.
#
# It records:
# - the selected provider result;
# - provider identity and display name;
# - coordinates;
# - country, venue-type and name-match assessments;
# - the completed manual-review status.
#
# It does not modify course_locations.csv. The selected location and derived
# timezone will be validated together in a separate proposed-reference row
# before the final reference is changed.

manual_alias_manifest_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(manual_alias_cache_record_id)
)

matching_manifest_rows = int(
    manual_alias_manifest_mask.sum()
)

if matching_manifest_rows != 1:
    raise RuntimeError(
        "Expected exactly one manifest row for the manual-alias request, "
        f"found {matching_manifest_rows}."
    )

selected_provider_place_id = (
    f"{selected_result.get('osm_type')}:"
    f"{selected_result.get('osm_id')}"
)

selected_display_name = selected_result.get(
    "display_name"
)

selected_country_code = (
    selected_address.get("country_code", "")
    .strip()
    .lower()
)

country_code_match = (
    selected_country_code == test_country_code
)

venue_type_match = (
    selected_result.get("type")
    in {"sports_centre", "raceway", "stadium", "track"}
)

name_match_status = (
    "exact_local_venue_name"
    if selected_result.get("name")
    == "Hipódromo de La Plata"
    else "non_exact"
)

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "selected_result_index",
] = selected_result_index

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "selected_provider_place_id",
] = selected_provider_place_id

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "selected_display_name",
] = selected_display_name

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "selected_latitude",
] = selected_latitude

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "selected_longitude",
] = selected_longitude

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "country_code_match",
] = country_code_match

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "venue_type_match",
] = venue_type_match

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "name_match_status",
] = name_match_status

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "review_status",
] = "manually_selected"

geocoding_cache.loc[
    manual_alias_manifest_mask,
    "review_notes",
] = (
    "Selected result 1. Exact venue name in La Plata, Buenos Aires; "
    "sports_centre classification; Argentina country code; result 0 rejected "
    "as a similarly named road in Salta. Derived timezone: "
    f"{selected_iana_timezone}."
)

geocoding_cache.to_csv(
    geocoding_cache_path,
    index=False,
)

display(
    geocoding_cache.loc[
        manual_alias_manifest_mask,
        [
            "exact_query",
            "selected_result_index",
            "selected_provider_place_id",
            "selected_display_name",
            "selected_latitude",
            "selected_longitude",
            "country_code_match",
            "venue_type_match",
            "name_match_status",
            "review_status",
        ],
    ]
)

,exact_query,selected_result_index,selected_provider_place_id,selected_display_name,selected_latitude,selected_longitude,country_code_match,venue_type_match,name_match_status,review_status
2,"Hipódromo de La Plata, Argentina",1.0,way:287312802,"Hipódromo de La Plata, Calle 121, La Plata, Pa...",-34.901267,-57.943804,True,True,exact_local_venue_name,manually_selected


In [31]:
# Build and validate the proposed course-location reference row for La Plata.
#
# This cell is read-only:
# - it does not send an external request;
# - it does not modify the geocoding cache;
# - it does not write to course_locations.csv.
#
# Several currently blank reference columns were loaded by pandas as float64.
# Before assigning text values, this cell explicitly converts the relevant
# text columns to object dtype. Coordinate columns are explicitly numeric.

test_reference_mask = (
    course_locations["candidate_course_label"]
    .eq(test_candidate["candidate_course_label"])
    & course_locations["candidate_jurisdiction"]
    .eq(test_jurisdiction)
)

matching_reference_rows = int(test_reference_mask.sum())

if matching_reference_rows != 1:
    raise RuntimeError(
        "Expected exactly one La Plata reference row, "
        f"found {matching_reference_rows}."
    )

proposed_course_location = (
    course_locations.loc[test_reference_mask]
    .copy()
)

text_location_columns = [
    "physical_venue_name",
    "locality",
    "region",
    "country",
    "iana_timezone",
    "location_evidence",
    "location_validation_status",
]

for column in text_location_columns:
    proposed_course_location[column] = (
        proposed_course_location[column].astype("object")
    )

for column in ["latitude", "longitude"]:
    proposed_course_location[column] = pd.to_numeric(
        proposed_course_location[column],
        errors="coerce",
    )

proposed_course_location["physical_venue_name"] = (
    selected_result.get("name")
)
proposed_course_location["locality"] = (
    selected_address.get("city")
)
proposed_course_location["region"] = (
    selected_address.get("state")
)
proposed_course_location["country"] = (
    selected_address.get("country")
)
proposed_course_location["latitude"] = selected_latitude
proposed_course_location["longitude"] = selected_longitude
proposed_course_location["iana_timezone"] = (
    selected_iana_timezone
)
proposed_course_location["location_evidence"] = (
    f"Nominatim manual selection from query "
    f"'{manual_alias_query}'; "
    f"provider place {selected_provider_place_id}; "
    f"raw response {manual_alias_cache_record_id}."
)
proposed_course_location["location_validation_status"] = (
    "manually_validated"
)

proposed_latitude = pd.to_numeric(
    proposed_course_location["latitude"],
    errors="coerce",
)

proposed_longitude = pd.to_numeric(
    proposed_course_location["longitude"],
    errors="coerce",
)

validation_checks = {
    "one proposed row": len(proposed_course_location) == 1,
    "physical venue present": proposed_course_location[
        "physical_venue_name"
    ].notna().all(),
    "locality present": proposed_course_location[
        "locality"
    ].notna().all(),
    "country present": proposed_course_location[
        "country"
    ].notna().all(),
    "latitude valid": proposed_latitude.between(
        -90,
        90,
    ).all(),
    "longitude valid": proposed_longitude.between(
        -180,
        180,
    ).all(),
    "timezone valid": timezone_is_valid,
    "evidence present": proposed_course_location[
        "location_evidence"
    ].notna().all(),
    "accepted status": proposed_course_location[
        "location_validation_status"
    ].eq("manually_validated").all(),
}

display(
    proposed_course_location[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "physical_venue_name",
            "locality",
            "region",
            "country",
            "latitude",
            "longitude",
            "iana_timezone",
            "location_evidence",
            "location_validation_status",
        ]
    ]
)

print()
for check_name, passed in validation_checks.items():
    print(f"{check_name}: {bool(passed)}")

print(
    "All proposed-row checks pass:",
    all(validation_checks.values()),
)

,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_evidence,location_validation_status
0,La Plata,Argentina,Hipódromo de La Plata,La Plata,Buenos Aires,Argentina,-34.901267,-57.943804,America/Argentina/Buenos_Aires,Nominatim manual selection from query 'Hipódro...,manually_validated



one proposed row: True
physical venue present: True
locality present: True
country present: True
latitude valid: True
longitude valid: True
timezone valid: True
evidence present: True
accepted status: True
All proposed-row checks pass: True


In [32]:
# Permanently update the La Plata course-location reference.
#
# This cell writes to:
# - data/reference/course_locations.csv
#
# It changes only the existing identity:
# - candidate_course_label = La Plata
# - candidate_jurisdiction = Argentina
#
# All other rows and source-coverage fields remain unchanged.
#
# The repository uses a `src` package layout. Because this notebook is running
# directly rather than through an installed package, the repository's `src`
# directory is added to sys.path before importing the project validator.
#
# The cell is safe to rerun:
# - one changed row means the validated La Plata reference is written;
# - zero changed rows means the same validated reference is already saved;
# - more than one changed row indicates an unexpected reference change.
#
# After writing, or reusing the existing saved row, the cell reloads the CSV
# through the normal project reference loader so duplicate identities,
# coordinate ranges and IANA timezone values are validated.

import sys


src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from inside_rails.course_locations import load_course_locations


reference_before_write = course_locations.copy()
updated_course_locations = course_locations.copy()

# Blank text columns may currently have float dtype because every value was
# initially missing. Convert the location fields before assigning text.
for column in text_location_columns:
    updated_course_locations[column] = (
        updated_course_locations[column].astype("object")
    )

# Copy only the proposed location fields into the existing La Plata row.
for column in proposed_course_location.columns:
    updated_course_locations.loc[
        test_reference_mask,
        column,
    ] = proposed_course_location[column].iloc[0]

# Compare the complete reference before and after the proposed assignment.
# A normal first run changes exactly one row. A later rerun changes none.
changed_row_count = int(
    (
        reference_before_write.astype("string")
        != updated_course_locations.astype("string")
    )
    .any(axis=1)
    .sum()
)

if changed_row_count > 1:
    raise RuntimeError(
        "Expected no more than one changed reference row, "
        f"found {changed_row_count}."
    )

if changed_row_count == 1:
    updated_course_locations.to_csv(
        course_locations_path,
        index=False,
    )
    write_action = "La Plata reference written"
else:
    write_action = "Existing La Plata reference reused"

# Reload through the project validator rather than trusting the in-memory
# dataframe. This confirms that the saved CSV still satisfies the reference
# schema and validation rules.
validated_course_locations = load_course_locations(
    course_locations_path
)

saved_la_plata = validated_course_locations.loc[
    validated_course_locations[
        "candidate_course_label"
    ].eq("La Plata")
    & validated_course_locations[
        "candidate_jurisdiction"
    ].eq("Argentina")
]

if len(saved_la_plata) != 1:
    raise RuntimeError(
        "The saved reference does not contain exactly one La Plata row."
    )

# Confirm that the persisted row still matches the manually validated result.
saved_location_checks = {
    "physical venue": saved_la_plata[
        "physical_venue_name"
    ].eq("Hipódromo de La Plata").all(),
    "latitude": pd.to_numeric(
        saved_la_plata["latitude"],
        errors="coerce",
    ).round(7).eq(
        round(selected_latitude, 7)
    ).all(),
    "longitude": pd.to_numeric(
        saved_la_plata["longitude"],
        errors="coerce",
    ).round(7).eq(
        round(selected_longitude, 7)
    ).all(),
    "timezone": saved_la_plata[
        "iana_timezone"
    ].eq(selected_iana_timezone).all(),
    "validation status": saved_la_plata[
        "location_validation_status"
    ].eq("manually_validated").all(),
}

if not all(saved_location_checks.values()):
    failed_checks = [
        check_name
        for check_name, passed in saved_location_checks.items()
        if not passed
    ]

    raise RuntimeError(
        "Saved La Plata reference failed checks: "
        + ", ".join(failed_checks)
    )

# Keep the notebook's in-memory reference aligned with the validated file.
course_locations = validated_course_locations

print("Source path added:", src_path)
print("Reference rows:", len(course_locations))
print("Changed rows this run:", changed_row_count)
print("Write action:", write_action)
print("Saved file:", course_locations_path)
print("Reference validation: passed")

display(
    saved_la_plata[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "physical_venue_name",
            "locality",
            "region",
            "country",
            "latitude",
            "longitude",
            "iana_timezone",
            "location_validation_status",
        ]
    ]
)

Source path added: /home/rob/Documents/inside-rails-horse-racing/src
Reference rows: 394
Changed rows this run: 0
Write action: Existing La Plata reference reused
Saved file: /home/rob/Documents/inside-rails-horse-racing/data/reference/course_locations.csv
Reference validation: passed


,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_validation_status
0,La Plata,Argentina,Hipódromo de La Plata,La Plata,Buenos Aires,Argentina,-34.901267,-57.943804,America/Argentina/Buenos_Aires,manually_validated


In [33]:
# Recalculate reusable-location status after saving the La Plata reference.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# The expected result is:
# - La Plata is now reusable;
# - one course is reusable overall;
# - 393 courses still require location work.

accepted_location_statuses = {
    "automatically_validated",
    "manually_validated",
    "validated",
}

course_locations["has_reusable_location"] = (
    course_locations["location_validation_status"]
    .isin(accepted_location_statuses)
    & course_locations["physical_venue_name"].notna()
    & pd.to_numeric(
        course_locations["latitude"],
        errors="coerce",
    ).between(-90, 90)
    & pd.to_numeric(
        course_locations["longitude"],
        errors="coerce",
    ).between(-180, 180)
    & course_locations["iana_timezone"].notna()
)

la_plata_reusable = course_locations.loc[
    course_locations["candidate_course_label"].eq("La Plata")
    & course_locations["candidate_jurisdiction"].eq("Argentina"),
    [
        "candidate_course_label",
        "candidate_jurisdiction",
        "physical_venue_name",
        "iana_timezone",
        "location_validation_status",
        "has_reusable_location",
    ],
]

print(
    "Reusable course locations:",
    int(course_locations["has_reusable_location"].sum()),
)
print(
    "Courses still requiring location work:",
    int((~course_locations["has_reusable_location"]).sum()),
)

display(la_plata_reusable)

Reusable course locations: 36
Courses still requiring location work: 358


,candidate_course_label,candidate_jurisdiction,physical_venue_name,iana_timezone,location_validation_status,has_reusable_location
0,La Plata,Argentina,Hipódromo de La Plata,America/Argentina/Buenos_Aires,manually_validated,True


In [34]:
# Summarise the complete La Plata geocoding audit trail.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# It confirms that the course has:
# - one generic query with no results;
# - one local-term query whose results were reviewed and rejected;
# - one precise alias query with a manually selected result;
# - exactly one reusable final reference row.

la_plata_cache_history = (
    geocoding_cache.loc[
        geocoding_cache["candidate_course_label"].eq("La Plata")
        & geocoding_cache["candidate_jurisdiction"].eq("Argentina"),
        [
            "requested_at_utc",
            "exact_query",
            "request_status",
            "result_count",
            "selected_result_index",
            "selected_provider_place_id",
            "review_status",
            "review_notes",
        ],
    ]
    .sort_values("requested_at_utc")
    .reset_index(drop=True)
)

la_plata_raw_record_count = 0

with geocoding_raw_responses_path.open(
    "r",
    encoding="utf-8",
) as raw_file:
    for line in raw_file:
        if not line.strip():
            continue

        record = json.loads(line)

        if (
            record.get("candidate_course_label") == "La Plata"
            and record.get("candidate_jurisdiction") == "Argentina"
        ):
            la_plata_raw_record_count += 1

audit_checks = {
    "three manifest attempts": len(la_plata_cache_history) == 3,
    "three raw responses": la_plata_raw_record_count == 3,
    "one selected attempt": (
        la_plata_cache_history["review_status"]
        .eq("manually_selected")
        .sum()
        == 1
    ),
    "one reusable reference": (
        course_locations.loc[
            course_locations["candidate_course_label"].eq("La Plata")
            & course_locations["candidate_jurisdiction"].eq("Argentina"),
            "has_reusable_location",
        ]
        .eq(True)
        .sum()
        == 1
    ),
}

display(la_plata_cache_history)

print()
print("Raw-response records:", la_plata_raw_record_count)

for check_name, passed in audit_checks.items():
    print(f"{check_name}: {bool(passed)}")

print(
    "Complete audit trail valid:",
    all(audit_checks.values()),
)

,requested_at_utc,exact_query,request_status,result_count,selected_result_index,selected_provider_place_id,review_status,review_notes
0,2026-07-26T13:11:34.250680+00:00,"La Plata racecourse, Argentina",success_no_results,0,NaN,NaN,reviewed_no_results,Request completed successfully but returned no...
1,2026-07-26T13:15:32.809207+00:00,"La Plata hipódromo, Argentina",success_with_results,3,NaN,NaN,reviewed_no_usable_result,All three returned results rejected: one resid...
2,2026-07-26T13:18:33.361871+00:00,"Hipódromo de La Plata, Argentina",success_with_results,2,1.0,way:287312802,manually_selected,Selected result 1. Exact venue name in La Plat...
3,2026-07-27T00:34:10.167834+00:00,"La Plata hipodromo, Argentina",success_with_results,4,NaN,NaN,not_reviewed,NaN



Raw-response records: 4
three manifest attempts: False
three raw responses: False
one selected attempt: True
one reusable reference: True
Complete audit trail valid: False


In [35]:
# Close successful zero-result requests as reviewed.
#
# This cell updates only course_geocoding_cache.csv.
#
# It is safe to rerun:
# - only rows still marked "not_reviewed" are changed;
# - already closed rows remain unchanged;
# - zero matching rows is a valid resumed-notebook outcome.
#
# A successful request with zero alternatives needs no candidate review.
# The timestamp remains in the displayed columns so the final table can be
# sorted without raising a KeyError.
#
# No external request is sent and course_locations.csv is not modified.

zero_result_mask = (
    geocoding_cache["request_status"].eq("success_no_results")
    & geocoding_cache["result_count"].fillna(0).eq(0)
    & geocoding_cache["review_status"].eq("not_reviewed")
)

zero_result_rows_to_close = int(zero_result_mask.sum())

if zero_result_rows_to_close > 0:
    geocoding_cache.loc[
        zero_result_mask,
        "review_status",
    ] = "reviewed_no_results"

    geocoding_cache.loc[
        zero_result_mask,
        "review_notes",
    ] = (
        "Request completed successfully but returned no provider results."
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

zero_result_review = (
    geocoding_cache.loc[
        geocoding_cache["request_status"].eq("success_no_results"),
        [
            "requested_at_utc",
            "candidate_course_label",
            "candidate_jurisdiction",
            "exact_query",
            "request_status",
            "result_count",
            "review_status",
            "review_notes",
        ],
    ]
    .sort_values("requested_at_utc")
    .reset_index(drop=True)
)

unclosed_zero_result_rows = int(
    (
        zero_result_review["result_count"].fillna(0).eq(0)
        & ~zero_result_review["review_status"].eq(
            "reviewed_no_results"
        )
    ).sum()
)

print("Zero-result rows closed this run:", zero_result_rows_to_close)
print("Unclosed zero-result rows:", unclosed_zero_result_rows)

display(zero_result_review)

Zero-result rows closed this run: 11
Unclosed zero-result rows: 0


,requested_at_utc,candidate_course_label,candidate_jurisdiction,exact_query,request_status,result_count,review_status,review_notes
0,2026-07-26T13:11:34.250680+00:00,La Plata,Argentina,"La Plata racecourse, Argentina",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
1,2026-07-26T21:49:30.239277+00:00,Palermo,Argentina,"Palermo racecourse, Argentina",success_no_results,0,reviewed_no_results,Generic English venue query returned no provid...
2,2026-07-27T00:36:43.566156+00:00,Wolverhampton (AW),Great Britain,"Wolverhampton (AW) racecourse, United Kingdom",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
3,2026-07-27T00:36:45.941104+00:00,Sha Tin,Hong Kong,"Sha Tin racecourse, Hong Kong",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
4,2026-07-27T00:36:47.064313+00:00,Chelmsford (AW),Great Britain,"Chelmsford (AW) racecourse, United Kingdom",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
5,2026-07-27T06:15:06.921988+00:00,Chantilly,France,"Chantilly racecourse, France",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
6,2026-07-27T06:15:08.283900+00:00,Deauville,France,"Deauville racecourse, France",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
7,2026-07-27T06:15:13.694421+00:00,Auteuil,France,"Auteuil racecourse, France",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
8,2026-07-27T06:15:15.960034+00:00,Happy Valley,Hong Kong,"Happy Valley racecourse, Hong Kong",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
9,2026-07-27T06:15:18.571797+00:00,Saint-Cloud,France,"Saint-Cloud racecourse, France",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...


In [36]:
# Verify the zero-result review update.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# The timestamp is included before sorting so the audit history displays in
# request order.

la_plata_review_history = geocoding_cache.loc[
    geocoding_cache["candidate_course_label"].eq("La Plata")
    & geocoding_cache["candidate_jurisdiction"].eq("Argentina"),
    [
        "requested_at_utc",
        "exact_query",
        "request_status",
        "result_count",
        "review_status",
        "review_notes",
    ],
].sort_values("requested_at_utc")

display(la_plata_review_history)

print(
    "Zero-result request closed:",
    la_plata_review_history.loc[
        la_plata_review_history["request_status"].eq(
            "success_no_results"
        ),
        "review_status",
    ].eq("reviewed_no_results").all(),
)

,requested_at_utc,exact_query,request_status,result_count,review_status,review_notes
0,2026-07-26T13:11:34.250680+00:00,"La Plata racecourse, Argentina",success_no_results,0,reviewed_no_results,Request completed successfully but returned no...
1,2026-07-26T13:15:32.809207+00:00,"La Plata hipódromo, Argentina",success_with_results,3,reviewed_no_usable_result,All three returned results rejected: one resid...
2,2026-07-26T13:18:33.361871+00:00,"Hipódromo de La Plata, Argentina",success_with_results,2,manually_selected,Selected result 1. Exact venue name in La Plat...
5,2026-07-27T00:34:10.167834+00:00,"La Plata hipodromo, Argentina",success_with_results,4,not_reviewed,NaN


Zero-result request closed: True


## Reusable per-course processing rule

Each unresolved course is processed through the following controlled sequence:

1. Skip the course when the accepted reference already contains a reusable
   physical venue, valid coordinates and valid IANA timezone.

2. Generate an ordered set of materially different exact queries:
   - generic English venue term;
   - jurisdiction-specific venue terms;
   - known full venue names or aliases when required.

3. Before every request, calculate the deterministic cache identifier and skip
   the external call when the exact request already exists in the manifest.

4. Send no more than one request at a time through the rate-limited client.

5. Cache every outcome:
   - successful response with alternatives;
   - successful response with no alternatives;
   - provider or network error.

6. Review returned alternatives before selection. A result is not accepted
   merely because it is first or because the request returned matches.

7. Record one terminal review outcome:
   - `reviewed_no_results`;
   - `reviewed_no_usable_result`;
   - `manually_selected`;
   - a later automated-selection status only where explicit validation rules
     support it.

8. Derive the IANA timezone from the selected coordinates and validate it with
   `ZoneInfo`.

9. Build and validate a proposed reference row before updating
   `course_locations.csv`.

10. Once accepted, later notebook runs reuse the reference and issue no
    further geocoding requests for that course unless an explicit refresh or
    historical-location review is required.

In [37]:
# Inspect the next unresolved course-location identities.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# It preserves the existing reference order and displays the first 20 courses
# that do not yet have a reusable validated location. Available source-coverage
# columns are included so the next controlled test can be chosen deliberately
# rather than simply taking an arbitrary course.

identity_columns = [
    "candidate_course_label",
    "candidate_jurisdiction",
]

possible_coverage_columns = [
    "provisional_races",
    "runner_records",
    "first_date",
    "last_date",
]

available_coverage_columns = [
    column
    for column in possible_coverage_columns
    if column in course_locations.columns
]

unresolved_course_locations = (
    course_locations.loc[
        ~course_locations["has_reusable_location"],
        identity_columns + available_coverage_columns,
    ]
    .reset_index()
    .rename(columns={"index": "reference_row_index"})
)

print(
    "Unresolved course identities:",
    len(unresolved_course_locations),
)
print(
    "Available coverage columns:",
    available_coverage_columns,
)

display(
    unresolved_course_locations.head(20)
)


Unresolved course identities: 358
Available coverage columns: ['provisional_races']


,reference_row_index,candidate_course_label,candidate_jurisdiction,provisional_races
0,2,San Isidro,Argentina,179
1,3,Albury,Australia,4
2,4,Alice springs,Australia,1
3,5,Armidale,Australia,1
4,6,Ascot,Australia,204
5,7,Balaklava,Australia,1
6,8,Ballarat,Australia,2
7,9,Belmont Park (Perth),Australia,31
8,10,Bendigo,Australia,16
9,11,Canberra,Australia,18


In [38]:
# Diagnose why the saved La Plata row is being classified as unresolved.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# It displays the saved fields used by the reusable-location rule and
# recalculates each component separately.

la_plata_diagnostic = course_locations.loc[
    course_locations["candidate_course_label"].eq("La Plata")
    & course_locations["candidate_jurisdiction"].eq("Argentina")
].copy()

la_plata_diagnostic["status_is_accepted"] = (
    la_plata_diagnostic["location_validation_status"]
    .isin(accepted_location_statuses)
)

la_plata_diagnostic["venue_is_present"] = (
    la_plata_diagnostic["physical_venue_name"].notna()
)

la_plata_diagnostic["latitude_is_valid"] = pd.to_numeric(
    la_plata_diagnostic["latitude"],
    errors="coerce",
).between(-90, 90)

la_plata_diagnostic["longitude_is_valid"] = pd.to_numeric(
    la_plata_diagnostic["longitude"],
    errors="coerce",
).between(-180, 180)

la_plata_diagnostic["timezone_is_present"] = (
    la_plata_diagnostic["iana_timezone"].notna()
)

la_plata_diagnostic["recalculated_reusable_location"] = (
    la_plata_diagnostic["status_is_accepted"]
    & la_plata_diagnostic["venue_is_present"]
    & la_plata_diagnostic["latitude_is_valid"]
    & la_plata_diagnostic["longitude_is_valid"]
    & la_plata_diagnostic["timezone_is_present"]
)

display(
    la_plata_diagnostic[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "physical_venue_name",
            "latitude",
            "longitude",
            "iana_timezone",
            "location_validation_status",
            "has_reusable_location",
            "status_is_accepted",
            "venue_is_present",
            "latitude_is_valid",
            "longitude_is_valid",
            "timezone_is_present",
            "recalculated_reusable_location",
        ]
    ]
)

,candidate_course_label,candidate_jurisdiction,physical_venue_name,latitude,longitude,iana_timezone,location_validation_status,has_reusable_location,status_is_accepted,venue_is_present,latitude_is_valid,longitude_is_valid,timezone_is_present,recalculated_reusable_location
0,La Plata,Argentina,Hipódromo de La Plata,-34.901267,-57.943804,America/Argentina/Buenos_Aires,manually_validated,True,True,True,True,True,True,True


In [39]:
# Recalculate reusable-location status for the complete reference.
#
# This cell is read-only with respect to project files:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# `has_reusable_location` is a derived notebook field rather than a persisted
# source field. It must therefore be recalculated whenever course_locations is
# reloaded from disk.
#
# A course is reusable only when it has:
# - an accepted validation status;
# - a nonblank physical venue name;
# - valid latitude and longitude;
# - a nonblank IANA timezone.

accepted_location_statuses = {
    "automatically_validated",
    "manually_validated",
    "validated",
}

course_locations["has_reusable_location"] = (
    course_locations["location_validation_status"]
    .isin(accepted_location_statuses)
    & course_locations["physical_venue_name"].notna()
    & pd.to_numeric(
        course_locations["latitude"],
        errors="coerce",
    ).between(-90, 90)
    & pd.to_numeric(
        course_locations["longitude"],
        errors="coerce",
    ).between(-180, 180)
    & course_locations["iana_timezone"].notna()
)

print(
    "Reusable course locations:",
    int(course_locations["has_reusable_location"].sum()),
)
print(
    "Courses still requiring location work:",
    int((~course_locations["has_reusable_location"]).sum()),
)

display(
    course_locations.loc[
        course_locations["candidate_course_label"].eq("La Plata")
        & course_locations["candidate_jurisdiction"].eq("Argentina"),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "location_validation_status",
            "has_reusable_location",
        ],
    ]
)

Reusable course locations: 36
Courses still requiring location work: 358


,candidate_course_label,candidate_jurisdiction,location_validation_status,has_reusable_location
0,La Plata,Argentina,manually_validated,True


In [40]:
# Identify the existing jurisdiction and venue-rule objects.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify project files;
# - it does not alter the geocoding cache.
#
# It lists currently defined notebook variables whose names suggest they hold
# jurisdiction mappings, country-code rules or venue-term rules.
#
# `globals()` is copied to a list first because directly iterating over the
# live dictionary can raise "dictionary changed size during iteration" in a
# notebook environment.

rule_name_keywords = (
    "jurisdiction",
    "venue",
    "country",
    "term",
)

possible_rule_objects = []

for variable_name, variable_value in list(globals().items()):
    if not any(
        keyword in variable_name.lower()
        for keyword in rule_name_keywords
    ):
        continue

    possible_rule_objects.append(
        {
            "variable_name": variable_name,
            "object_type": type(variable_value).__name__,
            "shape": (
                str(variable_value.shape)
                if hasattr(variable_value, "shape")
                else pd.NA
            ),
            "columns": (
                ", ".join(map(str, variable_value.columns))
                if isinstance(variable_value, pd.DataFrame)
                else pd.NA
            ),
        }
    )

possible_rule_objects = (
    pd.DataFrame(possible_rule_objects)
    .sort_values("variable_name")
    .reset_index(drop=True)
)

display(possible_rule_objects)

,variable_name,object_type,shape,columns
0,JURISDICTION_GEOCODING_RULES,dict,NaN,NaN
1,JURISDICTION_VENUE_TERMS,dict,NaN,NaN
2,country_code,str,NaN,NaN
3,country_code_match,bool,NaN,NaN
4,cross_jurisdiction_labels,DataFrame,"(6, 4)","candidate_course_label, candidate_jurisdiction..."
5,empty_term_rules,list,NaN,NaN
6,jurisdiction_profile,DataFrame,"(36, 3)","candidate_jurisdiction, candidate_courses, sam..."
7,mapped_jurisdictions,set,NaN,NaN
8,missing_term_rules,list,NaN,NaN
9,observed_jurisdictions,set,NaN,NaN


In [41]:
# Inspect the Argentina geocoding and venue-term rules.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify project files;
# - it does not alter either geocoding cache.
#
# It displays the exact dictionary values already defined earlier in the
# notebook so the Palermo request-planning cell can use their real structure
# rather than assuming column names or dataframe layouts.

argentina_geocoding_rule = (
    JURISDICTION_GEOCODING_RULES.get("Argentina")
)

argentina_venue_terms = (
    JURISDICTION_VENUE_TERMS.get("Argentina")
)

print("Argentina geocoding rule:")
display(argentina_geocoding_rule)

print()
print("Argentina venue terms:")
display(argentina_venue_terms)

print()
print(
    "Geocoding-rule object type:",
    type(argentina_geocoding_rule).__name__,
)
print(
    "Venue-terms object type:",
    type(argentina_venue_terms).__name__,
)

Argentina geocoding rule:


{'query_country': 'Argentina', 'country_codes': {'ar'}}


Argentina venue terms:


['racecourse', 'hipódromo', 'hipodromo']


Geocoding-rule object type: dict
Venue-terms object type: list


In [42]:
# Prepare the next controlled course test: Palermo, Argentina.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# It selects Palermo from the unresolved reference and shows:
# - the existing jurisdiction geocoding rule;
# - the ordered generic and Argentina-specific venue queries;
# - the deterministic cache identity of every exact request;
# - whether any request has already been cached.
#
# The purpose is to inspect the planned requests before deciding whether an
# external provider call is necessary.

next_course_label = "Palermo"
next_jurisdiction = "Argentina"

next_reference_mask = (
    course_locations["candidate_course_label"].eq(next_course_label)
    & course_locations["candidate_jurisdiction"].eq(next_jurisdiction)
)

next_reference_rows = course_locations.loc[
    next_reference_mask
]

if len(next_reference_rows) != 1:
    raise RuntimeError(
        "Expected exactly one Palermo, Argentina reference row, "
        f"found {len(next_reference_rows)}."
    )

next_course = next_reference_rows.iloc[0]

# Read the jurisdiction rules using the dictionary structures defined earlier
# in this notebook. The geocoding rule supplies the provider-facing country
# name and the set of accepted country-code filters.
next_geocoding_rule = (
    JURISDICTION_GEOCODING_RULES[next_jurisdiction]
)

next_country_name = next_geocoding_rule[
    "query_country"
]

next_country_codes = sorted(
    next_geocoding_rule["country_codes"]
)

if len(next_country_codes) != 1:
    raise RuntimeError(
        "Expected exactly one Argentina country-code filter, "
        f"found {next_country_codes}."
    )

next_country_code = next_country_codes[0]

# Preserve the venue-term order defined in the jurisdiction rules. For
# Argentina this is:
# - racecourse
# - hipódromo
# - hipodromo
next_venue_terms = list(
    JURISDICTION_VENUE_TERMS[next_jurisdiction]
)

next_candidate_query_rows = []

for query_order, venue_term in enumerate(
    next_venue_terms,
    start=1,
):
    exact_query = (
        f"{next_course_label} "
        f"{venue_term}, "
        f"{next_country_name}"
    )

    query_type = (
        "generic_english"
        if venue_term == "racecourse"
        else "jurisdiction_term"
    )

    cache_record_id = (
        build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=next_course_label,
            candidate_jurisdiction=next_jurisdiction,
            exact_query=exact_query,
            country_code_filter=next_country_code,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )
    )

    next_candidate_query_rows.append(
        {
            "query_order": query_order,
            "query_type": query_type,
            "venue_term": venue_term,
            "exact_query": exact_query,
            "country_code_filter": next_country_code,
            "cache_record_id": cache_record_id,
        }
    )

next_candidate_queries = pd.DataFrame(
    next_candidate_query_rows
)

cached_record_ids = set(
    geocoding_cache["cache_record_id"]
    .dropna()
    .astype(str)
)

next_candidate_queries["already_cached"] = (
    next_candidate_queries["cache_record_id"]
    .astype(str)
    .isin(cached_record_ids)
)

print("Selected course:", next_course_label)
print("Jurisdiction:", next_jurisdiction)
print(
    "Provisional races:",
    int(next_course["provisional_races"]),
)
print("Query country:", next_country_name)
print("Country-code filter:", next_country_code)
print("Candidate queries:", len(next_candidate_queries))
print(
    "Already cached:",
    int(next_candidate_queries["already_cached"].sum()),
)

display(
    next_candidate_queries[
        [
            "query_order",
            "query_type",
            "venue_term",
            "exact_query",
            "country_code_filter",
            "already_cached",
        ]
    ]
)

Selected course: Palermo
Jurisdiction: Argentina
Provisional races: 200
Query country: Argentina
Country-code filter: ar
Candidate queries: 3
Already cached: 2


,query_order,query_type,venue_term,exact_query,country_code_filter,already_cached
0,1,generic_english,racecourse,"Palermo racecourse, Argentina",ar,True
1,2,jurisdiction_term,hipódromo,"Palermo hipódromo, Argentina",ar,True
2,3,jurisdiction_term,hipodromo,"Palermo hipodromo, Argentina",ar,False


In [43]:
# Send or reuse the generic Palermo geocoding request.
#
# Exact request:
# - Palermo racecourse, Argentina
#
# This is the first query in the Palermo sequence. It tests the generic
# English venue term before trying the Argentina-specific alternatives.
#
# The cell is safe to rerun:
# - if the exact request is already cached, no external request is made;
# - otherwise one rate-limited request is sent and the complete outcome is
#   written to both geocoding cache files.
#
# It writes only when the request is new:
# - data/cache/course_geocoding_cache.csv
# - data/cache/course_geocoding_responses.jsonl
#
# This cell does not select a result, derive a timezone, or update the final
# course-location reference.

palermo_generic_request = (
    next_candidate_queries.loc[
        next_candidate_queries["query_order"].eq(1)
    ]
    .iloc[0]
)

palermo_generic_query = (
    palermo_generic_request["exact_query"]
)

palermo_generic_cache_record_id = (
    palermo_generic_request["cache_record_id"]
)

palermo_generic_cache_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(palermo_generic_cache_record_id)
)

palermo_generic_is_cached = bool(
    palermo_generic_cache_mask.any()
)

if palermo_generic_is_cached:
    # Reuse the previously saved provider outcome.
    cached_palermo_generic_row = geocoding_cache.loc[
        palermo_generic_cache_mask
    ].iloc[0]

    request_status = cached_palermo_generic_row[
        "request_status"
    ]

    cached_result_count = pd.to_numeric(
        cached_palermo_generic_row["result_count"],
        errors="coerce",
    )

    result_count = (
        0
        if pd.isna(cached_result_count)
        else int(cached_result_count)
    )

    print("Request action: reused cached result")

else:
    requested_at_utc = datetime.now(
        timezone.utc
    ).isoformat()

    raw_response_record_id = (
        palermo_generic_cache_record_id
    )

    request_status = "pending"
    result_count = 0
    raw_results = []
    error_type = None
    error_message = None

    try:
        # Send one request through the configured rate-limited client.
        locations = rate_limited_geocode(
            palermo_generic_query,
            country_codes=next_country_code,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        # Preserve each complete provider result for later review.
        raw_results = [
            location.raw
            for location in locations
        ]

        result_count = len(raw_results)

        request_status = (
            "success_with_results"
            if result_count > 0
            else "success_no_results"
        )

    except Exception as exc:
        # Cache errors as explicit request outcomes.
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_response_record = {
        "raw_response_record_id": raw_response_record_id,
        "cache_record_id": palermo_generic_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": next_course_label,
        "candidate_jurisdiction": next_jurisdiction,
        "exact_query": palermo_generic_query,
        "country_code_filter": next_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    with geocoding_raw_responses_path.open(
        "a",
        encoding="utf-8",
    ) as raw_file:
        raw_file.write(
            json.dumps(
                raw_response_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    manifest_row = {
        "cache_record_id": palermo_generic_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": next_course_label,
        "candidate_jurisdiction": next_jurisdiction,
        "exact_query": palermo_generic_query,
        "country_code_filter": next_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": result_count,
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": raw_response_record_id,
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

    print("Request action: sent and cached")

print("Exact query:", palermo_generic_query)
print("Request status:", request_status)
print("Returned results:", result_count)
print("Manifest rows:", len(geocoding_cache))

Request action: reused cached result
Exact query: Palermo racecourse, Argentina
Request status: success_no_results
Returned results: 0
Manifest rows: 76


In [44]:
# Send or reuse the accented local-term Palermo geocoding request.
#
# Exact request:
# - Palermo hipódromo, Argentina
#
# This is the second query in the Palermo sequence. It uses the
# Argentina-specific accented venue term after the generic English query
# returned no results.
#
# The cell is safe to rerun:
# - if the exact request is already cached, no external request is made;
# - otherwise one rate-limited request is sent and the complete outcome is
#   written to both geocoding cache files.
#
# It writes only when the request is new:
# - data/cache/course_geocoding_cache.csv
# - data/cache/course_geocoding_responses.jsonl
#
# This cell does not select a result, derive a timezone, or update the final
# course-location reference.

palermo_accented_request = (
    next_candidate_queries.loc[
        next_candidate_queries["query_order"].eq(2)
    ]
    .iloc[0]
)

palermo_accented_query = (
    palermo_accented_request["exact_query"]
)

palermo_accented_cache_record_id = (
    palermo_accented_request["cache_record_id"]
)

palermo_accented_cache_mask = (
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(palermo_accented_cache_record_id)
)

palermo_accented_is_cached = bool(
    palermo_accented_cache_mask.any()
)

if palermo_accented_is_cached:
    # Reuse the previously saved provider outcome.
    cached_palermo_accented_row = geocoding_cache.loc[
        palermo_accented_cache_mask
    ].iloc[0]

    request_status = cached_palermo_accented_row[
        "request_status"
    ]

    cached_result_count = pd.to_numeric(
        cached_palermo_accented_row["result_count"],
        errors="coerce",
    )

    result_count = (
        0
        if pd.isna(cached_result_count)
        else int(cached_result_count)
    )

    print("Request action: reused cached result")

else:
    requested_at_utc = datetime.now(
        timezone.utc
    ).isoformat()

    raw_response_record_id = (
        palermo_accented_cache_record_id
    )

    request_status = "pending"
    result_count = 0
    raw_results = []
    error_type = None
    error_message = None

    try:
        # Send one request through the configured rate-limited client.
        locations = rate_limited_geocode(
            palermo_accented_query,
            country_codes=next_country_code,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        # Preserve each complete provider result for later review.
        raw_results = [
            location.raw
            for location in locations
        ]

        result_count = len(raw_results)

        request_status = (
            "success_with_results"
            if result_count > 0
            else "success_no_results"
        )

    except Exception as exc:
        # Cache errors as explicit request outcomes.
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_response_record = {
        "raw_response_record_id": raw_response_record_id,
        "cache_record_id": palermo_accented_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": next_course_label,
        "candidate_jurisdiction": next_jurisdiction,
        "exact_query": palermo_accented_query,
        "country_code_filter": next_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    with geocoding_raw_responses_path.open(
        "a",
        encoding="utf-8",
    ) as raw_file:
        raw_file.write(
            json.dumps(
                raw_response_record,
                ensure_ascii=False,
            )
            + "\n"
        )

    manifest_row = {
        "cache_record_id": palermo_accented_cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": next_course_label,
        "candidate_jurisdiction": next_jurisdiction,
        "exact_query": palermo_accented_query,
        "country_code_filter": next_country_code,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": result_count,
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": raw_response_record_id,
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(
        geocoding_cache_path,
        index=False,
    )

    print("Request action: sent and cached")

print("Exact query:", palermo_accented_query)
print("Request status:", request_status)
print("Returned results:", result_count)
print("Manifest rows:", len(geocoding_cache))

Request action: reused cached result
Exact query: Palermo hipódromo, Argentina
Request status: success_with_results
Returned results: 3
Manifest rows: 76


In [45]:
# Inspect the cached results from the accented Palermo query.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache file;
# - it does not write to course_locations.csv.
#
# It retrieves the complete raw response associated with:
# - Palermo hipódromo, Argentina
#
# The results are flattened into review fields so we can determine whether
# any candidate is genuinely the Palermo racecourse rather than accepting the
# first provider result automatically.

palermo_accented_raw_record = None

with geocoding_raw_responses_path.open(
    "r",
    encoding="utf-8",
) as raw_file:
    for line in raw_file:
        if not line.strip():
            continue

        raw_record = json.loads(line)

        if (
            raw_record.get("cache_record_id")
            == palermo_accented_cache_record_id
        ):
            palermo_accented_raw_record = raw_record
            break

if palermo_accented_raw_record is None:
    raise RuntimeError(
        "The cached raw response for the accented Palermo query "
        "could not be found."
    )

palermo_accented_results = (
    palermo_accented_raw_record.get("results", [])
)

palermo_accented_review_rows = []

for result_index, result in enumerate(
    palermo_accented_results
):
    address = result.get("address") or {}

    provider_place_id = (
        f"{result.get('osm_type')}:{result.get('osm_id')}"
        if result.get("osm_type") is not None
        and result.get("osm_id") is not None
        else pd.NA
    )

    palermo_accented_review_rows.append(
        {
            "result_index": result_index,
            "provider_place_id": provider_place_id,
            "display_name": result.get("display_name"),
            "name": result.get("name"),
            "category": result.get("category"),
            "type": result.get("type"),
            "latitude": pd.to_numeric(
                result.get("lat"),
                errors="coerce",
            ),
            "longitude": pd.to_numeric(
                result.get("lon"),
                errors="coerce",
            ),
            "city_or_locality": (
                address.get("city")
                or address.get("town")
                or address.get("municipality")
                or address.get("village")
                or address.get("suburb")
            ),
            "state_or_region": (
                address.get("state")
                or address.get("province")
            ),
            "country": address.get("country"),
            "country_code": address.get("country_code"),
        }
    )

palermo_accented_review = pd.DataFrame(
    palermo_accented_review_rows
)

print("Exact query:", palermo_accented_query)
print(
    "Cached request status:",
    palermo_accented_raw_record["request_status"],
)
print(
    "Cached alternatives:",
    len(palermo_accented_review),
)

display(palermo_accented_review)

Exact query: Palermo hipódromo, Argentina
Cached request status: success_with_results
Cached alternatives: 3


,result_index,provider_place_id,display_name,name,category,type,latitude,longitude,city_or_locality,state_or_region,country,country_code
0,0,way:18772836,"Hipódromo Argentino de Palermo, 4101, Avenida ...",Hipódromo Argentino de Palermo,None,sports_centre,-34.566398,-58.425727,Buenos Aires,Autonomous City of Buenos Aires,Argentina,ar
1,1,node:13624444309,"339 - Hipódromo de Palermo, Avenida Del Libert...",339 - Hipódromo de Palermo,None,bicycle_rental,-34.568702,-58.427378,Buenos Aires,Autonomous City of Buenos Aires,Argentina,ar
2,2,node:1316709340,"Casino Hipodromo de Palermo, Avenida Del Liber...",Casino Hipodromo de Palermo,None,casino,-34.568879,-58.426021,Buenos Aires,Autonomous City of Buenos Aires,Argentina,ar


In [46]:
# Record the manual review decision for the Palermo requests.
#
# Decisions:
# - "Palermo racecourse, Argentina" returned no results and is marked as a
#   reviewed no-result request.
# - "Palermo hipódromo, Argentina" returned three alternatives.
# - result index 0 is selected because it is the physical racecourse:
#   Hipódromo Argentino de Palermo.
#
# The nearby bicycle-rental and casino results are not selected.
#
# This cell:
# - makes no external request;
# - updates the existing manifest rows only;
# - writes the updated manifest to the geocoding cache CSV;
# - does not yet update course_locations.csv or derive the timezone.
#
# It is safe to rerun because it assigns the same review values each time.

palermo_selected_result_index = 0

if palermo_selected_result_index >= len(
    palermo_accented_results
):
    raise RuntimeError(
        "The selected Palermo result index is outside the "
        "available cached results."
    )

palermo_selected_result = (
    palermo_accented_results[
        palermo_selected_result_index
    ]
)

palermo_selected_address = (
    palermo_selected_result.get("address") or {}
)

palermo_selected_provider_place_id = (
    f"{palermo_selected_result.get('osm_type')}:"
    f"{palermo_selected_result.get('osm_id')}"
)

# Mark the generic English request as reviewed with no results.
geocoding_cache.loc[
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(palermo_generic_cache_record_id),
    [
        "review_status",
        "review_notes",
    ],
] = [
    "reviewed_no_results",
    (
        "Generic English venue query returned no provider "
        "results; the jurisdiction-specific query was used."
    ),
]

# Record the manually selected racecourse result from the accented query.
geocoding_cache.loc[
    geocoding_cache["cache_record_id"]
    .astype(str)
    .eq(palermo_accented_cache_record_id),
    [
        "selected_result_index",
        "selected_provider_place_id",
        "selected_display_name",
        "selected_latitude",
        "selected_longitude",
        "country_code_match",
        "venue_type_match",
        "name_match_status",
        "review_status",
        "review_notes",
    ],
] = [
    palermo_selected_result_index,
    palermo_selected_provider_place_id,
    palermo_selected_result.get("display_name"),
    float(palermo_selected_result["lat"]),
    float(palermo_selected_result["lon"]),
    (
        palermo_selected_address.get("country_code")
        == next_country_code
    ),
    True,
    "strong_course_and_venue_name_match",
    "manually_selected",
    (
        "Selected Hipódromo Argentino de Palermo. "
        "Rejected the nearby bicycle-rental and casino results."
    ),
]

geocoding_cache.to_csv(
    geocoding_cache_path,
    index=False,
)

palermo_reviewed_rows = geocoding_cache.loc[
    geocoding_cache["cache_record_id"]
    .astype(str)
    .isin(
        {
            palermo_generic_cache_record_id,
            palermo_accented_cache_record_id,
        }
    ),
    [
        "exact_query",
        "request_status",
        "result_count",
        "selected_result_index",
        "selected_display_name",
        "selected_latitude",
        "selected_longitude",
        "review_status",
    ],
].sort_values("exact_query")

print("Palermo review decisions saved.")
display(palermo_reviewed_rows)

Palermo review decisions saved.


,exact_query,request_status,result_count,selected_result_index,selected_display_name,selected_latitude,selected_longitude,review_status
4,"Palermo hipódromo, Argentina",success_with_results,3,0.0,"Hipódromo Argentino de Palermo, 4101, Avenida ...",-34.566398,-58.425727,manually_selected
3,"Palermo racecourse, Argentina",success_no_results,0,NaN,NaN,NaN,NaN,reviewed_no_results


In [47]:
# Identify the existing timezone-related notebook objects.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify project files;
# - it does not alter either geocoding cache.
#
# A frozen copy of globals is used because directly iterating over the live
# notebook namespace can raise "dictionary changed size during iteration".

timezone_name_keywords = (
    "timezone",
    "time_zone",
    "tz",
)

possible_timezone_objects = []

for variable_name, variable_value in list(globals().items()):
    if not any(
        keyword in variable_name.lower()
        for keyword in timezone_name_keywords
    ):
        continue

    possible_timezone_objects.append(
        {
            "variable_name": variable_name,
            "object_type": type(variable_value).__name__,
            "shape": (
                str(variable_value.shape)
                if hasattr(variable_value, "shape")
                else pd.NA
            ),
        }
    )

possible_timezone_objects = (
    pd.DataFrame(possible_timezone_objects)
    .sort_values("variable_name")
    .reset_index(drop=True)
)

display(possible_timezone_objects)

,variable_name,object_type,shape
0,TimezoneFinder,ABCMeta,<NA>
1,possible_timezone_objects,list,<NA>
2,selected_iana_timezone,str,<NA>
3,test_timezone,str,<NA>
4,timezone,type,<NA>
5,timezone_finder,TimezoneFinder,<NA>
6,timezone_is_valid,bool,<NA>
7,timezone_name_keywords,tuple,<NA>
8,timezone_validation_error,NoneType,<NA>
9,timezonefinder,module,<NA>


In [48]:
# Derive and validate the IANA timezone for the selected Palermo racecourse.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache;
# - it does not write to course_locations.csv.
#
# It uses the already selected Palermo coordinates and the existing
# TimezoneFinder instance created earlier in the notebook.

palermo_selected_latitude = float(
    palermo_selected_result["lat"]
)

palermo_selected_longitude = float(
    palermo_selected_result["lon"]
)

palermo_iana_timezone = timezone_finder.timezone_at(
    lat=palermo_selected_latitude,
    lng=palermo_selected_longitude,
)

if palermo_iana_timezone is None:
    raise RuntimeError(
        "No IANA timezone could be derived for the selected "
        "Palermo coordinates."
    )

# Validate that Python's timezone database recognises the returned name.
try:
    ZoneInfo(palermo_iana_timezone)
    palermo_timezone_is_valid = True
except Exception as exc:
    palermo_timezone_is_valid = False
    raise RuntimeError(
        "The derived Palermo timezone is not a valid IANA timezone: "
        f"{palermo_iana_timezone}"
    ) from exc

print("Selected venue: Hipódromo Argentino de Palermo")
print("Latitude:", palermo_selected_latitude)
print("Longitude:", palermo_selected_longitude)
print("Derived IANA timezone:", palermo_iana_timezone)
print("Timezone valid:", palermo_timezone_is_valid)

Selected venue: Hipódromo Argentino de Palermo
Latitude: -34.5663978
Longitude: -58.4257272
Derived IANA timezone: America/Argentina/Buenos_Aires
Timezone valid: True


In [49]:
# Inspect the permanent course-location reference schema.
#
# This is the final structural inspection before the reusable batch function.
# It:
# - makes no external requests;
# - writes no files;
# - shows the actual column names;
# - shows the existing validated La Plata row as the model for Palermo and
#   subsequent automated writes.

print("course_locations columns:")
print(list(course_locations.columns))

print()
print("Existing validated La Plata row:")

display(
    course_locations.loc[
        course_locations["candidate_course_label"].eq("La Plata")
        & course_locations["candidate_jurisdiction"].eq("Argentina")
    ].T
)

course_locations columns:
['candidate_course_label', 'candidate_jurisdiction', 'physical_venue_name', 'locality', 'region', 'country', 'latitude', 'longitude', 'iana_timezone', 'location_evidence', 'location_validation_status', 'raw_course_labels', 'provisional_races', 'meeting_dates', 'earliest_date', 'latest_date', 'has_reusable_location', 'address_line', 'suburb', 'postcode', 'provider_place_id', 'provider_display_name']

Existing validated La Plata row:


,0
candidate_course_label,La Plata
candidate_jurisdiction,Argentina
physical_venue_name,Hipódromo de La Plata
locality,La Plata
region,Buenos Aires
country,Argentina
latitude,-34.901267
longitude,-57.943804
iana_timezone,America/Argentina/Buenos_Aires
location_evidence,Nominatim manual selection from query 'Hipódro...


In [50]:
# Write the validated Palermo location to the permanent course reference.
#
# This cell:
# - updates exactly one existing Palermo, Argentina row;
# - preserves the source-derived race and date fields already present;
# - writes the completed reference back to course_locations.csv;
# - recalculates the derived reusable-location flag;
# - validates the saved values after writing.
#
# It is safe to rerun:
# - the same validated values are assigned each time;
# - no duplicate row is created;
# - a second run should report zero changed rows.

palermo_reference_mask = (
    course_locations["candidate_course_label"].eq("Palermo")
    & course_locations["candidate_jurisdiction"].eq("Argentina")
)

if int(palermo_reference_mask.sum()) != 1:
    raise RuntimeError(
        "Expected exactly one Palermo, Argentina reference row, "
        f"found {int(palermo_reference_mask.sum())}."
    )

palermo_reference_values = {
    "physical_venue_name": "Hipódromo Argentino de Palermo",
    "locality": "Buenos Aires",
    "region": "Autonomous City of Buenos Aires",
    "country": "Argentina",
    "latitude": palermo_selected_latitude,
    "longitude": palermo_selected_longitude,
    "iana_timezone": palermo_iana_timezone,
    "location_evidence": (
        "Nominatim manual selection from query "
        "'Palermo hipódromo, Argentina'; selected result index 0, "
        "Hipódromo Argentino de Palermo."
    ),
    "location_validation_status": "manually_validated",
}

palermo_before = (
    course_locations.loc[
        palermo_reference_mask,
        list(palermo_reference_values),
    ]
    .copy()
)

for column_name, column_value in palermo_reference_values.items():
    course_locations.loc[
        palermo_reference_mask,
        column_name,
    ] = column_value

# Recalculate the derived flag from the permanent reusable fields rather than
# assigning it independently.
course_locations["has_reusable_location"] = (
    course_locations["latitude"].notna()
    & course_locations["longitude"].notna()
    & course_locations["iana_timezone"].notna()
    & course_locations["location_validation_status"].isin(
        {
            "manually_validated",
            "automatically_validated",
        }
    )
)

palermo_after = (
    course_locations.loc[
        palermo_reference_mask,
        list(palermo_reference_values),
    ]
    .copy()
)

changed_columns = [
    column_name
    for column_name in palermo_reference_values
    if not palermo_before[column_name].equals(
        palermo_after[column_name]
    )
]

course_locations.to_csv(
    course_locations_path,
    index=False,
)

# Reload from disk to verify that the persisted record is complete.
course_locations = pd.read_csv(
    course_locations_path
)

course_locations["has_reusable_location"] = (
    course_locations["latitude"].notna()
    & course_locations["longitude"].notna()
    & course_locations["iana_timezone"].notna()
    & course_locations["location_validation_status"].isin(
        {
            "manually_validated",
            "automatically_validated",
        }
    )
)

saved_palermo = course_locations.loc[
    course_locations["candidate_course_label"].eq("Palermo")
    & course_locations["candidate_jurisdiction"].eq("Argentina")
]

if len(saved_palermo) != 1:
    raise RuntimeError(
        "Palermo was not preserved as exactly one row after saving."
    )

saved_palermo_row = saved_palermo.iloc[0]

required_saved_values = {
    "physical_venue_name": "Hipódromo Argentino de Palermo",
    "country": "Argentina",
    "iana_timezone": "America/Argentina/Buenos_Aires",
    "location_validation_status": "manually_validated",
}

for column_name, expected_value in required_saved_values.items():
    if saved_palermo_row[column_name] != expected_value:
        raise RuntimeError(
            f"Saved Palermo {column_name!r} did not match "
            f"{expected_value!r}."
        )

if not bool(saved_palermo_row["has_reusable_location"]):
    raise RuntimeError(
        "The saved Palermo row was not recognised as reusable."
    )

print("Permanent reference rows:", len(course_locations))
print("Changed columns this run:", len(changed_columns))
print(
    "Changed column names:",
    changed_columns if changed_columns else "none",
)
print("Palermo reusable location saved and validated.")

display(
    saved_palermo[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "physical_venue_name",
            "locality",
            "region",
            "country",
            "latitude",
            "longitude",
            "iana_timezone",
            "location_validation_status",
            "has_reusable_location",
        ]
    ]
)

Permanent reference rows: 394
Changed columns this run: 0
Changed column names: none
Palermo reusable location saved and validated.


,candidate_course_label,candidate_jurisdiction,physical_venue_name,locality,region,country,latitude,longitude,iana_timezone,location_validation_status,has_reusable_location
1,Palermo,Argentina,Hipódromo Argentino de Palermo,Buenos Aires,Autonomous City of Buenos Aires,Argentina,-34.566398,-58.425727,America/Argentina/Buenos_Aires,manually_validated,True


In [51]:
# Inspect the complete address fields returned for the selected Palermo venue.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify either cache;
# - it does not write to course_locations.csv.
#
# It shows every address component supplied by Nominatim so we can decide
# which fields belong in the permanent course-location reference.

palermo_selected_address = (
    palermo_selected_result.get("address") or {}
)

palermo_address_components = pd.DataFrame(
    [
        {
            "address_component": key,
            "value": value,
        }
        for key, value in palermo_selected_address.items()
    ]
).sort_values("address_component")

print("Provider display name:")
print(
    palermo_selected_result.get("display_name")
)

print()
print("Provider place ID:")
print(
    palermo_selected_provider_place_id
)

print()
print("Returned address components:")

display(palermo_address_components)

Provider display name:
Hipódromo Argentino de Palermo, 4101, Avenida Del Libertador, Palermo Pacífico, Palermo, Buenos Aires, Comuna 14, Autonomous City of Buenos Aires, C1426BSD, Argentina

Provider place ID:
way:18772836

Returned address components:


,address_component,value
8,ISO3166-2-lvl4,AR-C
5,city,Buenos Aires
10,country,Argentina
11,country_code,ar
1,house_number,4101
0,leisure,Hipódromo Argentino de Palermo
3,neighbourhood,Palermo Pacífico
9,postcode,C1426BSD
2,road,Avenida Del Libertador
7,state,Autonomous City of Buenos Aires


In [52]:
# Add practical address and provider-lineage fields to the permanent
# course-location reference, then backfill the validated Palermo record.
#
# New permanent fields:
# - address_line: street address without repeating locality/region/country;
# - suburb: useful local district within a city;
# - postcode: venue postcode where supplied;
# - provider_place_id: stable provider object identifier;
# - provider_display_name: complete provider-returned address for audit.
#
# This cell:
# - adds missing columns without removing existing data;
# - fills the existing Palermo row from the selected provider result;
# - writes course_locations.csv;
# - reloads and validates the saved values.
#
# It is safe to rerun:
# - existing columns are reused;
# - the same Palermo values are assigned;
# - no duplicate course row is created.

location_enrichment_columns = [
    "address_line",
    "suburb",
    "postcode",
    "provider_place_id",
    "provider_display_name",
]

for column_name in location_enrichment_columns:
    if column_name not in course_locations.columns:
        course_locations[column_name] = pd.NA

palermo_reference_mask = (
    course_locations["candidate_course_label"].eq("Palermo")
    & course_locations["candidate_jurisdiction"].eq("Argentina")
)

if int(palermo_reference_mask.sum()) != 1:
    raise RuntimeError(
        "Expected exactly one Palermo, Argentina reference row, "
        f"found {int(palermo_reference_mask.sum())}."
    )

palermo_enrichment_values = {
    "address_line": "4101 Avenida Del Libertador",
    "suburb": "Palermo",
    "postcode": "C1426BSD",
    "provider_place_id": palermo_selected_provider_place_id,
    "provider_display_name": (
        palermo_selected_result.get("display_name")
    ),
}

palermo_enrichment_before = (
    course_locations.loc[
        palermo_reference_mask,
        list(palermo_enrichment_values),
    ]
    .copy()
)

for column_name, column_value in (
    palermo_enrichment_values.items()
):
    course_locations.loc[
        palermo_reference_mask,
        column_name,
    ] = column_value

palermo_enrichment_after = (
    course_locations.loc[
        palermo_reference_mask,
        list(palermo_enrichment_values),
    ]
    .copy()
)

changed_enrichment_columns = [
    column_name
    for column_name in palermo_enrichment_values
    if not palermo_enrichment_before[column_name].equals(
        palermo_enrichment_after[column_name]
    )
]

course_locations.to_csv(
    course_locations_path,
    index=False,
)

# Reload from disk and recalculate the derived reusable-location flag.
course_locations = pd.read_csv(
    course_locations_path
)

course_locations["has_reusable_location"] = (
    course_locations["latitude"].notna()
    & course_locations["longitude"].notna()
    & course_locations["iana_timezone"].notna()
    & course_locations["location_validation_status"].isin(
        {
            "manually_validated",
            "automatically_validated",
        }
    )
)

saved_palermo = course_locations.loc[
    course_locations["candidate_course_label"].eq("Palermo")
    & course_locations["candidate_jurisdiction"].eq("Argentina")
]

if len(saved_palermo) != 1:
    raise RuntimeError(
        "Palermo was not preserved as exactly one row after saving."
    )

saved_palermo_row = saved_palermo.iloc[0]

for column_name, expected_value in (
    palermo_enrichment_values.items()
):
    if saved_palermo_row[column_name] != expected_value:
        raise RuntimeError(
            f"Saved Palermo {column_name!r} did not match "
            f"{expected_value!r}."
        )

print("Permanent reference rows:", len(course_locations))
print(
    "Changed enrichment columns this run:",
    changed_enrichment_columns
    if changed_enrichment_columns
    else "none",
)
print("Palermo address and provider lineage saved.")

display(
    saved_palermo[
        [
            "candidate_course_label",
            "physical_venue_name",
            "address_line",
            "suburb",
            "locality",
            "region",
            "postcode",
            "country",
            "provider_place_id",
            "provider_display_name",
        ]
    ]
)

Permanent reference rows: 394
Changed enrichment columns this run: none
Palermo address and provider lineage saved.


,candidate_course_label,physical_venue_name,address_line,suburb,locality,region,postcode,country,provider_place_id,provider_display_name
1,Palermo,Hipódromo Argentino de Palermo,4101 Avenida Del Libertador,Palermo,Buenos Aires,Autonomous City of Buenos Aires,C1426BSD,Argentina,way:18772836,"Hipódromo Argentino de Palermo, 4101, Avenida ..."


## Production geocoding pipeline

The exploratory La Plata and Palermo work above established the safe operating rules:

- preserve every external request in a deterministic cache;
- remove source-only suffixes such as `(AW)` from provider queries;
- accept only named racecourse venue objects in the expected jurisdiction;
- reject roads, neighbourhoods, accommodation, casinos, parking and nearby facilities;
- write only unambiguous high-confidence matches;
- export every unresolved or ambiguous course to a manual-review queue.

The cells below replace the development batches with one rerunnable production workflow. A complete rerun reuses cached requests and does not duplicate cache records.


In [53]:
# Define the permanent reference enrichment columns and processing controls.
#
# Set RUN_FULL_GEOCODING to False when validating notebook structure without
# contacting Nominatim. With it set to True, every unresolved course is
# processed. Cached exact requests are reused, so later reruns are much faster.

LOCATION_ENRICHMENT_COLUMNS = [
    "address_line",
    "suburb",
    "postcode",
    "provider_place_id",
    "provider_display_name",
]

for column_name in LOCATION_ENRICHMENT_COLUMNS:
    if column_name not in course_locations.columns:
        course_locations[column_name] = pd.NA

RUN_FULL_GEOCODING = True
MAX_UNRESOLVED_COURSES = None  # Use an integer for a deliberately smaller run.

manual_review_path = course_locations_path.with_name(
    "course_location_manual_review.csv"
)
geocoding_run_summary_path = course_locations_path.with_name(
    "course_location_geocoding_run_summary.csv"
)

print("Production controls prepared.")
print("Run full geocoding:", RUN_FULL_GEOCODING)
print("Course limit:", MAX_UNRESOLVED_COURSES)


Production controls prepared.
Run full geocoding: True
Course limit: None


In [54]:
# Define conservative text, venue and address helpers.
#
# Automatic acceptance is deliberately narrower than the earlier exploratory
# matcher. A result must be a named leisure venue with a racecourse-like type.
# Tourism attractions, campsites and other nearby facilities remain available
# in the review queue but are never written automatically.

import re
import unicodedata


def normalise_location_text(value):
    if value is None or pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = re.sub(r"[^a-z0-9]+", " ", text.lower())
    return re.sub(r"\s+", " ", text).strip()


def clean_course_label_for_geocoding(course_label):
    cleaned = re.sub(r"\s*\([^)]*\)\s*", " ", str(course_label))
    return re.sub(r"\s+", " ", cleaned).strip()


RACECOURSE_NAME_TERMS = {
    "racecourse",
    "racetrack",
    "race track",
    "hipodromo",
    "hippodrome",
    "galopp",
    "turf club",
    "jockey club",
}

AUTOMATIC_PROVIDER_CLASSES = {"leisure"}

AUTOMATIC_PROVIDER_TYPES = {
    "track",
    "sports_centre",
    "raceway",
    "stadium",
    "horse_racing",
    "equestrian",
}

SUSPICIOUS_PRIMARY_NAME_TERMS = {
    "caravan",
    "camping",
    "camp site",
    "campsite",
    "hotel",
    "casino",
    "restaurant",
    "parking",
    "station",
    "bus stop",
    "bicycle",
    "golf",
    "apartments",
    "residence",
}


def result_provider_place_id(result):
    osm_type = result.get("osm_type")
    osm_id = result.get("osm_id")

    if osm_type is None or osm_id is None:
        return pd.NA

    return f"{osm_type}:{osm_id}"


def result_primary_name(result):
    address = result.get("address") or {}
    namedetails = result.get("namedetails") or {}

    return (
        result.get("name")
        or namedetails.get("official_name")
        or namedetails.get("name")
        or address.get("leisure")
        or address.get("stadium")
        or address.get("raceway")
        or address.get("sports_centre")
    )


def result_primary_name_text(result):
    return normalise_location_text(result_primary_name(result))


def course_label_words(course_label):
    return {
        word
        for word in normalise_location_text(
            clean_course_label_for_geocoding(course_label)
        ).split()
        if len(word) >= 3
    }


def automatic_candidate_assessment(
    *,
    course_label,
    expected_country_code,
    result,
):
    address = result.get("address") or {}
    primary_name_text = result_primary_name_text(result)
    label_words = course_label_words(course_label)

    provider_class = normalise_location_text(
        result.get("class") or result.get("category")
    ).replace(" ", "_")
    provider_type = normalise_location_text(
        result.get("type")
    ).replace(" ", "_")

    country_code_match = (
        normalise_location_text(address.get("country_code"))
        == normalise_location_text(expected_country_code)
    )
    course_name_match = bool(label_words) and label_words.issubset(
        set(primary_name_text.split())
    )
    racecourse_name_match = any(
        term in primary_name_text
        for term in RACECOURSE_NAME_TERMS
    )
    provider_object_match = (
        provider_class in AUTOMATIC_PROVIDER_CLASSES
        and provider_type in AUTOMATIC_PROVIDER_TYPES
    )
    suspicious_name_match = any(
        term in primary_name_text
        for term in SUSPICIOUS_PRIMARY_NAME_TERMS
    )

    automatic_eligible = all(
        [
            country_code_match,
            course_name_match,
            racecourse_name_match,
            provider_object_match,
            not suspicious_name_match,
        ]
    )

    return {
        "country_code_match": country_code_match,
        "course_name_match": course_name_match,
        "racecourse_name_match": racecourse_name_match,
        "provider_object_match": provider_object_match,
        "suspicious_name_match": suspicious_name_match,
        "automatic_eligible": automatic_eligible,
        "provider_class": provider_class,
        "provider_type": provider_type,
    }


def extract_course_address_fields(result):
    address = result.get("address") or {}

    address_line_parts = [
        str(value)
        for value in [
            address.get("house_number"),
            address.get("road"),
        ]
        if value
    ]

    latitude = pd.to_numeric(result.get("lat"), errors="coerce")
    longitude = pd.to_numeric(result.get("lon"), errors="coerce")

    return {
        "physical_venue_name": result_primary_name(result),
        "address_line": (
            " ".join(address_line_parts)
            if address_line_parts
            else pd.NA
        ),
        "suburb": (
            address.get("suburb")
            or address.get("neighbourhood")
        ),
        "locality": (
            address.get("city")
            or address.get("town")
            or address.get("municipality")
            or address.get("village")
        ),
        "region": (
            address.get("state")
            or address.get("province")
            or address.get("region")
        ),
        "postcode": address.get("postcode"),
        "country": address.get("country"),
        "provider_place_id": result_provider_place_id(result),
        "provider_display_name": result.get("display_name"),
        "latitude": latitude,
        "longitude": longitude,
    }


def derive_validated_timezone(latitude, longitude):
    if pd.isna(latitude) or pd.isna(longitude):
        return pd.NA

    timezone_name = timezone_finder.timezone_at(
        lat=float(latitude),
        lng=float(longitude),
    )

    if timezone_name is None:
        return pd.NA

    try:
        ZoneInfo(timezone_name)
    except Exception:
        return pd.NA

    return timezone_name


print("Conservative production matching helpers defined.")


Conservative production matching helpers defined.


In [55]:
# Define ordered query planning and deterministic cache lookup.

def build_course_geocoding_queries(course_label, jurisdiction):
    if jurisdiction not in JURISDICTION_GEOCODING_RULES:
        raise KeyError(
            f"No geocoding rule exists for jurisdiction {jurisdiction!r}."
        )

    if jurisdiction not in JURISDICTION_VENUE_TERMS:
        raise KeyError(
            f"No venue-term rule exists for jurisdiction {jurisdiction!r}."
        )

    jurisdiction_rule = JURISDICTION_GEOCODING_RULES[jurisdiction]
    query_country = jurisdiction_rule["query_country"]
    country_codes = sorted(jurisdiction_rule["country_codes"])

    if len(country_codes) != 1:
        raise RuntimeError(
            f"Expected one country code for {jurisdiction!r}; "
            f"found {country_codes}."
        )

    country_code = country_codes[0]
    query_label = clean_course_label_for_geocoding(course_label)

    query_rows = []

    for query_order, venue_term in enumerate(
        JURISDICTION_VENUE_TERMS[jurisdiction],
        start=1,
    ):
        exact_query = f"{query_label} {venue_term}, {query_country}"

        cache_record_id = build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=course_label,
            candidate_jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )

        query_rows.append(
            {
                "query_order": query_order,
                "venue_term": venue_term,
                "exact_query": exact_query,
                "country_code_filter": country_code,
                "cache_record_id": cache_record_id,
            }
        )

    return pd.DataFrame(query_rows)


def get_cache_manifest_row(cache_record_id):
    matching_rows = geocoding_cache.loc[
        geocoding_cache["cache_record_id"]
        .astype(str)
        .eq(str(cache_record_id))
    ]

    if matching_rows.empty:
        return None

    if len(matching_rows) != 1:
        raise RuntimeError(
            "Expected one manifest row for cache record "
            f"{cache_record_id!r}; found {len(matching_rows)}."
        )

    return matching_rows.iloc[0]


def get_raw_response_record(cache_record_id):
    matching_record = None

    with geocoding_raw_responses_path.open("r", encoding="utf-8") as raw_file:
        for line in raw_file:
            if not line.strip():
                continue

            record = json.loads(line)

            if str(record.get("cache_record_id")) != str(cache_record_id):
                continue

            if matching_record is not None:
                raise RuntimeError(
                    "Duplicate raw response records found for "
                    f"{cache_record_id!r}."
                )

            matching_record = record

    return matching_record


print("Production query and cache helpers defined.")


Production query and cache helpers defined.


In [56]:
# Define the cache-aware request function.
#
# Every exact query is sent at most once. Errors and zero-result responses are
# cached as outcomes rather than silently retried on every notebook run.

def request_or_reuse_geocoding_query(
    *,
    course_label,
    jurisdiction,
    exact_query,
    country_code_filter,
    cache_record_id,
):
    global geocoding_cache

    manifest_row = get_cache_manifest_row(cache_record_id)

    if manifest_row is not None:
        raw_record = get_raw_response_record(cache_record_id)

        if raw_record is None:
            raise RuntimeError(
                "Manifest row exists without a raw response for "
                f"{cache_record_id!r}."
            )

        return {
            "request_action": "reused_cached_result",
            "request_status": manifest_row["request_status"],
            "results": raw_record.get("results", []),
        }

    requested_at_utc = datetime.now(timezone.utc).isoformat()
    request_status = "pending"
    raw_results = []
    error_type = None
    error_message = None

    try:
        locations = rate_limited_geocode(
            exact_query,
            country_codes=country_code_filter,
            **NOMINATIM_REQUEST_OPTIONS,
        )

        if locations is None:
            locations = []

        raw_results = [location.raw for location in locations]
        request_status = (
            "success_with_results"
            if raw_results
            else "success_no_results"
        )

    except Exception as exc:
        request_status = "request_error"
        error_type = type(exc).__name__
        error_message = str(exc)

    raw_record = {
        "raw_response_record_id": cache_record_id,
        "cache_record_id": cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": course_label,
        "candidate_jurisdiction": jurisdiction,
        "exact_query": exact_query,
        "country_code_filter": country_code_filter,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "error_type": error_type,
        "error_message": error_message,
        "results": raw_results,
    }

    with geocoding_raw_responses_path.open("a", encoding="utf-8") as raw_file:
        raw_file.write(
            json.dumps(raw_record, ensure_ascii=False) + "\n"
        )

    manifest_row = {
        "cache_record_id": cache_record_id,
        "provider": NOMINATIM_PROVIDER,
        "candidate_course_label": course_label,
        "candidate_jurisdiction": jurisdiction,
        "exact_query": exact_query,
        "country_code_filter": country_code_filter,
        "result_limit": NOMINATIM_RESULT_LIMIT,
        "requested_at_utc": requested_at_utc,
        "request_status": request_status,
        "http_status": pd.NA,
        "result_count": len(raw_results),
        "selected_result_index": pd.NA,
        "selected_provider_place_id": pd.NA,
        "selected_display_name": pd.NA,
        "selected_latitude": pd.NA,
        "selected_longitude": pd.NA,
        "country_code_match": pd.NA,
        "venue_type_match": pd.NA,
        "name_match_status": pd.NA,
        "review_status": "not_reviewed",
        "review_notes": (
            error_message
            if request_status == "request_error"
            else pd.NA
        ),
        "raw_response_record_id": cache_record_id,
    }

    geocoding_cache = pd.concat(
        [
            geocoding_cache,
            pd.DataFrame(
                [manifest_row],
                columns=GEOCODING_CACHE_COLUMNS,
            ),
        ],
        ignore_index=True,
    )

    geocoding_cache.to_csv(geocoding_cache_path, index=False)

    return {
        "request_action": "sent_and_cached",
        "request_status": request_status,
        "results": raw_results,
    }


print("Production cache-aware request helper defined.")


Production cache-aware request helper defined.


In [57]:
# Define one complete course decision.
#
# All jurisdiction-specific queries are considered. Automatic selection occurs
# only when they collectively identify exactly one eligible provider object.
# Multiple eligible objects, weak objects, errors and zero results go to review.

def process_course_geocoding(course_label, jurisdiction):
    query_plan = build_course_geocoding_queries(
        course_label,
        jurisdiction,
    )

    request_rows = []
    review_rows = []
    eligible_candidates = {}

    for query_row in query_plan.itertuples(index=False):
        outcome = request_or_reuse_geocoding_query(
            course_label=course_label,
            jurisdiction=jurisdiction,
            exact_query=query_row.exact_query,
            country_code_filter=query_row.country_code_filter,
            cache_record_id=query_row.cache_record_id,
        )

        request_rows.append(
            {
                "query_order": query_row.query_order,
                "exact_query": query_row.exact_query,
                "cache_record_id": query_row.cache_record_id,
                "request_action": outcome["request_action"],
                "request_status": outcome["request_status"],
                "result_count": len(outcome["results"]),
            }
        )

        for result_index, result in enumerate(outcome["results"]):
            assessment = automatic_candidate_assessment(
                course_label=course_label,
                expected_country_code=query_row.country_code_filter,
                result=result,
            )

            place_id = result_provider_place_id(result)
            review_row = {
                "candidate_course_label": course_label,
                "candidate_jurisdiction": jurisdiction,
                "exact_query": query_row.exact_query,
                "result_index": result_index,
                "provider_place_id": place_id,
                "provider_name": result_primary_name(result),
                "provider_display_name": result.get("display_name"),
                **assessment,
            }
            review_rows.append(review_row)

            if assessment["automatic_eligible"]:
                eligible_candidates[str(place_id)] = {
                    "result": result,
                    "query": query_row.exact_query,
                    "cache_record_id": query_row.cache_record_id,
                    "result_index": result_index,
                    "assessment": assessment,
                }

    if len(eligible_candidates) == 1:
        candidate = next(iter(eligible_candidates.values()))
        selected_fields = extract_course_address_fields(
            candidate["result"]
        )
        selected_fields["iana_timezone"] = derive_validated_timezone(
            selected_fields["latitude"],
            selected_fields["longitude"],
        )

        if pd.notna(selected_fields["iana_timezone"]):
            decision = "single_strong_match"
            reason = (
                "Exactly one named leisure racecourse object passed every "
                "country, name, object-type and facility exclusion check."
            )
        else:
            decision = "manual_review_required"
            reason = (
                "One strong venue object was found, but its IANA timezone "
                "could not be validated."
            )
    else:
        candidate = None
        selected_fields = None
        decision = "manual_review_required"

        if len(eligible_candidates) == 0:
            reason = (
                "No returned object passed every conservative automatic "
                "acceptance check."
            )
        else:
            reason = (
                "More than one distinct provider object passed the automatic "
                "checks, so selection is ambiguous."
            )

    return {
        "course_label": course_label,
        "jurisdiction": jurisdiction,
        "decision": decision,
        "decision_reason": reason,
        "selected_candidate": candidate,
        "selected_fields": selected_fields,
        "request_summary": pd.DataFrame(request_rows),
        "review_rows": pd.DataFrame(review_rows),
    }


print("Production single-course decision helper defined.")


Production single-course decision helper defined.


In [58]:
# Define automatic writing and manifest review updates.

def recalculate_reusable_location(reference):
    return (
        reference["latitude"].notna()
        & reference["longitude"].notna()
        & reference["iana_timezone"].notna()
        & reference["location_validation_status"].isin(
            {"manually_validated", "automatically_validated"}
        )
    )


def write_automatic_course_location(processor_result):
    global course_locations
    global geocoding_cache

    if processor_result["decision"] != "single_strong_match":
        raise ValueError(
            "Only single_strong_match results may be written automatically."
        )

    course_label = processor_result["course_label"]
    jurisdiction = processor_result["jurisdiction"]
    candidate = processor_result["selected_candidate"]
    selected_fields = processor_result["selected_fields"]

    reference_mask = (
        course_locations["candidate_course_label"].eq(course_label)
        & course_locations["candidate_jurisdiction"].eq(jurisdiction)
    )

    if int(reference_mask.sum()) != 1:
        raise RuntimeError(
            f"Expected one reference row for {course_label!r}, "
            f"{jurisdiction!r}; found {int(reference_mask.sum())}."
        )

    permanent_values = {
        **selected_fields,
        "location_evidence": (
            "Nominatim conservative automatic selection from query "
            f"{candidate['query']!r}; selected result index "
            f"{candidate['result_index']}."
        ),
        "location_validation_status": "automatically_validated",
    }

    for column_name, column_value in permanent_values.items():
        course_locations.loc[
            reference_mask,
            column_name,
        ] = column_value

    course_locations["has_reusable_location"] = (
        recalculate_reusable_location(course_locations)
    )
    course_locations.to_csv(course_locations_path, index=False)

    manifest_mask = (
        geocoding_cache["cache_record_id"]
        .astype(str)
        .eq(str(candidate["cache_record_id"]))
    )

    if int(manifest_mask.sum()) != 1:
        raise RuntimeError(
            "Expected one selected manifest row for "
            f"{candidate['cache_record_id']!r}."
        )

    geocoding_cache.loc[
        manifest_mask,
        [
            "selected_result_index",
            "selected_provider_place_id",
            "selected_display_name",
            "selected_latitude",
            "selected_longitude",
            "country_code_match",
            "venue_type_match",
            "name_match_status",
            "review_status",
            "review_notes",
        ],
    ] = [
        candidate["result_index"],
        selected_fields["provider_place_id"],
        selected_fields["provider_display_name"],
        selected_fields["latitude"],
        selected_fields["longitude"],
        candidate["assessment"]["country_code_match"],
        candidate["assessment"]["provider_object_match"],
        (
            "cleaned_course_label_matches_primary_venue_name"
            if candidate["assessment"]["course_name_match"]
            else "no_primary_name_match"
        ),
        "automatically_selected",
        (
            "Exactly one provider object passed the conservative production "
            "acceptance rule."
        ),
    ]

    geocoding_cache.to_csv(geocoding_cache_path, index=False)


print("Production automatic writer defined.")


Production automatic writer defined.


In [59]:
# Process every currently unresolved course and checkpoint after each decision.
#
# The final review CSV contains one row per provider alternative. Courses with
# no alternatives still receive a summary row in the run-summary CSV.

def run_production_geocoding(
    *,
    max_courses=None,
):
    global course_locations

    course_locations["has_reusable_location"] = (
        recalculate_reusable_location(course_locations)
    )

    unresolved = (
        course_locations.loc[
            ~course_locations["has_reusable_location"]
        ]
        .sort_values(
            [
                "provisional_races",
                "candidate_jurisdiction",
                "candidate_course_label",
            ],
            ascending=[False, True, True],
        )
        .copy()
    )

    if max_courses is not None:
        unresolved = unresolved.head(int(max_courses))

    summary_rows = []
    review_frames = []

    for course_row in unresolved.itertuples(index=False):
        result = process_course_geocoding(
            course_row.candidate_course_label,
            course_row.candidate_jurisdiction,
        )

        if result["decision"] == "single_strong_match":
            write_automatic_course_location(result)

        request_summary = result["request_summary"]

        summary_rows.append(
            {
                "candidate_course_label": course_row.candidate_course_label,
                "candidate_jurisdiction": course_row.candidate_jurisdiction,
                "provisional_races": course_row.provisional_races,
                "decision": result["decision"],
                "decision_reason": result["decision_reason"],
                "physical_venue_name": (
                    result["selected_fields"]["physical_venue_name"]
                    if result["selected_fields"] is not None
                    else pd.NA
                ),
                "iana_timezone": (
                    result["selected_fields"]["iana_timezone"]
                    if result["selected_fields"] is not None
                    else pd.NA
                ),
                "requests_sent": int(
                    request_summary["request_action"]
                    .eq("sent_and_cached")
                    .sum()
                ),
                "requests_reused": int(
                    request_summary["request_action"]
                    .eq("reused_cached_result")
                    .sum()
                ),
            }
        )

        if not result["review_rows"].empty:
            review_frame = result["review_rows"].copy()
            review_frame["course_decision"] = result["decision"]
            review_frame["decision_reason"] = result["decision_reason"]
            review_frames.append(review_frame)

    run_summary = pd.DataFrame(summary_rows)

    manual_review = (
        pd.concat(review_frames, ignore_index=True)
        if review_frames
        else pd.DataFrame()
    )

    if not manual_review.empty:
        manual_review = manual_review.loc[
            manual_review["course_decision"]
            .eq("manual_review_required")
        ].copy()

    run_summary.to_csv(
        geocoding_run_summary_path,
        index=False,
    )
    manual_review.to_csv(
        manual_review_path,
        index=False,
    )

    course_locations = pd.read_csv(course_locations_path)
    course_locations["has_reusable_location"] = (
        recalculate_reusable_location(course_locations)
    )

    return run_summary, manual_review


if RUN_FULL_GEOCODING:
    geocoding_run_summary, course_location_manual_review = (
        run_production_geocoding(
            max_courses=MAX_UNRESOLVED_COURSES
        )
    )

    print("Courses processed:", len(geocoding_run_summary))
    print(
        "Automatically resolved:",
        int(
            geocoding_run_summary["decision"]
            .eq("single_strong_match")
            .sum()
        ),
    )
    print(
        "Manual review required:",
        int(
            geocoding_run_summary["decision"]
            .eq("manual_review_required")
            .sum()
        ),
    )
    print(
        "New requests:",
        int(geocoding_run_summary["requests_sent"].sum()),
    )
    print(
        "Cached requests reused:",
        int(geocoding_run_summary["requests_reused"].sum()),
    )

    display(geocoding_run_summary.head(25))
else:
    geocoding_run_summary = pd.DataFrame()
    course_location_manual_review = pd.DataFrame()
    print("Production geocoding skipped by configuration.")


Courses processed: 358
Automatically resolved: 90
Manual review required: 268
New requests: 529
Cached requests reused: 28


,candidate_course_label,candidate_jurisdiction,provisional_races,decision,decision_reason,physical_venue_name,iana_timezone,requests_sent,requests_reused
0,Kempton (AW),Great Britain,4787,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
1,Lingfield (AW),Great Britain,4489,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
2,Sha Tin,Hong Kong,4160,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
3,Dundalk (AW),Ireland,3318,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
4,Happy Valley,Hong Kong,2687,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
5,Doncaster,Great Britain,2660,single_strong_match,Exactly one named leisure racecourse object pa...,Doncaster Racecourse,Europe/London,0,1
6,Saint-Cloud,France,2558,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,2
7,Ayr,Great Britain,2229,manual_review_required,More than one distinct provider object passed ...,NaN,NaN,0,1
8,Chepstow,Great Britain,2225,manual_review_required,No returned object passed every conservative a...,NaN,NaN,0,1
9,Haydock,Great Britain,2166,manual_review_required,More than one distinct provider object passed ...,NaN,NaN,0,1


## Final validation and outputs

The permanent reference is considered complete for analysis when every accepted row has a valid coordinate pair and IANA timezone, while every unaccepted row is explicitly available in the manual-review output. “Complete” therefore does not mean guessing every venue: ambiguous cases remain unresolved by design.


In [60]:
# Validate permanent outputs and report final coverage.

course_locations = pd.read_csv(course_locations_path)
course_locations["has_reusable_location"] = (
    recalculate_reusable_location(course_locations)
)

duplicate_course_keys = (
    course_locations.groupby(
        [
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        dropna=False,
    )
    .size()
    .loc[lambda counts: counts > 1]
)

if not duplicate_course_keys.empty:
    raise RuntimeError(
        "Duplicate course-location identity rows remain in the reference."
    )

accepted_rows = course_locations.loc[
    course_locations["has_reusable_location"]
].copy()

invalid_accepted_rows = accepted_rows.loc[
    accepted_rows[
        [
            "physical_venue_name",
            "latitude",
            "longitude",
            "iana_timezone",
            "location_validation_status",
        ]
    ].isna().any(axis=1)
]

if not invalid_accepted_rows.empty:
    raise RuntimeError(
        "At least one accepted course-location row is incomplete."
    )

invalid_timezone_rows = []

for row in accepted_rows.itertuples(index=False):
    try:
        ZoneInfo(row.iana_timezone)
    except Exception:
        invalid_timezone_rows.append(
            {
                "candidate_course_label": row.candidate_course_label,
                "candidate_jurisdiction": row.candidate_jurisdiction,
                "iana_timezone": row.iana_timezone,
            }
        )

if invalid_timezone_rows:
    raise RuntimeError(
        "At least one accepted row has an invalid IANA timezone."
    )

final_status = pd.DataFrame(
    [
        {
            "measure": "Permanent course identities",
            "value": len(course_locations),
        },
        {
            "measure": "Reusable validated locations",
            "value": int(
                course_locations["has_reusable_location"].sum()
            ),
        },
        {
            "measure": "Manually validated locations",
            "value": int(
                course_locations["location_validation_status"]
                .eq("manually_validated")
                .sum()
            ),
        },
        {
            "measure": "Automatically validated locations",
            "value": int(
                course_locations["location_validation_status"]
                .eq("automatically_validated")
                .sum()
            ),
        },
        {
            "measure": "Still unresolved",
            "value": int(
                (~course_locations["has_reusable_location"]).sum()
            ),
        },
        {
            "measure": "Geocoding manifest rows",
            "value": len(geocoding_cache),
        },
    ]
)

display(final_status)

print()
print("Permanent reference:", course_locations_path)
print("Request manifest:", geocoding_cache_path)
print("Raw response cache:", geocoding_raw_responses_path)
print("Run summary:", geocoding_run_summary_path)
print("Manual-review queue:", manual_review_path)


,measure,value
0,Permanent course identities,394
1,Reusable validated locations,126
2,Manually validated locations,2
3,Automatically validated locations,124
4,Still unresolved,268
5,Geocoding manifest rows,605



Permanent reference: /home/rob/Documents/inside-rails-horse-racing/data/reference/course_locations.csv
Request manifest: /home/rob/Documents/inside-rails-horse-racing/data/cache/course_geocoding_cache.csv
Raw response cache: /home/rob/Documents/inside-rails-horse-racing/data/cache/course_geocoding_responses.jsonl
Run summary: /home/rob/Documents/inside-rails-horse-racing/data/reference/course_location_geocoding_run_summary.csv
Manual-review queue: /home/rob/Documents/inside-rails-horse-racing/data/reference/course_location_manual_review.csv


## Stage 2: Complete timezone coverage for off-time interpretation

The conservative geocoding stage established 126 reusable course locations, but 268 course identities remain without a validated physical venue.

That does not necessarily mean their timezone is unknown.

Notebook 11 needs the timezone that applies to each course’s published off time. For many jurisdictions, all relevant courses use one IANA timezone, so timezone assignment can be made safely at jurisdiction level even when the exact racecourse address has not been validated.

This stage therefore separates two questions:

1. **Where exactly is the physical venue?**
2. **Which timezone governs its local off time?**

The validated venue coordinates remain the preferred evidence. For unresolved venues, jurisdiction-level timezone defaults may be used only where the jurisdiction is demonstrably single-timezone for the racing data represented here. Jurisdictions spanning multiple relevant timezones must remain subject to course-level resolution.

The next profile measures current timezone coverage by jurisdiction and by provisional race volume before any defaults are assigned.

In [61]:
# Profile timezone coverage by jurisdiction.
#
# This cell is read-only:
# - it sends no external requests;
# - it does not modify course_locations.csv;
# - it does not assign any new timezones.
#
# The purpose is to identify:
# - jurisdictions where every validated course uses one timezone;
# - jurisdictions with multiple observed timezones;
# - jurisdictions with no validated timezone evidence yet;
# - the number of provisional races affected by unresolved timezones.
#
# We will use this evidence to separate safe jurisdiction-level defaults from
# jurisdictions that still require course-level resolution.

timezone_jurisdiction_profile = (
    course_locations
    .groupby(
        "candidate_jurisdiction",
        dropna=False,
    )
    .agg(
        course_identities=(
            "candidate_course_label",
            "size",
        ),
        provisional_races=(
            "provisional_races",
            "sum",
        ),
        courses_with_timezone=(
            "iana_timezone",
            lambda values: values.notna().sum(),
        ),
        resolved_provisional_races=(
            "provisional_races",
            lambda values: values[
                course_locations.loc[
                    values.index,
                    "iana_timezone",
                ].notna()
            ].sum(),
        ),
        distinct_validated_timezones=(
            "iana_timezone",
            lambda values: values.dropna().nunique(),
        ),
        observed_timezones=(
            "iana_timezone",
            lambda values: ", ".join(
                sorted(
                    values.dropna().astype(str).unique()
                )
            ),
        ),
    )
    .reset_index()
)

timezone_jurisdiction_profile[
    "unresolved_courses"
] = (
    timezone_jurisdiction_profile[
        "course_identities"
    ]
    - timezone_jurisdiction_profile[
        "courses_with_timezone"
    ]
)

timezone_jurisdiction_profile[
    "unresolved_provisional_races"
] = (
    timezone_jurisdiction_profile[
        "provisional_races"
    ]
    - timezone_jurisdiction_profile[
        "resolved_provisional_races"
    ]
)

timezone_jurisdiction_profile[
    "course_timezone_coverage_pct"
] = (
    100
    * timezone_jurisdiction_profile[
        "courses_with_timezone"
    ]
    / timezone_jurisdiction_profile[
        "course_identities"
    ]
).round(1)

timezone_jurisdiction_profile[
    "race_timezone_coverage_pct"
] = (
    100
    * timezone_jurisdiction_profile[
        "resolved_provisional_races"
    ]
    / timezone_jurisdiction_profile[
        "provisional_races"
    ]
).round(1)

timezone_jurisdiction_profile = (
    timezone_jurisdiction_profile
    .sort_values(
        [
            "unresolved_provisional_races",
            "unresolved_courses",
            "candidate_jurisdiction",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(timezone_jurisdiction_profile)

,candidate_jurisdiction,course_identities,provisional_races,courses_with_timezone,resolved_provisional_races,distinct_validated_timezones,observed_timezones,unresolved_courses,unresolved_provisional_races,course_timezone_coverage_pct,race_timezone_coverage_pct
0,Great Britain,65,105688,41,68254,1,Europe/London,24,37434,63.1,64.6
1,Ireland,27,29215,15,15098,1,Europe/Dublin,12,14117,55.6,51.7
2,Hong Kong,2,6847,0,0,0,,2,6847,0.0,0.0
3,France,73,19361,21,14986,1,Europe/Paris,52,4375,28.8,77.4
4,United States,56,5863,5,1663,3,"America/Boise, America/Los_Angeles, America/Ne...",51,4200,8.9,28.4
5,Australia,51,3798,20,1366,5,"Australia/Adelaide, Australia/Brisbane, Austra...",31,2432,39.2,36.0
6,United Arab Emirates,5,2219,1,606,1,Asia/Dubai,4,1613,20.0,27.3
7,Japan,21,1470,9,744,1,Asia/Tokyo,12,726,42.9,50.6
8,Germany,17,710,0,0,0,,17,710,0.0,0.0
9,Canada,4,462,1,23,1,America/Vancouver,3,439,25.0,5.0


### Timezone coverage findings

The current reference resolves 126 of 394 course identities, covering a larger share of races than courses because many high-volume venues were successfully matched.

The unresolved workload is concentrated in a small number of jurisdictions:

- Great Britain: 37,434 unresolved provisional races
- Ireland: 14,117
- Hong Kong: 6,847
- France: 4,375
- United States: 4,200
- Australia: 2,432
- United Arab Emirates: 1,613

The validated evidence shows a single observed timezone for Great Britain, Ireland, France, Japan, the United Arab Emirates, South Africa, Argentina, Brazil and New Zealand. These jurisdictions are candidates for explicit jurisdiction-level timezone assignment, subject to a documented policy rather than further venue geocoding.

The United States and Australia already contain multiple validated timezones and therefore cannot use one jurisdiction-wide default.

Canada must also remain course-specific: the single observed timezone reflects only the one validated Canadian course and does not establish a national default.

Jurisdictions with no validated venue evidence may still receive an explicit timezone where the jurisdiction has one relevant civil timezone, but those assignments must be recorded separately from venue-derived timezones.

The next stage will therefore classify jurisdictions into:

1. safe explicit jurisdiction-level timezone assignments;
2. course-level timezone resolution required;
3. unresolved pending further evidence.

In [63]:
# Define an explicit timezone-resolution policy by racing jurisdiction.
#
# This is deliberately manual and reviewable. We are not inferring that a
# jurisdiction is safe merely because the currently validated sample contains
# one observed timezone.
#
# Policy classes:
# - jurisdiction_default:
#     All unresolved courses represented by this racing jurisdiction may use
#     the stated IANA timezone for off-time interpretation.
# - course_level_required:
#     The jurisdiction spans multiple relevant timezones, so each unresolved
#     course must be resolved separately.
#
# This cell does not modify course_locations.csv.

jurisdiction_timezone_policy = {
    "Great Britain": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/London",
        "reason": "British racecourses use UK civil time.",
    },
    "Ireland": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Dublin",
        "reason": "Irish racecourses use Irish civil time.",
    },
    "Hong Kong": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Hong_Kong",
        "reason": "Both represented Hong Kong racecourses use Hong Kong time.",
    },
    "France": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Paris",
        "reason": "The dataset represents mainland French racing.",
    },
    "United Arab Emirates": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Dubai",
        "reason": "Represented UAE racecourses use Gulf Standard Time.",
    },
    "Japan": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Tokyo",
        "reason": "Japan uses one civil timezone.",
    },
    "Germany": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Berlin",
        "reason": "German racecourses use German civil time.",
    },
    "Jersey": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Jersey",
        "reason": "The represented jurisdiction has one applicable timezone.",
    },
    "Italy": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Rome",
        "reason": "Italian racecourses use Italian civil time.",
    },
    "Chile": {
        "policy": "jurisdiction_default",
        "iana_timezone": "America/Santiago",
        "reason": "The represented racecourses are mainland Chilean venues.",
    },
    "New Zealand": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Pacific/Auckland",
        "reason": "The represented racecourses use mainland New Zealand time.",
    },
    "South Africa": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Africa/Johannesburg",
        "reason": "South Africa uses one civil timezone.",
    },
    "Argentina": {
        "policy": "jurisdiction_default",
        "iana_timezone": "America/Argentina/Buenos_Aires",
        "reason": "Argentina currently uses one national civil offset.",
    },
    "Sweden": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Stockholm",
        "reason": "Swedish racecourses use Swedish civil time.",
    },
    "Bahrain": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Bahrain",
        "reason": "Bahrain has one applicable civil timezone.",
    },
    "Peru": {
        "policy": "jurisdiction_default",
        "iana_timezone": "America/Lima",
        "reason": "Peru uses one civil timezone.",
    },
    "Uruguay": {
        "policy": "jurisdiction_default",
        "iana_timezone": "America/Montevideo",
        "reason": "Uruguay uses one civil timezone.",
    },
    "Qatar": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Qatar",
        "reason": "Qatar has one applicable civil timezone.",
    },
    "Turkey": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Istanbul",
        "reason": "Turkey uses one national civil timezone.",
    },
    "Switzerland": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Zurich",
        "reason": "Swiss racecourses use Swiss civil time.",
    },
    "Saudi Arabia": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Riyadh",
        "reason": "Saudi Arabia uses one civil timezone.",
    },
    "Norway": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Oslo",
        "reason": "The represented Norwegian racing venue uses Norwegian civil time.",
    },
    "Guernsey": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Guernsey",
        "reason": "The represented jurisdiction has one applicable timezone.",
    },
    "South Korea": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Seoul",
        "reason": "South Korea uses one civil timezone.",
    },
    "Denmark": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Copenhagen",
        "reason": "The represented Danish racing venue uses Danish civil time.",
    },
    "Spain": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Madrid",
        "reason": "The represented racecourses are in mainland Spain.",
    },
    "Czech Republic": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Prague",
        "reason": "The Czech Republic uses one civil timezone.",
    },
    "China": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Shanghai",
        "reason": "China uses one official civil timezone.",
    },
    "Belgium": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Brussels",
        "reason": "Belgian racecourses use Belgian civil time.",
    },
    "Hungary": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Budapest",
        "reason": "Hungary uses one civil timezone.",
    },
    "Poland": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Europe/Warsaw",
        "reason": "Poland uses one civil timezone.",
    },
    "Singapore": {
        "policy": "jurisdiction_default",
        "iana_timezone": "Asia/Singapore",
        "reason": "Singapore uses one civil timezone.",
    },

    # These jurisdictions require venue-specific resolution.
    "United States": {
        "policy": "course_level_required",
        "iana_timezone": None,
        "reason": "The represented courses span multiple US timezones.",
    },
    "Australia": {
        "policy": "course_level_required",
        "iana_timezone": None,
        "reason": "The represented courses span multiple Australian timezones.",
    },
    "Canada": {
        "policy": "course_level_required",
        "iana_timezone": None,
        "reason": "Canadian courses may fall in several timezones.",
    },
    "Brazil": {
        "policy": "course_level_required",
        "iana_timezone": None,
        "reason": "Brazil spans multiple timezones; one observed zone is insufficient.",
    },
}

jurisdiction_timezone_policy_df = (
    pd.DataFrame.from_dict(
        jurisdiction_timezone_policy,
        orient="index",
    )
    .rename_axis("candidate_jurisdiction")
    .reset_index()
)

policy_review = (
    timezone_jurisdiction_profile
    .merge(
        jurisdiction_timezone_policy_df,
        on="candidate_jurisdiction",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "candidate_jurisdiction",
            "course_identities",
            "provisional_races",
            "unresolved_courses",
            "unresolved_provisional_races",
            "observed_timezones",
            "policy",
            "iana_timezone",
            "reason",
        ]
    ]
    .sort_values(
        [
            "unresolved_provisional_races",
            "candidate_jurisdiction",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(policy_review)

,candidate_jurisdiction,course_identities,provisional_races,unresolved_courses,unresolved_provisional_races,observed_timezones,policy,iana_timezone,reason
0,Great Britain,65,105688,24,37434,Europe/London,jurisdiction_default,Europe/London,British racecourses use UK civil time.
1,Ireland,27,29215,12,14117,Europe/Dublin,jurisdiction_default,Europe/Dublin,Irish racecourses use Irish civil time.
2,Hong Kong,2,6847,2,6847,,jurisdiction_default,Asia/Hong_Kong,Both represented Hong Kong racecourses use Hon...
3,France,73,19361,52,4375,Europe/Paris,jurisdiction_default,Europe/Paris,The dataset represents mainland French racing.
4,United States,56,5863,51,4200,"America/Boise, America/Los_Angeles, America/Ne...",course_level_required,NaN,The represented courses span multiple US timez...
5,Australia,51,3798,31,2432,"Australia/Adelaide, Australia/Brisbane, Austra...",course_level_required,NaN,The represented courses span multiple Australi...
6,United Arab Emirates,5,2219,4,1613,Asia/Dubai,jurisdiction_default,Asia/Dubai,Represented UAE racecourses use Gulf Standard ...
7,Japan,21,1470,12,726,Asia/Tokyo,jurisdiction_default,Asia/Tokyo,Japan uses one civil timezone.
8,Germany,17,710,17,710,,jurisdiction_default,Europe/Berlin,German racecourses use German civil time.
9,Canada,4,462,3,439,America/Vancouver,course_level_required,NaN,Canadian courses may fall in several timezones.


In [64]:
# Preview jurisdiction-level timezone assignments.
#
# This cell:
# - preserves every existing validated timezone;
# - assigns a proposed timezone only to unresolved courses covered by an
#   explicit jurisdiction_default policy;
# - leaves course_level_required jurisdictions unresolved;
# - does not modify course_locations.csv.

timezone_assignment_preview = (
    course_locations
    .merge(
        jurisdiction_timezone_policy_df[
            [
                "candidate_jurisdiction",
                "policy",
                "iana_timezone",
                "reason",
            ]
        ].rename(
            columns={
                "iana_timezone": "jurisdiction_default_timezone",
                "reason": "timezone_policy_reason",
            }
        ),
        on="candidate_jurisdiction",
        how="left",
        validate="many_to_one",
    )
)

timezone_assignment_preview["proposed_iana_timezone"] = (
    timezone_assignment_preview["iana_timezone"]
)

jurisdiction_default_mask = (
    timezone_assignment_preview["iana_timezone"].isna()
    & timezone_assignment_preview["policy"].eq("jurisdiction_default")
    & timezone_assignment_preview[
        "jurisdiction_default_timezone"
    ].notna()
)

timezone_assignment_preview.loc[
    jurisdiction_default_mask,
    "proposed_iana_timezone",
] = timezone_assignment_preview.loc[
    jurisdiction_default_mask,
    "jurisdiction_default_timezone",
]

timezone_assignment_preview["proposed_timezone_resolution_method"] = (
    timezone_assignment_preview["timezone_resolution_method"]
    if "timezone_resolution_method"
    in timezone_assignment_preview.columns
    else pd.Series(
        pd.NA,
        index=timezone_assignment_preview.index,
        dtype="string",
    )
)

timezone_assignment_preview.loc[
    timezone_assignment_preview["iana_timezone"].notna()
    & timezone_assignment_preview[
        "proposed_timezone_resolution_method"
    ].isna(),
    "proposed_timezone_resolution_method",
] = "validated_venue_location"

timezone_assignment_preview.loc[
    jurisdiction_default_mask,
    "proposed_timezone_resolution_method",
] = "jurisdiction_default"

preview_summary = pd.DataFrame(
    {
        "measure": [
            "Course identities",
            "Currently resolved courses",
            "New jurisdiction-default assignments",
            "Courses resolved after proposed assignments",
            "Courses still requiring course-level resolution",
            "Provisional races currently covered",
            "Additional provisional races covered",
            "Provisional races covered after proposed assignments",
            "Provisional races still unresolved",
        ],
        "value": [
            len(timezone_assignment_preview),
            int(
                timezone_assignment_preview[
                    "iana_timezone"
                ].notna().sum()
            ),
            int(jurisdiction_default_mask.sum()),
            int(
                timezone_assignment_preview[
                    "proposed_iana_timezone"
                ].notna().sum()
            ),
            int(
                timezone_assignment_preview[
                    "proposed_iana_timezone"
                ].isna().sum()
            ),
            int(
                timezone_assignment_preview.loc[
                    timezone_assignment_preview[
                        "iana_timezone"
                    ].notna(),
                    "provisional_races",
                ].sum()
            ),
            int(
                timezone_assignment_preview.loc[
                    jurisdiction_default_mask,
                    "provisional_races",
                ].sum()
            ),
            int(
                timezone_assignment_preview.loc[
                    timezone_assignment_preview[
                        "proposed_iana_timezone"
                    ].notna(),
                    "provisional_races",
                ].sum()
            ),
            int(
                timezone_assignment_preview.loc[
                    timezone_assignment_preview[
                        "proposed_iana_timezone"
                    ].isna(),
                    "provisional_races",
                ].sum()
            ),
        ],
    }
)

remaining_course_level_review = (
    timezone_assignment_preview.loc[
        timezone_assignment_preview[
            "proposed_iana_timezone"
        ].isna(),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
            "policy",
            "timezone_policy_reason",
        ],
    ]
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

display(preview_summary)
display(remaining_course_level_review)

,measure,value
0,Course identities,394
1,Currently resolved courses,126
2,New jurisdiction-default assignments,182
3,Courses resolved after proposed assignments,308
4,Courses still requiring course-level resolution,86
5,Provisional races currently covered,103433
6,Additional provisional races covered,68051
7,Provisional races covered after proposed assig...,171484
8,Provisional races still unresolved,7207


,candidate_course_label,candidate_jurisdiction,provisional_races,policy,timezone_policy_reason
0,Gulfstream Park,United States,745,course_level_required,The represented courses span multiple US timez...
1,Randwick,Australia,719,course_level_required,The represented courses span multiple Australi...
2,Caulfield,Australia,574,course_level_required,The represented courses span multiple Australi...
3,Saratoga,United States,528,course_level_required,The represented courses span multiple US timez...
4,Belmont Park,United States,482,course_level_required,The represented courses span multiple US timez...
...,...,...,...,...,...
81,Middleburg,United States,1,course_level_required,The represented courses span multiple US timez...
82,Montpelier,United States,1,course_level_required,The represented courses span multiple US timez...
83,Pine Mountain,United States,1,course_level_required,The represented courses span multiple US timez...
84,Retama Park,United States,1,course_level_required,The represented courses span multiple US timez...


In [66]:
# Inspect available location evidence for courses in multi-timezone jurisdictions.
#
# This cell does not write anything.
# It checks whether the remaining 86 courses already have coordinates,
# locality, region or provider evidence that can support course-level
# timezone assignment.

multi_timezone_courses = timezone_assignment_preview.loc[
    timezone_assignment_preview["proposed_iana_timezone"].isna()
].copy()

evidence_columns = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "provisional_races",
    "latitude",
    "longitude",
    "locality",
    "region",
    "provider_display_name",
    "provider_place_id",
]

available_evidence_columns = [
    column
    for column in evidence_columns
    if column in multi_timezone_courses.columns
]

multi_timezone_evidence = (
    multi_timezone_courses[
        available_evidence_columns
    ]
    .copy()
)

latitude_column = (
    "latitude"
    if "latitude" in multi_timezone_evidence.columns
    else "lat"
    if "lat" in multi_timezone_evidence.columns
    else None
)

longitude_column = (
    "longitude"
    if "longitude" in multi_timezone_evidence.columns
    else "lon"
    if "lon" in multi_timezone_evidence.columns
    else None
)

if latitude_column and longitude_column:
    multi_timezone_evidence["has_coordinates"] = (
        multi_timezone_evidence[latitude_column].notna()
        & multi_timezone_evidence[longitude_column].notna()
    )
else:
    multi_timezone_evidence["has_coordinates"] = False

location_text_columns = [
    column
    for column in [
        "locality",
        "region",
        "provider_display_name",
    ]
    if column in multi_timezone_evidence.columns
]

if location_text_columns:
    multi_timezone_evidence["has_location_text"] = (
        multi_timezone_evidence[
            location_text_columns
        ]
        .notna()
        .any(axis=1)
    )
else:
    multi_timezone_evidence["has_location_text"] = False

multi_timezone_evidence["evidence_bucket"] = "course_name_only"

multi_timezone_evidence.loc[
    multi_timezone_evidence["has_location_text"],
    "evidence_bucket",
] = "location_text_available"

multi_timezone_evidence.loc[
    multi_timezone_evidence["has_coordinates"],
    "evidence_bucket",
] = "coordinates_available"

evidence_summary = (
    multi_timezone_evidence
    .groupby(
        [
            "candidate_jurisdiction",
            "evidence_bucket",
        ],
        dropna=False,
    )
    .agg(
        courses=(
            "candidate_course_label",
            "size",
        ),
        provisional_races=(
            "provisional_races",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "candidate_jurisdiction",
            "provisional_races",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

multi_timezone_evidence = (
    multi_timezone_evidence
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(evidence_summary)
display(multi_timezone_evidence)

,candidate_jurisdiction,evidence_bucket,courses,provisional_races
0,Australia,course_name_only,31,2432
1,Brazil,course_name_only,1,136
2,Canada,course_name_only,3,439
3,United States,course_name_only,51,4200


,candidate_course_label,candidate_jurisdiction,provisional_races,latitude,longitude,locality,region,provider_display_name,provider_place_id,has_coordinates,has_location_text,evidence_bucket
0,Gulfstream Park,United States,745,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
1,Randwick,Australia,719,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
2,Caulfield,Australia,574,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
3,Saratoga,United States,528,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
4,Belmont Park,United States,482,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
...,...,...,...,...,...,...,...,...,...,...,...,...
81,Middleburg,United States,1,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
82,Montpelier,United States,1,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
83,Pine Mountain,United States,1,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only
84,Retama Park,United States,1,NaN,NaN,NaN,NaN,NaN,NaN,False,False,course_name_only


## Stage 3: Course-level timezone resolution for multi-timezone jurisdictions

The remaining 86 courses are in the United States, Australia, Canada and Brazil.

None currently retain coordinates, locality, region or provider evidence in the permanent course reference. These courses therefore cannot be assigned a timezone from the existing validated-location fields.

This stage uses a narrower standard than physical venue validation.

For timezone assignment, a result may be accepted where it provides sufficiently clear evidence of the course’s city, state, province or region, even if the exact racecourse object itself cannot be validated. The resulting assignment must be recorded as course-level timezone resolution rather than validated venue location.

The workflow will:

1. search specifically for the named course and jurisdiction;
2. retain city, state, province or regional evidence;
3. derive the IANA timezone from the returned coordinates;
4. reject genuinely ambiguous names;
5. preserve the provider response and resolution method;
6. leave unresolved only courses for which the applicable regional timezone cannot be established safely.

The next step prepares the 86-course timezone-only work queue.

In [67]:
# Prepare the course-level timezone work queue for multi-timezone jurisdictions.
#
# This cell:
# - includes only courses still lacking a proposed timezone;
# - preserves race volume for prioritisation;
# - assigns a stable queue identifier;
# - does not send requests;
# - does not write to the permanent reference.

timezone_course_queue = (
    timezone_assignment_preview.loc[
        timezone_assignment_preview["proposed_iana_timezone"].isna(),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
        ],
    ]
    .copy()
)

timezone_course_queue = (
    timezone_course_queue
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

timezone_course_queue.insert(
    0,
    "timezone_queue_id",
    range(1, len(timezone_course_queue) + 1),
)

timezone_course_queue["timezone_query_status"] = "pending"
timezone_course_queue["timezone_query"] = pd.NA
timezone_course_queue["resolved_locality"] = pd.NA
timezone_course_queue["resolved_region"] = pd.NA
timezone_course_queue["resolved_latitude"] = pd.NA
timezone_course_queue["resolved_longitude"] = pd.NA
timezone_course_queue["resolved_iana_timezone"] = pd.NA
timezone_course_queue["timezone_resolution_method"] = pd.NA
timezone_course_queue["timezone_resolution_reason"] = pd.NA
timezone_course_queue["timezone_review_notes"] = pd.NA

queue_summary = (
    timezone_course_queue
    .groupby(
        "candidate_jurisdiction",
        dropna=False,
    )
    .agg(
        courses=("candidate_course_label", "size"),
        provisional_races=("provisional_races", "sum"),
    )
    .reset_index()
    .sort_values(
        "provisional_races",
        ascending=False,
    )
    .reset_index(drop=True)
)

assert len(timezone_course_queue) == 86
assert int(timezone_course_queue["provisional_races"].sum()) == 7207

display(queue_summary)
display(timezone_course_queue.head(25))

,candidate_jurisdiction,courses,provisional_races
0,United States,51,4200
1,Australia,31,2432
2,Canada,3,439
3,Brazil,1,136


,timezone_queue_id,candidate_course_label,candidate_jurisdiction,provisional_races,timezone_query_status,timezone_query,resolved_locality,resolved_region,resolved_latitude,resolved_longitude,resolved_iana_timezone,timezone_resolution_method,timezone_resolution_reason,timezone_review_notes
0,1,Gulfstream Park,United States,745,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2,Randwick,Australia,719,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,3,Caulfield,Australia,574,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,4,Saratoga,United States,528,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,5,Belmont Park,United States,482,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,6,Churchill Downs,United States,474,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,7,Rosehill,Australia,472,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,8,Woodbine,Canada,434,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,9,Keeneland,United States,409,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,10,Morphettville,Australia,205,pending,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [68]:
# Build targeted timezone-only geocoding queries.
#
# These queries seek enough regional evidence to identify the applicable
# timezone. They do not require the provider to return the exact racecourse
# leisure object.
#
# Multiple query forms are retained so that the later request stage can stop
# after the first sufficiently clear result.

TIMEZONE_QUERY_COUNTRY_TEXT = {
    "United States": "United States",
    "Australia": "Australia",
    "Canada": "Canada",
    "Brazil": "Brazil",
}

TIMEZONE_QUERY_VENUE_TERMS = {
    "United States": ["racecourse", "racetrack"],
    "Australia": ["racecourse"],
    "Canada": ["racecourse", "racetrack"],
    "Brazil": ["hipodromo", "racecourse"],
}


def build_timezone_queries(course_label, jurisdiction):
    course_text = str(course_label).strip()
    country_text = TIMEZONE_QUERY_COUNTRY_TEXT[jurisdiction]
    venue_terms = TIMEZONE_QUERY_VENUE_TERMS[jurisdiction]

    queries = []

    for venue_term in venue_terms:
        queries.append(
            f"{course_text} {venue_term}, {country_text}"
        )

    # A plain named-place query is useful when the provider recognises the
    # course name directly but not the racing terminology.
    queries.append(
        f"{course_text}, {country_text}"
    )

    # Preserve order while removing exact duplicates.
    return list(dict.fromkeys(queries))


timezone_query_plan = timezone_course_queue[
    [
        "timezone_queue_id",
        "candidate_course_label",
        "candidate_jurisdiction",
        "provisional_races",
    ]
].copy()

timezone_query_plan["planned_queries"] = timezone_query_plan.apply(
    lambda row: build_timezone_queries(
        row["candidate_course_label"],
        row["candidate_jurisdiction"],
    ),
    axis=1,
)

timezone_query_plan["planned_query_count"] = (
    timezone_query_plan["planned_queries"].str.len()
)

timezone_query_plan_long = (
    timezone_query_plan
    .explode("planned_queries")
    .rename(columns={"planned_queries": "timezone_query"})
    .reset_index(drop=True)
)

timezone_query_plan_long["query_sequence"] = (
    timezone_query_plan_long
    .groupby("timezone_queue_id")
    .cumcount()
    .add(1)
)

query_plan_summary = pd.DataFrame(
    {
        "measure": [
            "Courses in timezone-only queue",
            "Planned timezone queries",
            "Minimum queries per course",
            "Maximum queries per course",
        ],
        "value": [
            len(timezone_query_plan),
            len(timezone_query_plan_long),
            int(timezone_query_plan["planned_query_count"].min()),
            int(timezone_query_plan["planned_query_count"].max()),
        ],
    }
)

assert len(timezone_query_plan) == 86
assert timezone_query_plan["planned_queries"].map(bool).all()

display(query_plan_summary)
display(timezone_query_plan_long.head(40))

,measure,value
0,Courses in timezone-only queue,86
1,Planned timezone queries,227
2,Minimum queries per course,2
3,Maximum queries per course,3


,timezone_queue_id,candidate_course_label,candidate_jurisdiction,provisional_races,timezone_query,planned_query_count,query_sequence
0,1,Gulfstream Park,United States,745,"Gulfstream Park racecourse, United States",3,1
1,1,Gulfstream Park,United States,745,"Gulfstream Park racetrack, United States",3,2
2,1,Gulfstream Park,United States,745,"Gulfstream Park, United States",3,3
3,2,Randwick,Australia,719,"Randwick racecourse, Australia",2,1
4,2,Randwick,Australia,719,"Randwick, Australia",2,2
5,3,Caulfield,Australia,574,"Caulfield racecourse, Australia",2,1
6,3,Caulfield,Australia,574,"Caulfield, Australia",2,2
7,4,Saratoga,United States,528,"Saratoga racecourse, United States",3,1
8,4,Saratoga,United States,528,"Saratoga racetrack, United States",3,2
9,4,Saratoga,United States,528,"Saratoga, United States",3,3


In [69]:
# Define the timezone-only result reviewer.
#
# This reviewer is intentionally different from physical venue validation:
# - the exact racecourse object does not need to be proven;
# - results must still belong to the expected country;
# - every usable country-matching result must imply the same timezone;
# - conflicting timezone evidence remains unresolved.
#
# This cell defines the reviewer only. It sends no requests and writes no files.

from zoneinfo import ZoneInfo


TIMEZONE_COUNTRY_CODES = {
    "United States": {"us"},
    "Australia": {"au"},
    "Canada": {"ca"},
    "Brazil": {"br"},
}


def review_timezone_query_results(raw_results, jurisdiction):
    expected_country_codes = TIMEZONE_COUNTRY_CODES[jurisdiction]

    reviewed_results = []

    for result_index, result in enumerate(raw_results):
        address = result.get("address") or {}
        country_code = str(address.get("country_code") or "").lower()

        if country_code not in expected_country_codes:
            continue

        try:
            latitude = float(result["lat"])
            longitude = float(result["lon"])
        except (KeyError, TypeError, ValueError):
            continue

        derived_timezone = timezone_finder.timezone_at(
            lat=latitude,
            lng=longitude,
        )

        if not derived_timezone:
            continue

        try:
            ZoneInfo(derived_timezone)
        except Exception:
            continue

        locality = (
            address.get("city")
            or address.get("town")
            or address.get("municipality")
            or address.get("village")
            or address.get("suburb")
        )

        region = (
            address.get("state")
            or address.get("province")
            or address.get("region")
        )

        reviewed_results.append(
            {
                "result_index": result_index,
                "display_name": result.get("display_name"),
                "provider_name": result.get("name"),
                "provider_place_id": (
                    f"{result.get('osm_type')}:{result.get('osm_id')}"
                    if result.get("osm_type") and result.get("osm_id")
                    else pd.NA
                ),
                "locality": locality,
                "region": region,
                "country": address.get("country"),
                "country_code": country_code,
                "latitude": latitude,
                "longitude": longitude,
                "iana_timezone": derived_timezone,
            }
        )

    distinct_timezones = sorted(
        {
            row["iana_timezone"]
            for row in reviewed_results
        }
    )

    if not reviewed_results:
        decision = "no_usable_country_match"
        resolved_timezone = pd.NA
    elif len(distinct_timezones) == 1:
        decision = "single_timezone_consensus"
        resolved_timezone = distinct_timezones[0]
    else:
        decision = "conflicting_timezones"
        resolved_timezone = pd.NA

    return {
        "decision": decision,
        "resolved_iana_timezone": resolved_timezone,
        "distinct_timezones": distinct_timezones,
        "usable_result_count": len(reviewed_results),
        "reviewed_results": reviewed_results,
    }


print("Timezone-only result reviewer defined.")

Timezone-only result reviewer defined.


In [70]:
# Test the timezone-only workflow on the five highest-volume unresolved courses.
#
# This cell:
# - uses the existing cache-aware Nominatim request helper;
# - stops for a course after the first query producing one-timezone consensus;
# - preserves every request in the existing cache;
# - does not update course_locations.csv.

TIMEZONE_COUNTRY_CODE_FILTERS = {
    "United States": "us",
    "Australia": "au",
    "Canada": "ca",
    "Brazil": "br",
}

timezone_test_rows = []

for course_row in (
    timezone_query_plan
    .head(5)
    .itertuples(index=False)
):
    course_label = course_row.candidate_course_label
    jurisdiction = course_row.candidate_jurisdiction
    country_code_filter = TIMEZONE_COUNTRY_CODE_FILTERS[jurisdiction]

    final_decision = "all_queries_exhausted"
    resolved_timezone = pd.NA
    accepted_review = None
    queries_attempted = 0
    requests_sent = 0
    requests_reused = 0

    for exact_query in course_row.planned_queries:
        queries_attempted += 1

        cache_record_id = build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=course_label,
            candidate_jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )

        outcome = request_or_reuse_geocoding_query(
            course_label=course_label,
            jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            cache_record_id=cache_record_id,
        )

        if outcome["request_action"] == "sent_and_cached":
            requests_sent += 1
        else:
            requests_reused += 1

        review = review_timezone_query_results(
            raw_results=outcome["results"],
            jurisdiction=jurisdiction,
        )

        timezone_test_rows.append(
            {
                "candidate_course_label": course_label,
                "candidate_jurisdiction": jurisdiction,
                "provisional_races": course_row.provisional_races,
                "exact_query": exact_query,
                "request_action": outcome["request_action"],
                "request_status": outcome["request_status"],
                "raw_result_count": len(outcome["results"]),
                "review_decision": review["decision"],
                "usable_result_count": review["usable_result_count"],
                "distinct_timezones": ", ".join(
                    review["distinct_timezones"]
                ),
                "resolved_iana_timezone": (
                    review["resolved_iana_timezone"]
                ),
            }
        )

        if review["decision"] == "single_timezone_consensus":
            final_decision = "single_timezone_consensus"
            resolved_timezone = review["resolved_iana_timezone"]
            accepted_review = review
            break

    matching_test_rows = [
        row
        for row in timezone_test_rows
        if row["candidate_course_label"] == course_label
        and row["candidate_jurisdiction"] == jurisdiction
    ]

    for row in matching_test_rows:
        row["course_final_decision"] = final_decision
        row["course_resolved_timezone"] = resolved_timezone
        row["course_queries_attempted"] = queries_attempted
        row["course_requests_sent"] = requests_sent
        row["course_requests_reused"] = requests_reused

timezone_test_results = pd.DataFrame(timezone_test_rows)

timezone_test_course_summary = (
    timezone_test_results[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
            "course_final_decision",
            "course_resolved_timezone",
            "course_queries_attempted",
            "course_requests_sent",
            "course_requests_reused",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(timezone_test_course_summary)
display(
    timezone_test_results[
        [
            "candidate_course_label",
            "exact_query",
            "request_action",
            "raw_result_count",
            "review_decision",
            "usable_result_count",
            "distinct_timezones",
        ]
    ]
)

,candidate_course_label,candidate_jurisdiction,provisional_races,course_final_decision,course_resolved_timezone,course_queries_attempted,course_requests_sent,course_requests_reused
0,Gulfstream Park,United States,745,all_queries_exhausted,NaN,3,1,2
1,Randwick,Australia,719,single_timezone_consensus,Australia/Sydney,1,0,1
2,Caulfield,Australia,574,single_timezone_consensus,Australia/Melbourne,1,0,1
3,Saratoga,United States,528,all_queries_exhausted,NaN,3,1,2
4,Belmont Park,United States,482,all_queries_exhausted,NaN,3,1,2


,candidate_course_label,exact_query,request_action,raw_result_count,review_decision,usable_result_count,distinct_timezones
0,Gulfstream Park,"Gulfstream Park racecourse, United States",reused_cached_result,0,no_usable_country_match,0,
1,Gulfstream Park,"Gulfstream Park racetrack, United States",reused_cached_result,0,no_usable_country_match,0,
2,Gulfstream Park,"Gulfstream Park, United States",sent_and_cached,4,conflicting_timezones,4,"America/Chicago, America/New_York"
3,Randwick,"Randwick racecourse, Australia",reused_cached_result,1,single_timezone_consensus,1,Australia/Sydney
4,Caulfield,"Caulfield racecourse, Australia",reused_cached_result,2,single_timezone_consensus,2,Australia/Melbourne
5,Saratoga,"Saratoga racecourse, United States",reused_cached_result,0,no_usable_country_match,0,
6,Saratoga,"Saratoga racetrack, United States",reused_cached_result,0,no_usable_country_match,0,
7,Saratoga,"Saratoga, United States",sent_and_cached,5,conflicting_timezones,5,"America/Denver, America/Indiana/Indianapolis, ..."
8,Belmont Park,"Belmont Park racecourse, United States",reused_cached_result,0,no_usable_country_match,0,
9,Belmont Park,"Belmont Park racetrack, United States",reused_cached_result,0,no_usable_country_match,0,


In [71]:
# Expand the reviewed provider results from the five-course test.
#
# This lets us distinguish:
# - a genuine racecourse result mixed with irrelevant same-name places;
# - results referring only to unrelated towns, roads or neighbourhoods;
# - queries that need additional state or city qualification.
#
# No requests are sent and no files are written.

timezone_test_detail_rows = []

for course_row in (
    timezone_query_plan
    .head(5)
    .itertuples(index=False)
):
    course_label = course_row.candidate_course_label
    jurisdiction = course_row.candidate_jurisdiction
    country_code_filter = TIMEZONE_COUNTRY_CODE_FILTERS[jurisdiction]

    for exact_query in course_row.planned_queries:
        cache_record_id = build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=course_label,
            candidate_jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )

        outcome = request_or_reuse_geocoding_query(
            course_label=course_label,
            jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            cache_record_id=cache_record_id,
        )

        review = review_timezone_query_results(
            raw_results=outcome["results"],
            jurisdiction=jurisdiction,
        )

        for reviewed_result in review["reviewed_results"]:
            timezone_test_detail_rows.append(
                {
                    "candidate_course_label": course_label,
                    "candidate_jurisdiction": jurisdiction,
                    "exact_query": exact_query,
                    "result_index": reviewed_result["result_index"],
                    "provider_name": reviewed_result["provider_name"],
                    "display_name": reviewed_result["display_name"],
                    "locality": reviewed_result["locality"],
                    "region": reviewed_result["region"],
                    "latitude": reviewed_result["latitude"],
                    "longitude": reviewed_result["longitude"],
                    "iana_timezone": reviewed_result["iana_timezone"],
                    "provider_place_id": reviewed_result[
                        "provider_place_id"
                    ],
                }
            )

timezone_test_details = pd.DataFrame(timezone_test_detail_rows)

if timezone_test_details.empty:
    print("No usable country-matching results were available.")
else:
    timezone_test_details = (
        timezone_test_details
        .sort_values(
            [
                "candidate_course_label",
                "exact_query",
                "result_index",
            ]
        )
        .reset_index(drop=True)
    )

    display(timezone_test_details)

,candidate_course_label,candidate_jurisdiction,exact_query,result_index,provider_name,display_name,locality,region,latitude,longitude,iana_timezone,provider_place_id
0,Belmont Park,United States,"Belmont Park, United States",0,Belmont Park,"Belmont Park, Hempstead Avenue, Queens, Queens...",New York,New York,40.714386,-73.717913,America/New_York,way:982378259
1,Belmont Park,United States,"Belmont Park, United States",1,Belmont Park,"Belmont Park, Cross Island Parkway, Queens, Qu...",New York,New York,40.713669,-73.728301,America/New_York,node:7089672831
2,Belmont Park,United States,"Belmont Park, United States",2,Colonel Summers Park,"Colonel Summers Park, Central Eastside, Buckma...",Portland,Oregon,45.515749,-122.647340,America/Los_Angeles,way:39338024
3,Belmont Park,United States,"Belmont Park, United States",3,Belmont Park,"Belmont Park, 3146, Mission Boulevard, Mission...",San Diego,California,32.770888,-117.252047,America/Los_Angeles,way:31284612
4,Belmont Park,United States,"Belmont Park, United States",4,Belmont Park,"Belmont Park, Beat 2521, Belmont Cragin, Chica...",Chicago,Illinois,41.933921,-87.752836,America/Chicago,node:153786895
5,Caulfield,Australia,"Caulfield racecourse, Australia",0,Caulfield Racecourse,"Caulfield Racecourse, Booran Road, Caulfield E...",Melbourne,Victoria,-37.881842,145.039429,Australia/Melbourne,way:46418586
6,Caulfield,Australia,"Caulfield racecourse, Australia",1,Caulfield Racecourse,"Caulfield Racecourse, Neerim Road, Caulfield E...",Melbourne,Victoria,-37.882437,145.037575,Australia/Melbourne,relation:60837
7,Caulfield,Australia,"Caulfield, Australia",0,Caulfield,"Caulfield, Normanby Road entrance, Caulfield E...",Melbourne,Victoria,-37.877343,145.042271,Australia/Melbourne,way:397008500
8,Caulfield,Australia,"Caulfield, Australia",1,Caulfield,"Caulfield, Melbourne, Victoria, 3162, Australia",Melbourne,Victoria,-37.884064,145.026370,Australia/Melbourne,relation:2397440
9,Gulfstream Park,United States,"Gulfstream Park, United States",0,Gulfstream Park,"Gulfstream Park, Palm Beach County, Florida, 3...",NaN,Florida,26.504794,-80.051430,America/New_York,node:358725443


In [72]:
# Inspect provider object types for the five-course test.
#
# This will show whether relevant results are classified as leisure,
# stadium, sports venue, neighbourhood, park, administrative area, etc.
# No new requests should be sent because all outcomes are cached.

timezone_test_provider_rows = []

for course_row in timezone_query_plan.head(5).itertuples(index=False):
    course_label = course_row.candidate_course_label
    jurisdiction = course_row.candidate_jurisdiction
    country_code_filter = TIMEZONE_COUNTRY_CODE_FILTERS[jurisdiction]

    for exact_query in course_row.planned_queries:
        cache_record_id = build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=course_label,
            candidate_jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )

        outcome = request_or_reuse_geocoding_query(
            course_label=course_label,
            jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            cache_record_id=cache_record_id,
        )

        for result_index, result in enumerate(outcome["results"]):
            address = result.get("address") or {}

            try:
                latitude = float(result["lat"])
                longitude = float(result["lon"])
                derived_timezone = timezone_finder.timezone_at(
                    lat=latitude,
                    lng=longitude,
                )
            except (KeyError, TypeError, ValueError):
                latitude = pd.NA
                longitude = pd.NA
                derived_timezone = pd.NA

            timezone_test_provider_rows.append(
                {
                    "candidate_course_label": course_label,
                    "exact_query": exact_query,
                    "result_index": result_index,
                    "provider_name": result.get("name"),
                    "display_name": result.get("display_name"),
                    "provider_category": (
                        result.get("category")
                        or result.get("class")
                    ),
                    "provider_type": result.get("type"),
                    "addresstype": result.get("addresstype"),
                    "importance": result.get("importance"),
                    "locality": (
                        address.get("city")
                        or address.get("town")
                        or address.get("municipality")
                        or address.get("village")
                        or address.get("suburb")
                    ),
                    "region": (
                        address.get("state")
                        or address.get("province")
                        or address.get("region")
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "iana_timezone": derived_timezone,
                }
            )

timezone_test_provider_details = pd.DataFrame(
    timezone_test_provider_rows
)

display(
    timezone_test_provider_details.sort_values(
        [
            "candidate_course_label",
            "exact_query",
            "result_index",
        ]
    ).reset_index(drop=True)
)

,candidate_course_label,exact_query,result_index,provider_name,display_name,provider_category,provider_type,addresstype,importance,locality,region,latitude,longitude,iana_timezone
0,Belmont Park,"Belmont Park, United States",0,Belmont Park,"Belmont Park, Hempstead Avenue, Queens, Queens...",leisure,sports_centre,leisure,0.492433,New York,New York,40.714386,-73.717913,America/New_York
1,Belmont Park,"Belmont Park, United States",1,Belmont Park,"Belmont Park, Cross Island Parkway, Queens, Qu...",railway,station,railway,0.367845,New York,New York,40.713669,-73.728301,America/New_York
2,Belmont Park,"Belmont Park, United States",2,Colonel Summers Park,"Colonel Summers Park, Central Eastside, Buckma...",leisure,park,park,0.375267,Portland,Oregon,45.515749,-122.647340,America/Los_Angeles
3,Belmont Park,"Belmont Park, United States",3,Belmont Park,"Belmont Park, 3146, Mission Boulevard, Mission...",tourism,theme_park,tourism,0.282755,San Diego,California,32.770888,-117.252047,America/Los_Angeles
4,Belmont Park,"Belmont Park, United States",4,Belmont Park,"Belmont Park, Beat 2521, Belmont Cragin, Chica...",place,neighbourhood,neighbourhood,0.080085,Chicago,Illinois,41.933921,-87.752836,America/Chicago
5,Caulfield,"Caulfield racecourse, Australia",0,Caulfield Racecourse,"Caulfield Racecourse, Booran Road, Caulfield E...",leisure,sports_centre,leisure,0.372443,Melbourne,Victoria,-37.881842,145.039429,Australia/Melbourne
6,Caulfield,"Caulfield racecourse, Australia",1,Caulfield Racecourse,"Caulfield Racecourse, Neerim Road, Caulfield E...",leisure,track,leisure,0.000080,Melbourne,Victoria,-37.882437,145.037575,Australia/Melbourne
7,Caulfield,"Caulfield, Australia",0,Caulfield,"Caulfield, Normanby Road entrance, Caulfield E...",railway,station,railway,0.385119,Melbourne,Victoria,-37.877343,145.042271,Australia/Melbourne
8,Caulfield,"Caulfield, Australia",1,Caulfield,"Caulfield, Melbourne, Victoria, 3162, Australia",boundary,administrative,suburb,0.381741,Melbourne,Victoria,-37.884064,145.026370,Australia/Melbourne
9,Gulfstream Park,"Gulfstream Park, United States",0,Gulfstream Park,"Gulfstream Park, Palm Beach County, Florida, 3...",leisure,park,park,0.080057,NaN,Florida,26.504794,-80.051430,America/New_York


In [73]:
# Record manually reviewed timezone decisions from the five-course test.
#
# Approved:
# - Gulfstream Park -> America/New_York
# - Randwick -> Australia/Sydney
# - Caulfield -> Australia/Melbourne
# - Belmont Park -> America/New_York
#
# Saratoga remains unresolved.
#
# This cell does not write to course_locations.csv.

manual_timezone_decisions = pd.DataFrame(
    [
        {
            "candidate_course_label": "Gulfstream Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Manual review identified the Hallandale Beach, Florida "
                "sports-centre result as the relevant course; Texas results "
                "were unrelated namesakes."
            ),
        },
        {
            "candidate_course_label": "Randwick",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Sydney",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Randwick Racecourse result in Sydney, New South Wales."
            ),
        },
        {
            "candidate_course_label": "Caulfield",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Melbourne",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Two exact Caulfield Racecourse results in Melbourne, Victoria, "
                "both derived the same timezone."
            ),
        },
        {
            "candidate_course_label": "Belmont Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Manual review identified the Queens, New York sports-centre "
                "result and adjacent railway station as the relevant course; "
                "other returned parks were unrelated namesakes."
            ),
        },
    ]
)

manual_timezone_decisions = (
    manual_timezone_decisions
    .merge(
        timezone_course_queue[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "provisional_races",
            ]
        ],
        on=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(manual_timezone_decisions) == 4
assert manual_timezone_decisions["provisional_races"].notna().all()

display(manual_timezone_decisions)

,candidate_course_label,candidate_jurisdiction,resolved_iana_timezone,timezone_resolution_method,timezone_resolution_reason,provisional_races
0,Gulfstream Park,United States,America/New_York,manual_timezone_review,"Manual review identified the Hallandale Beach,...",745
1,Randwick,Australia,Australia/Sydney,manual_timezone_review,"Exact Randwick Racecourse result in Sydney, Ne...",719
2,Caulfield,Australia,Australia/Melbourne,manual_timezone_review,Two exact Caulfield Racecourse results in Melb...,574
3,Belmont Park,United States,America/New_York,manual_timezone_review,"Manual review identified the Queens, New York ...",482


In [76]:
# Run timezone-only candidate generation for the remaining 81 courses.
#
# Important:
# - the five test courses are excluded;
# - no timezone is automatically accepted;
# - no permanent reference data is modified;
# - all provider responses use the existing cache;
# - plausible sporting/racing objects are ranked for manual inspection.
#
# The output is a compact candidate table rather than a final decision table.

RACE_LIKE_CLASSIFICATIONS = {
    ("leisure", "sports_centre"),
    ("leisure", "track"),
    ("leisure", "stadium"),
    ("sport", "horse_racing"),
    ("landuse", "recreation_ground"),
}

EXCLUDED_TEST_COURSES = {
    ("Gulfstream Park", "United States"),
    ("Randwick", "Australia"),
    ("Caulfield", "Australia"),
    ("Saratoga", "United States"),
    ("Belmont Park", "United States"),
}


def normalise_candidate_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def candidate_name_similarity(course_label, provider_name):
    course_tokens = set(
        normalise_candidate_text(course_label).split()
    )
    provider_tokens = set(
        normalise_candidate_text(provider_name).split()
    )

    if not course_tokens or not provider_tokens:
        return 0.0

    return len(course_tokens & provider_tokens) / len(course_tokens)


remaining_timezone_query_plan = timezone_query_plan.loc[
    ~timezone_query_plan.apply(
        lambda row: (
            row["candidate_course_label"],
            row["candidate_jurisdiction"],
        ) in EXCLUDED_TEST_COURSES,
        axis=1,
    )
].copy()

assert len(remaining_timezone_query_plan) == 81

timezone_candidate_rows = []
timezone_run_rows = []

for course_row in remaining_timezone_query_plan.itertuples(index=False):
    course_label = course_row.candidate_course_label
    jurisdiction = course_row.candidate_jurisdiction
    country_code_filter = TIMEZONE_COUNTRY_CODE_FILTERS[
        jurisdiction
    ]

    course_requests_sent = 0
    course_requests_reused = 0

    for query_sequence, exact_query in enumerate(
        course_row.planned_queries,
        start=1,
    ):
        cache_record_id = build_geocoding_cache_record_id(
            provider=NOMINATIM_PROVIDER,
            candidate_course_label=course_label,
            candidate_jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            result_limit=NOMINATIM_RESULT_LIMIT,
        )

        outcome = request_or_reuse_geocoding_query(
            course_label=course_label,
            jurisdiction=jurisdiction,
            exact_query=exact_query,
            country_code_filter=country_code_filter,
            cache_record_id=cache_record_id,
        )

        if outcome["request_action"] == "sent_and_cached":
            course_requests_sent += 1
        else:
            course_requests_reused += 1

        timezone_run_rows.append(
            {
                "candidate_course_label": course_label,
                "candidate_jurisdiction": jurisdiction,
                "provisional_races": course_row.provisional_races,
                "query_sequence": query_sequence,
                "exact_query": exact_query,
                "request_action": outcome["request_action"],
                "request_status": outcome["request_status"],
                "raw_result_count": len(outcome["results"]),
            }
        )

        for result_index, result in enumerate(outcome["results"]):
            address = result.get("address") or {}
            country_code = str(
                address.get("country_code") or ""
            ).lower()

            if country_code != country_code_filter:
                continue

            try:
                latitude = float(result["lat"])
                longitude = float(result["lon"])
            except (KeyError, TypeError, ValueError):
                continue

            derived_timezone = timezone_finder.timezone_at(
                lat=latitude,
                lng=longitude,
            )

            if not derived_timezone:
                continue

            provider_category = (
                result.get("category")
                or result.get("class")
            )
            provider_type = result.get("type")
            provider_name = result.get("name")

            name_similarity = candidate_name_similarity(
                course_label,
                provider_name,
            )

            race_like_classification = (
                provider_category,
                provider_type,
            ) in RACE_LIKE_CLASSIFICATIONS

            query_is_racing_specific = any(
                term in exact_query.lower()
                for term in [
                    "racecourse",
                    "racetrack",
                    "hipodromo",
                ]
            )

            candidate_score = (
                4 * int(race_like_classification)
                + 3 * name_similarity
                + 1 * int(query_is_racing_specific)
                + 0.25 * float(result.get("importance") or 0)
            )

            timezone_candidate_rows.append(
                {
                    "candidate_course_label": course_label,
                    "candidate_jurisdiction": jurisdiction,
                    "provisional_races": course_row.provisional_races,
                    "query_sequence": query_sequence,
                    "exact_query": exact_query,
                    "result_index": result_index,
                    "provider_name": provider_name,
                    "display_name": result.get("display_name"),
                    "provider_category": provider_category,
                    "provider_type": provider_type,
                    "addresstype": result.get("addresstype"),
                    "importance": result.get("importance"),
                    "locality": (
                        address.get("city")
                        or address.get("town")
                        or address.get("municipality")
                        or address.get("village")
                        or address.get("suburb")
                    ),
                    "region": (
                        address.get("state")
                        or address.get("province")
                        or address.get("region")
                    ),
                    "latitude": latitude,
                    "longitude": longitude,
                    "iana_timezone": derived_timezone,
                    "provider_place_id": (
                        f"{result.get('osm_type')}:{result.get('osm_id')}"
                        if result.get("osm_type")
                        and result.get("osm_id")
                        else pd.NA
                    ),
                    "name_similarity": round(
                        name_similarity,
                        3,
                    ),
                    "race_like_classification": (
                        race_like_classification
                    ),
                    "candidate_score": round(
                        candidate_score,
                        3,
                    ),
                }
            )

timezone_candidate_results = pd.DataFrame(
    timezone_candidate_rows
)

timezone_request_run = pd.DataFrame(
    timezone_run_rows
)

if timezone_candidate_results.empty:
    timezone_manual_review_candidates = pd.DataFrame()
else:
    timezone_candidate_results = (
        timezone_candidate_results
        .sort_values(
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "candidate_score",
                "query_sequence",
                "result_index",
            ],
            ascending=[
                True,
                True,
                False,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    # Retain plausible candidates only.
    timezone_manual_review_candidates = (
        timezone_candidate_results.loc[
            (
                timezone_candidate_results[
                    "race_like_classification"
                ]
            )
            & (
                timezone_candidate_results[
                    "name_similarity"
                ] >= 0.5
            )
        ]
        .copy()
    )

    timezone_manual_review_candidates[
        "candidate_rank"
    ] = (
        timezone_manual_review_candidates
        .groupby(
            [
                "candidate_course_label",
                "candidate_jurisdiction",
            ]
        )
        .cumcount()
        .add(1)
    )

    timezone_manual_review_candidates = (
        timezone_manual_review_candidates.loc[
            timezone_manual_review_candidates[
                "candidate_rank"
            ] <= 3
        ]
        .sort_values(
            [
                "provisional_races",
                "candidate_course_label",
                "candidate_rank",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

candidate_generation_summary = pd.DataFrame(
    {
        "measure": [
            "Courses processed",
            "Queries attempted",
            "New requests",
            "Cached requests reused",
            "Courses with at least one plausible candidate",
            "Courses with no plausible candidate",
            "Plausible candidate rows retained",
        ],
        "value": [
            len(remaining_timezone_query_plan),
            len(timezone_request_run),
            int(
                timezone_request_run[
                    "request_action"
                ].eq("sent_and_cached").sum()
            ),
            int(
            (
                ~timezone_request_run[
                    "request_action"
                ].eq("sent_and_cached")
            ).sum()
            ),
            int(
                timezone_manual_review_candidates[
                    [
                        "candidate_course_label",
                        "candidate_jurisdiction",
                    ]
                ]
                .drop_duplicates()
                .shape[0]
            )
            if not timezone_manual_review_candidates.empty
            else 0,
            int(
                len(remaining_timezone_query_plan)
                - (
                    timezone_manual_review_candidates[
                        [
                            "candidate_course_label",
                            "candidate_jurisdiction",
                        ]
                    ]
                    .drop_duplicates()
                    .shape[0]
                    if not timezone_manual_review_candidates.empty
                    else 0
                )
            ),
            len(timezone_manual_review_candidates),
        ],
    }
)

display(candidate_generation_summary)

display(
    timezone_manual_review_candidates[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
            "candidate_rank",
            "provider_name",
            "display_name",
            "provider_category",
            "provider_type",
            "locality",
            "region",
            "iana_timezone",
            "candidate_score",
        ]
    ]
)


,measure,value
0,Courses processed,81
1,Queries attempted,214
2,New requests,0
3,Cached requests reused,214
4,Courses with at least one plausible candidate,21
5,Courses with no plausible candidate,60
6,Plausible candidate rows retained,28


,candidate_course_label,candidate_jurisdiction,provisional_races,candidate_rank,provider_name,display_name,provider_category,provider_type,locality,region,iana_timezone,candidate_score
0,Churchill Downs,United States,474,1,Churchill Downs,"Churchill Downs, Kentucky Derby Drive, Louisvi...",leisure,stadium,Louisville,Kentucky,America/Kentucky/Louisville,7.125
1,Woodbine,Canada,434,1,Woodbine Racetrack,"Woodbine Racetrack, Etobicoke, Toronto, Golden...",landuse,recreation_ground,Toronto,Ontario,America/Toronto,8.020
2,Morphettville,Australia,205,1,Morphettville Racecourse,"Morphettville Racecourse, Park Terrace, Morphe...",leisure,sports_centre,Adelaide,South Australia,Australia/Adelaide,8.084
3,Morphettville,Australia,205,2,Morphettville Racecourse,"Morphettville Racecourse, Mike Turtur Bikeway,...",leisure,track,Adelaide,South Australia,Australia/Adelaide,8.000
4,Arlington Park,United States,70,1,Arlington Park Racecourse,"Arlington Park Racecourse, 5th Avenue, Babcock...",leisure,sports_centre,East Moline,Illinois,America/Chicago,8.000
5,Arlington Park,United States,70,2,Arlington Park Racecourse,"Arlington Park Racecourse, Morton Drive, Babco...",leisure,track,East Moline,Illinois,America/Chicago,8.000
6,Arlington Park,United States,70,3,Arlington Park,"Arlington Park, 2200, West Euclid Avenue, Arli...",leisure,sports_centre,Arlington Heights,Illinois,America/Chicago,7.100
7,Ellis Park,United States,33,1,Ellis Park Race Course,"Ellis Park Race Course, 3300, Ellis Park Entra...",leisure,sports_centre,NaN,Kentucky,America/Chicago,7.069
8,Turfway Park,United States,24,1,Turfway Park,"Turfway Park, Turfway Access, Florence, Boone ...",leisure,stadium,Florence,Kentucky,America/New_York,7.079
9,Launceston,Australia,21,1,Launceston Tennis Centre,"Launceston Tennis Centre, Racecourse Crescent,...",leisure,sports_centre,Launceston,Tasmania,Australia/Hobart,8.000


In [77]:
# Add the 18 manually approved candidates from the full candidate run.
#
# This extends the existing manual_timezone_decisions table.
# Duplicated provider objects were reviewed as supporting evidence, but each
# course receives only one timezone decision.
#
# No permanent files are written in this cell.

additional_manual_timezone_decisions = pd.DataFrame(
    [
        {
            "candidate_course_label": "Churchill Downs",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Kentucky/Louisville",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Churchill Downs stadium result in Louisville, Kentucky."
            ),
        },
        {
            "candidate_course_label": "Woodbine",
            "candidate_jurisdiction": "Canada",
            "resolved_iana_timezone": "America/Toronto",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Woodbine Racetrack result in Toronto, Ontario."
            ),
        },
        {
            "candidate_course_label": "Morphettville",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Adelaide",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Morphettville Racecourse sporting and track objects "
                "in Adelaide, South Australia agreed on timezone."
            ),
        },
        {
            "candidate_course_label": "Arlington Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Chicago",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Manual review selected the Arlington Heights, Illinois "
                "sports-centre result. East Moline results were false positives."
            ),
        },
        {
            "candidate_course_label": "Ellis Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Chicago",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Ellis Park Race Course result in western Kentucky."
            ),
        },
        {
            "candidate_course_label": "Turfway Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Turfway Park stadium result in Florence, Kentucky."
            ),
        },
        {
            "candidate_course_label": "Bendigo",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Melbourne",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Bendigo Racecourse sporting and track objects in "
                "Victoria agreed on timezone."
            ),
        },
        {
            "candidate_course_label": "Colonial Downs",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Colonial Downs stadium result in Virginia."
            ),
        },
        {
            "candidate_course_label": "Lone Star Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Chicago",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Lone Star Park track result in Grand Prairie, Texas."
            ),
        },
        {
            "candidate_course_label": "Sunland Park",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Denver",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Sunland Park Racetrack result in New Mexico."
            ),
        },
        {
            "candidate_course_label": "Emerald Downs",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Los_Angeles",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Two exact Emerald Downs racing objects in Auburn, Washington "
                "agreed on timezone."
            ),
        },
        {
            "candidate_course_label": "Finger Lakes",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Finger Lakes Casino and Racetrack sporting and track "
                "objects in New York agreed on timezone."
            ),
        },
        {
            "candidate_course_label": "Seymour",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Melbourne",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Seymour Racecourse result in Victoria."
            ),
        },
        {
            "candidate_course_label": "Moe",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Melbourne",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Moe Racing Club result in Victoria."
            ),
        },
        {
            "candidate_course_label": "Armidale",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Sydney",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Armidale Racecourse result in New South Wales."
            ),
        },
        {
            "candidate_course_label": "Meadowlands",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/New_York",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Meadowlands Racing and Entertainment result in "
                "East Rutherford, New Jersey."
            ),
        },
        {
            "candidate_course_label": "Turf Paradise",
            "candidate_jurisdiction": "United States",
            "resolved_iana_timezone": "America/Phoenix",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Turf Paradise sports-centre result in Phoenix, Arizona."
            ),
        },
        {
            "candidate_course_label": "Wangaratta",
            "candidate_jurisdiction": "Australia",
            "resolved_iana_timezone": "Australia/Melbourne",
            "timezone_resolution_method": "manual_timezone_review",
            "timezone_resolution_reason": (
                "Exact Wangaratta Race Course result in Victoria."
            ),
        },
    ]
)

additional_manual_timezone_decisions = (
    additional_manual_timezone_decisions
    .merge(
        timezone_course_queue[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "provisional_races",
            ]
        ],
        on=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(additional_manual_timezone_decisions) == 18
assert additional_manual_timezone_decisions[
    "provisional_races"
].notna().all()

manual_timezone_decisions = pd.concat(
    [
        manual_timezone_decisions,
        additional_manual_timezone_decisions,
    ],
    ignore_index=True,
)

manual_timezone_decisions = (
    manual_timezone_decisions
    .drop_duplicates(
        subset=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        keep="last",
    )
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

assert len(manual_timezone_decisions) == 22

resolved_manual_keys = set(
    zip(
        manual_timezone_decisions["candidate_course_label"],
        manual_timezone_decisions["candidate_jurisdiction"],
    )
)

remaining_timezone_courses = (
    timezone_course_queue.loc[
        ~timezone_course_queue.apply(
            lambda row: (
                row["candidate_course_label"],
                row["candidate_jurisdiction"],
            ) in resolved_manual_keys,
            axis=1,
        )
    ]
    .copy()
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

remaining_timezone_summary = pd.DataFrame(
    {
        "measure": [
            "Manually resolved multi-timezone courses",
            "Provisional races covered by manual course review",
            "Multi-timezone courses still unresolved",
            "Provisional races still unresolved in multi-timezone jurisdictions",
        ],
        "value": [
            len(manual_timezone_decisions),
            int(
                manual_timezone_decisions[
                    "provisional_races"
                ].sum()
            ),
            len(remaining_timezone_courses),
            int(
                remaining_timezone_courses[
                    "provisional_races"
                ].sum()
            ),
        ],
    }
)

display(remaining_timezone_summary)
display(
    remaining_timezone_courses[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
        ]
    ].head(40)
)

,measure,value
0,Manually resolved multi-timezone courses,22
1,Provisional races covered by manual course review,3836
2,Multi-timezone courses still unresolved,64
3,Provisional races still unresolved in multi-ti...,3371


,candidate_course_label,candidate_jurisdiction,provisional_races
0,Saratoga,United States,528
1,Rosehill,Australia,472
2,Keeneland,United States,409
3,Will Rogers Downs,United States,178
4,Eagle Farm,Australia,166
5,Oaklawn Park,United States,157
6,Gavea,Brazil,136
7,Fair Grounds,United States,112
8,Tampa Bay Downs,United States,107
9,Pimlico,United States,100


In [78]:
# Generate a compact unresolved-course list for manual lookup.
#
# This prints one course per line in a format that can be pasted directly
# back into ChatGPT. No files are written.

manual_lookup_list = (
    remaining_timezone_courses[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "provisional_races",
        ]
    ]
    .sort_values(
        [
            "provisional_races",
            "candidate_jurisdiction",
            "candidate_course_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "\n".join(
        f"{row.candidate_course_label} | "
        f"{row.candidate_jurisdiction} | "
        f"{int(row.provisional_races)} races"
        for row in manual_lookup_list.itertuples(index=False)
    )
)

Saratoga | United States | 528 races
Rosehill | Australia | 472 races
Keeneland | United States | 409 races
Will Rogers Downs | United States | 178 races
Eagle Farm | Australia | 166 races
Oaklawn Park | United States | 157 races
Gavea | Brazil | 136 races
Fair Grounds | United States | 112 races
Tampa Bay Downs | United States | 107 races
Pimlico | United States | 100 races
Kentucky Downs | United States | 86 races
Parx | United States | 85 races
Newcastle | Australia | 77 races
Laurel Park | United States | 71 races
Fonner Park | United States | 68 races
Golden Gate Fields | United States | 54 races
Delaware Park | United States | 49 races
Los Alamitos | United States | 48 races
Indiana Grand | United States | 40 races
Kembla Grange | Australia | 38 races
Hawkesbury | Australia | 33 races
Remington Park | United States | 25 races
Charles town | United States | 24 races
Far Hills | United States | 23 races
Hobart | Australia | 22 races
Prairie Meadows | United States | 22 races
Launce

In [81]:
# Load and validate the 64 manually researched course resolutions.
#
# This cell reads only. It does not yet update course_locations.csv.

from pathlib import Path

MANUAL_TIMEZONE_RESOLUTION_PATH = Path(
    "/home/rob/Documents/inside-rails-horse-racing/"
    "data/reference/course_location_manual_timezone_resolution.csv"
)

manual_course_resolutions = pd.read_csv(
    MANUAL_TIMEZONE_RESOLUTION_PATH,
    dtype={
        "candidate_course_label": "string",
        "candidate_jurisdiction": "string",
        "official_venue_name": "string",
        "street_address": "string",
        "locality": "string",
        "region": "string",
        "postal_code": "string",
        "country": "string",
        "latitude": "Float64",
        "longitude": "Float64",
        "iana_timezone": "string",
        "venue_status": "string",
        "former_or_alternative_name": "string",
        "manual_review_confidence": "string",
        "manual_review_note": "string",
        "source_urls": "string",
    },
)

manual_course_resolutions["provisional_races"] = pd.to_numeric(
    manual_course_resolutions["provisional_races"],
    errors="raise",
).astype("int64")

assert len(manual_course_resolutions) == 64

assert not manual_course_resolutions.duplicated(
    subset=[
        "candidate_course_label",
        "candidate_jurisdiction",
    ]
).any()

assert manual_course_resolutions[
    [
        "candidate_course_label",
        "candidate_jurisdiction",
        "official_venue_name",
        "locality",
        "region",
        "country",
        "iana_timezone",
        "manual_review_confidence",
    ]
].notna().all().all()

for timezone_name in manual_course_resolutions["iana_timezone"]:
    ZoneInfo(timezone_name)

expected_unresolved_keys = set(
    zip(
        remaining_timezone_courses["candidate_course_label"],
        remaining_timezone_courses["candidate_jurisdiction"],
    )
)

manual_resolution_keys = set(
    zip(
        manual_course_resolutions["candidate_course_label"],
        manual_course_resolutions["candidate_jurisdiction"],
    )
)

missing_manual_resolutions = (
    expected_unresolved_keys
    - manual_resolution_keys
)

unexpected_manual_resolutions = (
    manual_resolution_keys
    - expected_unresolved_keys
)

validation_summary = pd.DataFrame(
    {
        "measure": [
            "Manual resolution rows",
            "Expected unresolved courses",
            "Missing expected courses",
            "Unexpected courses",
            "Rows with valid IANA timezone",
            "Low-confidence rows",
        ],
        "value": [
            len(manual_course_resolutions),
            len(expected_unresolved_keys),
            len(missing_manual_resolutions),
            len(unexpected_manual_resolutions),
            len(manual_course_resolutions),
            int(
                manual_course_resolutions[
                    "manual_review_confidence"
                ]
                .str.lower()
                .eq("low")
                .sum()
            ),
        ],
    }
)

display(validation_summary)

if missing_manual_resolutions:
    print("Missing expected courses:")
    display(
        pd.DataFrame(
            sorted(missing_manual_resolutions),
            columns=[
                "candidate_course_label",
                "candidate_jurisdiction",
            ],
        )
    )

if unexpected_manual_resolutions:
    print("Unexpected courses:")
    display(
        pd.DataFrame(
            sorted(unexpected_manual_resolutions),
            columns=[
                "candidate_course_label",
                "candidate_jurisdiction",
            ],
        )
    )

display(
    manual_course_resolutions[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "official_venue_name",
            "locality",
            "region",
            "iana_timezone",
            "venue_status",
            "manual_review_confidence",
        ]
    ].head(15)
)

,measure,value
0,Manual resolution rows,64
1,Expected unresolved courses,64
2,Missing expected courses,0
3,Unexpected courses,0
4,Rows with valid IANA timezone,64
5,Low-confidence rows,1


,candidate_course_label,candidate_jurisdiction,official_venue_name,locality,region,iana_timezone,venue_status,manual_review_confidence
0,Saratoga,United States,Saratoga Race Course,Saratoga Springs,New York,America/New_York,active,high
1,Rosehill,Australia,Rosehill Gardens Racecourse,Rosehill,New South Wales,Australia/Sydney,active,high
2,Keeneland,United States,Keeneland Race Course,Lexington,Kentucky,America/New_York,active,high
3,Will Rogers Downs,United States,Will Rogers Downs,Claremore,Oklahoma,America/Chicago,active,high
4,Eagle Farm,Australia,Eagle Farm Racecourse,Ascot,Queensland,Australia/Brisbane,active,high
5,Oaklawn Park,United States,Oaklawn Racing Casino Resort,Hot Springs,Arkansas,America/Chicago,active,high
6,Gavea,Brazil,Hipódromo da Gávea,Rio de Janeiro,Rio de Janeiro,America/Sao_Paulo,active,high
7,Fair Grounds,United States,Fair Grounds Race Course & Slots,New Orleans,Louisiana,America/Chicago,active,high
8,Tampa Bay Downs,United States,Tampa Bay Downs,Tampa,Florida,America/New_York,active,high
9,Pimlico,United States,Pimlico Race Course,Baltimore,Maryland,America/New_York,active,high


In [82]:
# Build a complete timezone assignment preview for all course identities.
#
# This still does not write to course_locations.csv.

manual_csv_timezone_assignments = (
    manual_course_resolutions[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "iana_timezone",
            "manual_review_confidence",
            "manual_review_note",
        ]
    ]
    .rename(
        columns={
            "iana_timezone": "resolved_iana_timezone",
        }
    )
    .assign(
        timezone_resolution_method="manual_reference_csv",
        timezone_resolution_reason=lambda df: (
            df["manual_review_note"]
        ),
    )
)

manual_csv_timezone_assignments = (
    manual_csv_timezone_assignments[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "resolved_iana_timezone",
            "timezone_resolution_method",
            "timezone_resolution_reason",
            "manual_review_confidence",
        ]
    ]
)

all_manual_timezone_assignments = pd.concat(
    [
        manual_timezone_decisions.assign(
            manual_review_confidence="high"
        )[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "resolved_iana_timezone",
                "timezone_resolution_method",
                "timezone_resolution_reason",
                "manual_review_confidence",
            ]
        ],
        manual_csv_timezone_assignments,
    ],
    ignore_index=True,
)

assert len(all_manual_timezone_assignments) == 86

assert not all_manual_timezone_assignments.duplicated(
    subset=[
        "candidate_course_label",
        "candidate_jurisdiction",
    ]
).any()

complete_timezone_preview = (
    course_locations
    .merge(
        jurisdiction_timezone_policy_df[
            [
                "candidate_jurisdiction",
                "timezone_policy",
                "jurisdiction_default_iana_timezone",
            ]
        ],
        on="candidate_jurisdiction",
        how="left",
        validate="many_to_one",
    )
    .merge(
        all_manual_timezone_assignments,
        on=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        how="left",
        validate="one_to_one",
    )
)

existing_timezone_column = next(
    column
    for column in [
        "iana_timezone",
        "resolved_iana_timezone_existing",
        "timezone",
    ]
    if column in complete_timezone_preview.columns
)

complete_timezone_preview[
    "final_iana_timezone"
] = complete_timezone_preview[
    existing_timezone_column
]

manual_mask = complete_timezone_preview[
    "resolved_iana_timezone"
].notna()

complete_timezone_preview.loc[
    manual_mask,
    "final_iana_timezone",
] = complete_timezone_preview.loc[
    manual_mask,
    "resolved_iana_timezone",
]

default_mask = (
    complete_timezone_preview[
        "final_iana_timezone"
    ].isna()
    & complete_timezone_preview[
        "timezone_policy"
    ].eq("jurisdiction_default")
)

complete_timezone_preview.loc[
    default_mask,
    "final_iana_timezone",
] = complete_timezone_preview.loc[
    default_mask,
    "jurisdiction_default_iana_timezone",
]

complete_timezone_preview[
    "final_timezone_resolution_method"
] = pd.NA

complete_timezone_preview.loc[
    complete_timezone_preview[
        existing_timezone_column
    ].notna(),
    "final_timezone_resolution_method",
] = "existing_resolved_location"

complete_timezone_preview.loc[
    default_mask,
    "final_timezone_resolution_method",
] = "jurisdiction_default"

complete_timezone_preview.loc[
    manual_mask,
    "final_timezone_resolution_method",
] = complete_timezone_preview.loc[
    manual_mask,
    "timezone_resolution_method",
]

for timezone_name in complete_timezone_preview[
    "final_iana_timezone"
].dropna():
    ZoneInfo(timezone_name)

complete_timezone_summary = pd.DataFrame(
    {
        "measure": [
            "Course identities",
            "Final timezone assignments",
            "Still unresolved",
            "Existing resolved-location assignments",
            "Jurisdiction-default assignments",
            "Manual course-level assignments",
        ],
        "value": [
            len(complete_timezone_preview),
            int(
                complete_timezone_preview[
                    "final_iana_timezone"
                ].notna().sum()
            ),
            int(
                complete_timezone_preview[
                    "final_iana_timezone"
                ].isna().sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ].eq("existing_resolved_location").sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ].eq("jurisdiction_default").sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ].isin(
                    [
                        "manual_timezone_review",
                        "manual_reference_csv",
                    ]
                ).sum()
            ),
        ],
    }
)

display(complete_timezone_summary)

display(
    complete_timezone_preview.loc[
        complete_timezone_preview[
            "final_iana_timezone"
        ].isna(),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
    ]
)

KeyError: "['timezone_policy', 'jurisdiction_default_iana_timezone'] not in index"

In [83]:
print(jurisdiction_timezone_policy_df.columns.tolist())

display(jurisdiction_timezone_policy_df.head(10))

['candidate_jurisdiction', 'policy', 'iana_timezone', 'reason']


,candidate_jurisdiction,policy,iana_timezone,reason
0,Great Britain,jurisdiction_default,Europe/London,British racecourses use UK civil time.
1,Ireland,jurisdiction_default,Europe/Dublin,Irish racecourses use Irish civil time.
2,Hong Kong,jurisdiction_default,Asia/Hong_Kong,Both represented Hong Kong racecourses use Hon...
3,France,jurisdiction_default,Europe/Paris,The dataset represents mainland French racing.
4,United Arab Emirates,jurisdiction_default,Asia/Dubai,Represented UAE racecourses use Gulf Standard ...
5,Japan,jurisdiction_default,Asia/Tokyo,Japan uses one civil timezone.
6,Germany,jurisdiction_default,Europe/Berlin,German racecourses use German civil time.
7,Jersey,jurisdiction_default,Europe/Jersey,The represented jurisdiction has one applicabl...
8,Italy,jurisdiction_default,Europe/Rome,Italian racecourses use Italian civil time.
9,Chile,jurisdiction_default,America/Santiago,The represented racecourses are mainland Chile...


In [84]:
# Build a complete timezone assignment preview for all course identities.
#
# Sources:
# - existing timezone on course_locations;
# - jurisdiction defaults;
# - 22 manually reviewed course assignments;
# - 64 manual CSV assignments.
#
# This cell still does not write to course_locations.csv.

manual_csv_timezone_assignments = (
    manual_course_resolutions[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "iana_timezone",
            "manual_review_confidence",
            "manual_review_note",
        ]
    ]
    .rename(
        columns={
            "iana_timezone": "resolved_iana_timezone",
        }
    )
    .assign(
        timezone_resolution_method="manual_reference_csv",
        timezone_resolution_reason=lambda df: (
            df["manual_review_note"]
        ),
    )
    [
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "resolved_iana_timezone",
            "timezone_resolution_method",
            "timezone_resolution_reason",
            "manual_review_confidence",
        ]
    ]
)

all_manual_timezone_assignments = pd.concat(
    [
        manual_timezone_decisions.assign(
            manual_review_confidence="high"
        )[
            [
                "candidate_course_label",
                "candidate_jurisdiction",
                "resolved_iana_timezone",
                "timezone_resolution_method",
                "timezone_resolution_reason",
                "manual_review_confidence",
            ]
        ],
        manual_csv_timezone_assignments,
    ],
    ignore_index=True,
)

assert len(all_manual_timezone_assignments) == 86

assert not all_manual_timezone_assignments.duplicated(
    subset=[
        "candidate_course_label",
        "candidate_jurisdiction",
    ]
).any()

timezone_policy_for_merge = (
    jurisdiction_timezone_policy_df
    .rename(
        columns={
            "policy": "timezone_policy",
            "iana_timezone": (
                "jurisdiction_default_iana_timezone"
            ),
            "reason": "jurisdiction_timezone_reason",
        }
    )
)

complete_timezone_preview = (
    course_locations
    .merge(
        timezone_policy_for_merge[
            [
                "candidate_jurisdiction",
                "timezone_policy",
                "jurisdiction_default_iana_timezone",
                "jurisdiction_timezone_reason",
            ]
        ],
        on="candidate_jurisdiction",
        how="left",
        validate="many_to_one",
    )
    .merge(
        all_manual_timezone_assignments,
        on=[
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
        how="left",
        validate="one_to_one",
    )
)

existing_timezone_column = next(
    column
    for column in [
        "iana_timezone",
        "resolved_iana_timezone_existing",
        "timezone",
    ]
    if column in complete_timezone_preview.columns
)

complete_timezone_preview[
    "final_iana_timezone"
] = complete_timezone_preview[
    existing_timezone_column
].astype("string")

manual_mask = complete_timezone_preview[
    "resolved_iana_timezone"
].notna()

complete_timezone_preview.loc[
    manual_mask,
    "final_iana_timezone",
] = complete_timezone_preview.loc[
    manual_mask,
    "resolved_iana_timezone",
]

default_mask = (
    complete_timezone_preview[
        "final_iana_timezone"
    ].isna()
    & complete_timezone_preview[
        "timezone_policy"
    ].eq("jurisdiction_default")
)

complete_timezone_preview.loc[
    default_mask,
    "final_iana_timezone",
] = complete_timezone_preview.loc[
    default_mask,
    "jurisdiction_default_iana_timezone",
]

complete_timezone_preview[
    "final_timezone_resolution_method"
] = pd.Series(
    pd.NA,
    index=complete_timezone_preview.index,
    dtype="string",
)

existing_mask = complete_timezone_preview[
    existing_timezone_column
].notna()

complete_timezone_preview.loc[
    existing_mask,
    "final_timezone_resolution_method",
] = "existing_resolved_location"

complete_timezone_preview.loc[
    default_mask,
    "final_timezone_resolution_method",
] = "jurisdiction_default"

complete_timezone_preview.loc[
    manual_mask,
    "final_timezone_resolution_method",
] = complete_timezone_preview.loc[
    manual_mask,
    "timezone_resolution_method",
]

complete_timezone_preview[
    "final_timezone_resolution_reason"
] = pd.Series(
    pd.NA,
    index=complete_timezone_preview.index,
    dtype="string",
)

complete_timezone_preview.loc[
    existing_mask,
    "final_timezone_resolution_reason",
] = "Timezone already derived from a resolved course location."

complete_timezone_preview.loc[
    default_mask,
    "final_timezone_resolution_reason",
] = complete_timezone_preview.loc[
    default_mask,
    "jurisdiction_timezone_reason",
]

complete_timezone_preview.loc[
    manual_mask,
    "final_timezone_resolution_reason",
] = complete_timezone_preview.loc[
    manual_mask,
    "timezone_resolution_reason",
]

for timezone_name in complete_timezone_preview[
    "final_iana_timezone"
].dropna():
    ZoneInfo(timezone_name)

complete_timezone_summary = pd.DataFrame(
    {
        "measure": [
            "Course identities",
            "Final timezone assignments",
            "Still unresolved",
            "Existing resolved-location assignments",
            "Jurisdiction-default assignments",
            "Manual course-level assignments",
        ],
        "value": [
            len(complete_timezone_preview),
            int(
                complete_timezone_preview[
                    "final_iana_timezone"
                ].notna().sum()
            ),
            int(
                complete_timezone_preview[
                    "final_iana_timezone"
                ].isna().sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ]
                .eq("existing_resolved_location")
                .sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ]
                .eq("jurisdiction_default")
                .sum()
            ),
            int(
                complete_timezone_preview[
                    "final_timezone_resolution_method"
                ]
                .isin(
                    [
                        "manual_timezone_review",
                        "manual_reference_csv",
                    ]
                )
                .sum()
            ),
        ],
    }
)

display(complete_timezone_summary)

display(
    complete_timezone_preview.loc[
        complete_timezone_preview[
            "final_iana_timezone"
        ].isna(),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
        ],
    ]
)

,measure,value
0,Course identities,394
1,Final timezone assignments,394
2,Still unresolved,0
3,Existing resolved-location assignments,126
4,Jurisdiction-default assignments,182
5,Manual course-level assignments,86


,candidate_course_label,candidate_jurisdiction


### Complete timezone coverage achieved

All 394 permanent course identities now have a valid IANA timezone assignment.

Assignments were derived from three evidence paths:

- 126 courses retained a timezone derived from an already resolved course location;
- 182 courses used a documented jurisdiction-level timezone policy where one civil timezone safely applies;
- 86 courses in multi-timezone jurisdictions were resolved individually through manual course-level review.

No course identities remain unresolved. The next step is to persist the final timezone, resolution method, and resolution reason into the permanent course-location reference.

In [86]:
# Persist complete timezone coverage into course_locations.csv.
#
# This updates only the final timezone and its resolution metadata.
# Existing course identity and location fields are preserved.
from pathlib import Path

COURSE_LOCATION_REFERENCE_PATH = Path(
    "/home/rob/Documents/inside-rails-horse-racing/"
    "data/reference/course_locations.csv"
)
course_locations_updated = complete_timezone_preview.copy()

course_locations_updated["iana_timezone"] = (
    course_locations_updated["final_iana_timezone"]
)

course_locations_updated["timezone_resolution_method"] = (
    course_locations_updated[
        "final_timezone_resolution_method"
    ]
)

course_locations_updated["timezone_resolution_reason"] = (
    course_locations_updated[
        "final_timezone_resolution_reason"
    ]
)

# Remove preview-only merge columns before writing.
preview_only_columns = [
    "timezone_policy",
    "jurisdiction_default_iana_timezone",
    "jurisdiction_timezone_reason",
    "resolved_iana_timezone",
    "manual_review_confidence",
    "final_iana_timezone",
    "final_timezone_resolution_method",
    "final_timezone_resolution_reason",
]

course_locations_updated = course_locations_updated.drop(
    columns=[
        column
        for column in preview_only_columns
        if column in course_locations_updated.columns
    ]
)

assert len(course_locations_updated) == 394
assert course_locations_updated["iana_timezone"].notna().all()

for timezone_name in course_locations_updated["iana_timezone"]:
    ZoneInfo(timezone_name)

course_locations_updated.to_csv(
    COURSE_LOCATION_REFERENCE_PATH,
    index=False,
)

# Reload from disk to confirm the persisted file is valid.
course_locations_reloaded = pd.read_csv(
    COURSE_LOCATION_REFERENCE_PATH,
    dtype={
        "candidate_course_label": "string",
        "candidate_jurisdiction": "string",
        "iana_timezone": "string",
        "timezone_resolution_method": "string",
        "timezone_resolution_reason": "string",
    },
)

assert len(course_locations_reloaded) == 394
assert course_locations_reloaded["iana_timezone"].notna().all()

persisted_timezone_summary = pd.DataFrame(
    {
        "measure": [
            "Persisted course identities",
            "Persisted timezone assignments",
            "Persisted unresolved timezones",
            "Distinct IANA timezones",
        ],
        "value": [
            len(course_locations_reloaded),
            int(
                course_locations_reloaded[
                    "iana_timezone"
                ].notna().sum()
            ),
            int(
                course_locations_reloaded[
                    "iana_timezone"
                ].isna().sum()
            ),
            int(
                course_locations_reloaded[
                    "iana_timezone"
                ].nunique()
            ),
        ],
    }
)

display(persisted_timezone_summary)

display(
    course_locations_reloaded[
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "iana_timezone",
            "timezone_resolution_method",
        ]
    ].head(15)
)

,measure,value
0,Persisted course identities,394
1,Persisted timezone assignments,394
2,Persisted unresolved timezones,0
3,Distinct IANA timezones,51


,candidate_course_label,candidate_jurisdiction,iana_timezone,timezone_resolution_method
0,La Plata,Argentina,America/Argentina/Buenos_Aires,existing_resolved_location
1,Palermo,Argentina,America/Argentina/Buenos_Aires,existing_resolved_location
2,San Isidro,Argentina,America/Argentina/Buenos_Aires,jurisdiction_default
3,Albury,Australia,Australia/Sydney,manual_reference_csv
4,Alice springs,Australia,Australia/Darwin,manual_reference_csv
5,Armidale,Australia,Australia/Sydney,manual_timezone_review
6,Ascot,Australia,Australia/Perth,existing_resolved_location
7,Balaklava,Australia,Australia/Adelaide,existing_resolved_location
8,Ballarat,Australia,Australia/Melbourne,manual_reference_csv
9,Belmont Park (Perth),Australia,Australia/Perth,existing_resolved_location


### Timezone reference persisted

The permanent course-location reference now contains a valid IANA timezone for all 394 course identities.

Coverage was completed through:

- existing resolved course locations;
- documented jurisdiction-level defaults where one timezone safely applies;
- individually reviewed course-level assignments for multi-timezone jurisdictions.

The persisted reference contains 51 distinct IANA timezones and no unresolved course identities.

Exact venue-location work remains incomplete for some courses, but that no longer blocks interpretation of source off times. More detailed venue research can be revisited when individual courses or jurisdictions are studied.